# Healthcare Challenge 3 - Baseline Submission

This notebook provides a simple baseline for **Healthcare Challenge 3: Discharge Readiness Prediction**.

**Goal**: Predict `discharge_ready_day11` (0/1) for each hospital stay
**Metric**: Macro-F1 Score - Higher is better

## Instructions:
1. **Replace API credentials** in the first cell with your team's API key and name
2. **Run all cells** to generate and submit baseline predictions
3. **Check the output** for your submission score

This baseline uses only tabular stay data with a simple Random Forest classifier.


In [ ]:
# 1. Initialize Client and Load Data

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from agentds import BenchmarkClient

# 🔑 REPLACE WITH YOUR CREDENTIALS
client = BenchmarkClient(
    api_key="your_api_key",        # Get from your team dashboard
    team_name="your_team_name"     # Your exact team name
)

# Iteration 1
- 0.7767

In [3]:
# clai_ch3_v0: end-to-end baseline training + deterministic CV + submission + iteration log
import os, json, random
from pathlib import Path
from datetime import datetime

# --- determinism knobs (best-effort in notebook environments) ---
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
random.seed(SEED)

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MaxAbsScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix
import joblib

# -----------------------
# 1) Locate and load data
# -----------------------
candidate_dirs = [
    Path("."),                 # local
    Path("/mnt/data"),         # this environment
    Path("D:/AgentDs/agent_ds_healthcare"),  # user's windows hint (harmless fallback)
]
base_dir = None
for d in candidate_dirs:
    if (d / "stays_train.csv").exists() and (d / "vitals_timeseries.json").exists():
        base_dir = d
        break
if base_dir is None:
    raise FileNotFoundError("Could not find stays_train.csv and vitals_timeseries.json in candidate directories.")

stays_train_path = base_dir / "stays_train.csv"
stays_test_path  = base_dir / "stays_test.csv"
patients_path    = base_dir / "patients.csv"
vitals_path      = base_dir / "vitals_timeseries.json"

st_tr = pd.read_csv(stays_train_path)
st_te = pd.read_csv(stays_test_path)
patients = pd.read_csv(patients_path)
with open(vitals_path, "r") as f:
    vitals_list = json.load(f)

# -----------------------------------------
# 2) Feature engineering from vitals + notes
# -----------------------------------------
def extract_vitals_features(vitals_list):
    """
    Per stay_id:
      - For each vital and derived signal: mean/std/min/max/first/last/delta/slope,
        plus last3 mean/std and last3/first3 ratio
      - Abnormal day counts + stability counts
      - Note length stats + concatenated notes
    Uses only day1-10 info.
    """
    rows = []
    base_vitals = ["hr", "sbp", "dbp", "temp_c", "rr"]

    def safe_stats(arr):
        arr = np.asarray(arr, dtype=float)
        mask = ~np.isnan(arr)
        if mask.any():
            vals = arr[mask]
            return (
                float(vals.mean()),
                float(vals.std(ddof=0)),
                float(vals.min()),
                float(vals.max()),
                mask,
            )
        return (np.nan, np.nan, np.nan, np.nan, mask)

    def safe_mean(arr):
        arr = np.asarray(arr, dtype=float)
        mask = ~np.isnan(arr)
        return float(arr[mask].mean()) if mask.any() else np.nan

    def safe_std(arr):
        arr = np.asarray(arr, dtype=float)
        mask = ~np.isnan(arr)
        return float(arr[mask].std(ddof=0)) if mask.any() else np.nan

    for rec in vitals_list:
        stay_id = rec.get("stay_id")
        days = sorted(rec.get("days", []), key=lambda d: d.get("day", 0))

        # collect notes + note lengths
        notes = []
        for d in days:
            note = d.get("note")
            if note is None:
                note = ""
            notes.append(str(note))
        notes_concat = " ".join([n.strip() for n in notes if n is not None])

        row = {"stay_id": stay_id, "notes_concat": notes_concat}

        # build arrays for vitals
        arrs = {}
        for v in base_vitals:
            vals = []
            for d in days:
                x = d.get(v, np.nan)
                if x is None:
                    x = np.nan
                try:
                    vals.append(float(x))
                except Exception:
                    vals.append(np.nan)
            arrs[v] = np.array(vals, dtype=float)

        # derived signals
        sbp = arrs["sbp"]
        dbp = arrs["dbp"]
        arrs["pp"] = sbp - dbp
        arrs["map"] = (sbp + 2 * dbp) / 3.0

        # day index for slope
        x_idx = np.arange(1, len(days) + 1, dtype=float) if len(days) > 0 else np.array([], dtype=float)

        # stats for each signal (including derived pp/map)
        for name, arr in arrs.items():
            mean, std, mn, mx, mask = safe_stats(arr)
            row[f"{name}_mean"] = mean
            row[f"{name}_std"] = std
            row[f"{name}_min"] = mn
            row[f"{name}_max"] = mx

            row[f"{name}_first"] = float(arr[0]) if len(arr) > 0 else np.nan
            row[f"{name}_last"] = float(arr[-1]) if len(arr) > 0 else np.nan
            if not (np.isnan(row[f"{name}_first"]) or np.isnan(row[f"{name}_last"])):
                row[f"{name}_delta"] = row[f"{name}_last"] - row[f"{name}_first"]
            else:
                row[f"{name}_delta"] = np.nan

            row[f"{name}_missing_frac"] = float((~mask).mean()) if len(arr) > 0 else np.nan

            # slope
            if len(x_idx) > 0 and mask.sum() >= 2:
                row[f"{name}_slope"] = float(np.polyfit(x_idx[mask], arr[mask], 1)[0])
            else:
                row[f"{name}_slope"] = 0.0

            # windows
            last3 = arr[-3:] if len(arr) >= 3 else arr
            first3 = arr[:3] if len(arr) >= 3 else arr

            row[f"{name}_last3_mean"] = safe_mean(last3)
            row[f"{name}_last3_std"] = safe_std(last3)
            row[f"{name}_first3_mean"] = safe_mean(first3)

            denom = row[f"{name}_first3_mean"]
            numer = row[f"{name}_last3_mean"]
            if denom is not None and not np.isnan(denom) and denom != 0 and numer is not None and not np.isnan(numer):
                row[f"{name}_last3_over_first3"] = numer / denom
            else:
                row[f"{name}_last3_over_first3"] = np.nan

        # abnormal/stability counts (simple clinical heuristics)
        hr = arrs["hr"]
        temp = arrs["temp_c"]
        rr = arrs["rr"]

        row["tachy_days"] = int(np.nansum(hr > 100))
        row["brady_days"] = int(np.nansum(hr < 50))
        row["fever_days"] = int(np.nansum(temp >= 38.0))
        row["hypotension_days"] = int(np.nansum(sbp < 90))
        row["tachypnea_days"] = int(np.nansum(rr > 20))

        stable = (hr < 100) & (temp < 38.0) & (sbp >= 90) & (rr <= 20)
        row["stable_days"] = int(np.nansum(stable))
        row["stable_last_day"] = int(bool(stable[-1])) if len(stable) > 0 else 0

        # note length stats
        note_lens = np.array([len(n) for n in notes], dtype=float)
        row["note_len_mean"] = float(note_lens.mean()) if len(note_lens) > 0 else 0.0
        row["note_len_std"] = float(note_lens.std(ddof=0)) if len(note_lens) > 0 else 0.0
        row["note_len_last"] = float(note_lens[-1]) if len(note_lens) > 0 else 0.0

        rows.append(row)

    return pd.DataFrame(rows)

vitals_feat = extract_vitals_features(vitals_list)

# ---------------
# 3) Merge tables
# ---------------
df_tr = st_tr.merge(patients, on="patient_id", how="left").merge(vitals_feat, on="stay_id", how="left")
df_te = st_te.merge(patients, on="patient_id", how="left").merge(vitals_feat, on="stay_id", how="left")

# sanity checks
assert df_tr.shape[0] == st_tr.shape[0], "Train merge changed row count (unexpected)."
assert df_te.shape[0] == st_te.shape[0], "Test merge changed row count (unexpected)."

target = "discharge_ready_day11"
y = df_tr[target].astype(int).values
X = df_tr.drop(columns=[target])

# ---------------------------------------
# 4) Build deterministic baseline pipeline
# ---------------------------------------
cat_cols = ["unit_type", "admission_reason", "sex", "insurance", "zip3"]
text_col = "notes_concat"
id_cols = ["stay_id", "patient_id"]
num_cols = [c for c in X.columns if c not in (id_cols + cat_cols + [text_col])]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MaxAbsScaler()),
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
text_transformer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    max_features=5000,
    strip_accents="unicode",
    lowercase=True,
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
        ("txt", text_transformer, text_col),
    ],
    sparse_threshold=0.3,
)

clf = LogisticRegression(
    solver="liblinear",
    penalty="l2",
    dual=True,                  # faster when n_features > n_samples
    C=2.0,
    max_iter=5000,
    class_weight="balanced",
    random_state=SEED,
)

pipe_template = Pipeline(steps=[("preprocess", preprocess), ("clf", clf)])

# -----------------------------------
# 5) Deterministic CV + threshold tune
# -----------------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_proba = np.zeros(len(X), dtype=float)
fold_payload = []
fold_f1_thr05 = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    model = clone(pipe_template)
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_va)[:, 1]

    oof_proba[va_idx] = proba
    fold_payload.append((va_idx, y_va, proba))

    pred_05 = (proba >= 0.5).astype(int)
    fold_f1_thr05.append(float(f1_score(y_va, pred_05, average="macro")))

# tune threshold on OOF probabilities for macro-F1
thr_grid = np.linspace(0.05, 0.95, 181)  # step=0.005
best_thr = 0.5
best_macro_f1 = -1.0
for t in thr_grid:
    pred = (oof_proba >= t).astype(int)
    f1 = float(f1_score(y, pred, average="macro"))
    if f1 > best_macro_f1 + 1e-12:
        best_macro_f1 = f1
        best_thr = float(t)

fold_f1_tuned = []
for _, y_va, proba in fold_payload:
    fold_f1_tuned.append(float(f1_score(y_va, (proba >= best_thr).astype(int), average="macro")))

oof_pred = (oof_proba >= best_thr).astype(int)
cm = confusion_matrix(y, oof_pred, labels=[0, 1])

# -----------------------------------
# 6) Failure analysis: top FP / top FN
# -----------------------------------
err_df = pd.DataFrame({
    "stay_id": X["stay_id"].values,
    "true": y,
    "proba": oof_proba,
    "pred": oof_pred,
})
top_fp = (
    err_df[(err_df["true"] == 0) & (err_df["pred"] == 1)]
    .sort_values("proba", ascending=False)
    .head(5)
    .to_dict("records")
)
top_fn = (
    err_df[(err_df["true"] == 1) & (err_df["pred"] == 0)]
    .sort_values("proba", ascending=True)
    .head(5)
    .to_dict("records")
)

# -----------------------------
# 7) Train final + make test pred
# -----------------------------
final_model = clone(pipe_template)
final_model.fit(X, y)

X_test = df_te[X.columns]  # ensure identical columns/order
test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_thr).astype(int)

submission = pd.DataFrame({
    "stay_id": df_te["stay_id"].astype(int),
    "discharge_ready_day11": test_pred.astype(int),
})
sub_path = Path("clai_ch3_v0_submission.csv")
submission.to_csv(sub_path, index=False)

# -------------------------
# 8) Save artifacts (optional)
# -------------------------
# feature counts after final fit
pre = final_model.named_steps["preprocess"]
tfidf_vocab_size = len(pre.named_transformers_["txt"].vocabulary_)
ohe = pre.named_transformers_["cat"].named_steps["onehot"]
cat_feat_count = len(ohe.get_feature_names_out(cat_cols))
num_feat_count = len(num_cols)
total_feat_count = int(num_feat_count + cat_feat_count + tfidf_vocab_size)

artifact_path = Path("clai_ch3_v0_model.joblib")
joblib.dump(
    {
        "pipeline": final_model,
        "threshold": best_thr,
        "feature_columns": {
            "num_cols": num_cols,
            "cat_cols": cat_cols,
            "text_col": text_col,
            "id_cols": id_cols,
        },
        "meta": {
            "seed": SEED,
            "total_feature_count_est": total_feat_count,
        },
    },
    artifact_path,
)

# ----------------------------------------
# 9) Append iteration log (append-only jsonl)
# ----------------------------------------
log_path = Path("clai_ch3_v0_iteration_detail.jsonl")
iteration_id = 0
if log_path.exists():
    with open(log_path, "r", encoding="utf-8") as f:
        iteration_id = sum(1 for line in f if line.strip())

timestamp = datetime.now().isoformat()

log_entry = {
    "version": "v0",
    "iteration_id": int(iteration_id),
    "timestamp": timestamp,
    "data_paths_used": {
        "base_dir": str(base_dir),
        "stays_train": str(stays_train_path),
        "stays_test": str(stays_test_path),
        "patients": str(patients_path),
        "vitals_timeseries": str(vitals_path),
        "join_keys": {
            "stays<->patients": "patient_id",
            "stays<->vitals_timeseries": "stay_id",
        },
    },
    "feature_summary": {
        "categorical_cols": cat_cols,
        "text_col": text_col,
        "numeric_feature_count": int(num_feat_count),
        "tfidf": {
            "ngram_range": [1, 2],
            "min_df": 1,
            "max_features": 5000,
            "vocab_size_fit": int(tfidf_vocab_size),
        },
        "engineered_numeric_notes": [
            "per-signal stats: mean/std/min/max/first/last/delta/slope",
            "windowed stats: last3 mean/std + last3/first3 ratio",
            "derived signals: pulse pressure (pp) and MAP (map)",
            "abnormal-day counts + stable day counts",
            "note length stats + TF-IDF on concatenated daily notes",
        ],
        "total_feature_count_est": int(total_feat_count),
    },
    "models_tried": [
        {
            "name": "LogisticRegression(liblinear, L2)",
            "hyperparams": {
                "solver": "liblinear",
                "penalty": "l2",
                "dual": True,
                "C": 2.0,
                "max_iter": 5000,
                "class_weight": "balanced",
                "random_state": SEED,
            },
        }
    ],
    "validation_protocol": {
        "cv": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": SEED,
        "macro_f1_definition": "sklearn.metrics.f1_score(average='macro')",
    },
    "metrics": {
        "macro_f1_oof_best_threshold": float(best_macro_f1),
        "best_threshold": float(best_thr),
        "macro_f1_per_fold_thr0.5": [float(x) for x in fold_f1_thr05],
        "macro_f1_per_fold_tuned": [float(x) for x in fold_f1_tuned],
        "macro_f1_mean_tuned": float(np.mean(fold_f1_tuned)),
        "confusion_matrix_oof_tuned_labels_0_1": cm.tolist(),
        "class_balance_train": {"0": int((y == 0).sum()), "1": int((y == 1).sum())},
        "pred_positive_rate_oof": float(oof_pred.mean()),
    },
    "thresholding_strategy": {
        "strategy": "grid search threshold on out-of-fold probabilities to maximize macro-F1",
        "grid": {"min": 0.05, "max": 0.95, "num": 181, "step": 0.005},
        "best_threshold": float(best_thr),
    },
    "top_errors_failure_analysis": {
        "false_positive_count_oof": int(((y == 0) & (oof_pred == 1)).sum()),
        "false_negative_count_oof": int(((y == 1) & (oof_pred == 0)).sum()),
        "top_false_positives": top_fp,
        "top_false_negatives": top_fn,
    },
    "next_step_hypothesis": (
        "Blend/stack a non-linear numeric-only model (e.g., gradient boosting) with the linear TF-IDF model, "
        "or calibrate probabilities before threshold tuning to improve minority-class recall without hurting precision."
    ),
    "leakage_overfitting_warnings_considered": [
        "Used only day1-10 vitals + daily notes and static demographics/unit/admission_reason.",
        "Did NOT join admissions_* tables (no reliable stay_id mapping; potential one-to-many and leakage via LOS/outcomes).",
        "Threshold tuned only on out-of-fold predictions (no in-fold leakage).",
    ],
    "artifacts_written": {
        "submission_csv": str(sub_path.resolve()),
        "model_joblib": str(artifact_path.resolve()),
        "iteration_log_jsonl": str(log_path.resolve()),
    },
}

with open(log_path, "a", encoding="utf-8") as f:
    f.write(json.dumps(log_entry) + "\n")

# --------------------
# 10) Print run summary
# --------------------
print("=== clai_ch3_v0 run complete ===")
print(f"Base dir: {base_dir}")
print(f"Train shape: {df_tr.shape} | Test shape: {df_te.shape}")
print(f"CV macro-F1 (thr=0.5) per-fold: {np.round(fold_f1_thr05, 4).tolist()} | mean={np.mean(fold_f1_thr05):.4f}")
print(f"Best threshold (OOF tuned): {best_thr:.3f}")
print(f"CV macro-F1 (tuned thr) per-fold: {np.round(fold_f1_tuned, 4).tolist()} | mean={np.mean(fold_f1_tuned):.4f}")
print(f"OOF macro-F1 @ best_thr: {best_macro_f1:.4f}")
print("OOF confusion matrix (labels [0,1]):")
print(cm)
print(f"Estimated total feature count after fit: {total_feat_count}")
print(f"Wrote submission: {sub_path.resolve()}  (rows={len(submission)})")
print(f"Appended iteration log: {log_path.resolve()}  (iteration_id={iteration_id})")
print(f"Saved model artifact: {artifact_path.resolve()}")

=== clai_ch3_v0 run complete ===
Base dir: .
Train shape: (1000, 111) | Test shape: (1000, 110)
CV macro-F1 (thr=0.5) per-fold: [0.8204, 0.7587, 0.7353, 0.7473, 0.8168] | mean=0.7757
Best threshold (OOF tuned): 0.540
CV macro-F1 (tuned thr) per-fold: [0.8225, 0.7708, 0.7613, 0.7724, 0.8095] | mean=0.7873
OOF macro-F1 @ best_thr: 0.7878
OOF confusion matrix (labels [0,1]):
[[260  84]
 [111 545]]
Estimated total feature count after fit: 3955
Wrote submission: D:\AgentDs\agent_ds_healthcare\clai_ch3_v0_submission.csv  (rows=1000)
Appended iteration log: D:\AgentDs\agent_ds_healthcare\clai_ch3_v0_iteration_detail.jsonl  (iteration_id=0)
Saved model artifact: D:\AgentDs\agent_ds_healthcare\clai_ch3_v0_model.joblib


# Iteration 2
- 0.8039

In [5]:
# clai_ch3_v0 — Iteration 1: LightGBM on engineered vitals + demographics + small note keyword flags + OOF threshold tuning
import os, json, re, random, datetime
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

import lightgbm as lgb
import joblib

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# -----------------------------
# Paths (portable: project dir or /mnt/data fallback)
# -----------------------------
BASE = Path(".")
if not (BASE / "stays_train.csv").exists():
    BASE = Path("/mnt/data")

stays_train_path = BASE / "stays_train.csv"
stays_test_path  = BASE / "stays_test.csv"
patients_path    = BASE / "patients.csv"
vitals_path      = BASE / "vitals_timeseries.json"

sub_path = BASE / "clai_ch3_v0_submission.csv"
log_path = BASE / "clai_ch3_v0_iteration_detail.jsonl"
artifact_path = BASE / "clai_ch3_v0_model.joblib"

# -----------------------------
# Load data
# -----------------------------
stays_train = pd.read_csv(stays_train_path)
stays_test  = pd.read_csv(stays_test_path)
patients    = pd.read_csv(patients_path)
vitals_json = json.loads(vitals_path.read_text())

TARGET = "discharge_ready_day11"
ID_COLS = ["stay_id", "patient_id"]
CAT_COLS = ["unit_type", "admission_reason", "sex", "insurance", "zip3"]

# -----------------------------
# Feature engineering from vitals_timeseries.json
# -----------------------------
def _safe_stats(arr: np.ndarray):
    arr = np.asarray(arr, dtype=float)
    m = np.isfinite(arr)
    if m.sum() == 0:
        return dict(mean=np.nan, std=np.nan, min=np.nan, max=np.nan)
    x = arr[m]
    return dict(mean=float(np.mean(x)),
                std=float(np.std(x, ddof=0)),
                min=float(np.min(x)),
                max=float(np.max(x)))

def _safe_slope(days: np.ndarray, vals: np.ndarray):
    days = np.asarray(days, dtype=float)
    vals = np.asarray(vals, dtype=float)
    m = np.isfinite(days) & np.isfinite(vals)
    if m.sum() < 2:
        return np.nan
    x = days[m]
    y = vals[m]
    x_mean = x.mean()
    y_mean = y.mean()
    denom = ((x - x_mean) ** 2).sum()
    if denom == 0:
        return np.nan
    return float(((x - x_mean) * (y - y_mean)).sum() / denom)

def extract_vitals_features(vitals_list):
    rows = []
    for rec in vitals_list:
        sid = rec.get("stay_id")
        days = sorted(rec.get("days", []), key=lambda d: d.get("day", 0))
        df = pd.DataFrame(days).sort_values("day")

        # Ensure columns exist
        for col in ["day", "hr", "sbp", "dbp", "temp_c", "rr", "note"]:
            if col not in df.columns:
                df[col] = np.nan if col != "note" else ""

        # Derived signals
        df["pp"] = df["sbp"] - df["dbp"]                          # pulse pressure
        df["map"] = (df["sbp"] + 2 * df["dbp"]) / 3.0             # mean arterial pressure

        feats = {"stay_id": sid}

        day_idx = df["day"].to_numpy()
        signals = ["hr", "sbp", "dbp", "temp_c", "rr", "pp", "map"]

        # Per-signal summary + trend + stabilization
        for sig in signals:
            vals = df[sig].astype(float).to_numpy()

            st = _safe_stats(vals)
            feats[f"{sig}_mean"] = st["mean"]
            feats[f"{sig}_std"]  = st["std"]
            feats[f"{sig}_min"]  = st["min"]
            feats[f"{sig}_max"]  = st["max"]

            # first/last observed
            s = pd.Series(vals)
            feats[f"{sig}_first_obs"] = float(s[s.notna()].iloc[0]) if s.notna().any() else np.nan
            feats[f"{sig}_last_obs"]  = float(s[s.notna()].iloc[-1]) if s.notna().any() else np.nan
            feats[f"{sig}_delta"]     = (feats[f"{sig}_last_obs"] - feats[f"{sig}_first_obs"]
                                         if np.isfinite(feats[f"{sig}_last_obs"]) and np.isfinite(feats[f"{sig}_first_obs"])
                                         else np.nan)

            # slope over days
            feats[f"{sig}_slope"] = _safe_slope(day_idx, vals)

            # last3 vs first3 stabilization
            first3 = df[sig].iloc[:3].astype(float).to_numpy()
            last3  = df[sig].iloc[-3:].astype(float).to_numpy()
            st_f3 = _safe_stats(first3)
            st_l3 = _safe_stats(last3)
            feats[f"{sig}_first3_mean"] = st_f3["mean"]
            feats[f"{sig}_first3_std"]  = st_f3["std"]
            feats[f"{sig}_last3_mean"]  = st_l3["mean"]
            feats[f"{sig}_last3_std"]   = st_l3["std"]
            feats[f"{sig}_last3_first3_ratio"] = (st_l3["mean"] / st_f3["mean"]
                                                  if np.isfinite(st_l3["mean"]) and np.isfinite(st_f3["mean"]) and st_f3["mean"] != 0
                                                  else np.nan)

            # missingness
            feats[f"{sig}_missing_count"] = int(pd.isna(vals).sum())
            feats[f"{sig}_missing_rate"]  = float(pd.isna(vals).mean())

        # Simple abnormal/stable day counts (clinically plausible thresholds)
        feats["hr_high_days"]    = int(df["hr"].gt(100).fillna(False).sum())
        feats["hr_low_days"]     = int(df["hr"].lt(60).fillna(False).sum())
        feats["sbp_low_days"]    = int(df["sbp"].lt(90).fillna(False).sum())
        feats["temp_fever_days"] = int(df["temp_c"].gt(37.8).fillna(False).sum())
        feats["rr_high_days"]    = int(df["rr"].gt(20).fillna(False).sum())

        stable = 0
        any_abn = 0
        for _, r in df.iterrows():
            ok = True
            if pd.notna(r["hr"]) and not (60 <= r["hr"] <= 100): ok = False
            if pd.notna(r["sbp"]) and not (90 <= r["sbp"] <= 160): ok = False
            if pd.notna(r["temp_c"]) and not (36.0 <= r["temp_c"] <= 37.8): ok = False
            if pd.notna(r["rr"]) and not (12 <= r["rr"] <= 20): ok = False
            stable += int(ok)

            abn = False
            if pd.notna(r["hr"]) and (r["hr"] > 100 or r["hr"] < 60): abn = True
            if pd.notna(r["sbp"]) and (r["sbp"] < 90): abn = True
            if pd.notna(r["temp_c"]) and (r["temp_c"] > 37.8): abn = True
            if pd.notna(r["rr"]) and (r["rr"] > 20): abn = True
            any_abn += int(abn)

        feats["stable_days_count"] = stable
        feats["any_abnormal_days"] = any_abn

        # Notes (string + length stats; no heavy TF-IDF in this iteration)
        notes = df["note"].astype(str).fillna("")
        feats["notes_all"]  = " ".join(notes.tolist())
        feats["notes_last3"] = " ".join(notes.iloc[-3:].tolist())
        feats["note_last"]  = notes.iloc[-1] if len(notes) else ""

        lens = notes.apply(lambda s: len(str(s)))
        feats["note_len_mean"] = float(lens.mean()) if len(lens) else 0.0
        feats["note_len_std"]  = float(lens.std(ddof=0)) if len(lens) else 0.0
        feats["note_len_last"] = float(lens.iloc[-1]) if len(lens) else 0.0

        rows.append(feats)

    return pd.DataFrame(rows)

vitals_feats = extract_vitals_features(vitals_json)

# -----------------------------
# Merge tables
# -----------------------------
train_df = stays_train.merge(patients, on="patient_id", how="left").merge(vitals_feats, on="stay_id", how="left")
test_df  = stays_test.merge(patients, on="patient_id", how="left").merge(vitals_feats, on="stay_id", how="left")

# Ensure categorical types are strings (OneHotEncoder friendly)
for c in CAT_COLS:
    train_df[c] = train_df[c].astype(str)
    test_df[c]  = test_df[c].astype(str)

# -----------------------------
# Lightweight note keyword flags (inject some text signal cheaply)
# -----------------------------
KEYWORDS = {
    "ambulat":   r"ambulat|walk|out of bed|oob|to chair",
    "pt":        r"\bpt\b|physical therapy",
    "pain":      r"\bpain\b|analges|controlled",
    "afebrile":  r"\bafebrile\b",
    "fever":     r"fever|febrile|temp elev|warm",
    "wean_o2":   r"wean|room air|o2|oxygen|sats",
    "iv_abx":    r"\biv\b|antibiotic|\babx\b",
    "nausea":    r"n\/v|nausea|vomit",
    "diet":      r"diet|tolerat|nutrition",
    "rehab":     r"rehab|placement|\bsnf\b",
    "no_issues": r"no new issues|overnight",
    "dvt":       r"dvt prophylaxis",
}

def add_keyword_flags(df: pd.DataFrame) -> pd.DataFrame:
    all_text  = df["notes_all"].astype(str).fillna("").str.lower()
    last3_txt = df["notes_last3"].astype(str).fillna("").str.lower()
    for name, pat in KEYWORDS.items():
        df[f"kw_{name}_all"]   = all_text.str.contains(pat, regex=True).astype(int)
        df[f"kw_{name}_last3"] = last3_txt.str.contains(pat, regex=True).astype(int)
    return df

train_df = add_keyword_flags(train_df)
test_df  = add_keyword_flags(test_df)

# -----------------------------
# Define feature columns
# -----------------------------
TEXT_COLS = ["notes_all", "notes_last3", "note_last"]  # kept for keyword creation; excluded from model

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET].astype(int).to_numpy()
X_test  = test_df.copy()

# Numeric columns: everything that's not ID/CAT/TEXT
numeric_cols = [c for c in X_train.columns if c not in ID_COLS + CAT_COLS + TEXT_COLS]

# Drop fully-missing numeric cols (SimpleImputer(median) would fail)
numeric_cols = [c for c in numeric_cols if pd.to_numeric(X_train[c], errors="coerce").notna().any()]

# -----------------------------
# Model: LightGBM + preprocessing
# -----------------------------
preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), CAT_COLS),
    ],
    sparse_threshold=0.3
)

lgbm = lgb.LGBMClassifier(
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    min_child_samples=20,
    random_state=SEED,
    n_jobs=1,
    objective="binary",
    verbosity=-1,
    deterministic=True,
    force_col_wise=True
)

pipe = Pipeline([("preprocess", preprocess), ("model", lgbm)])

# -----------------------------
# Deterministic CV + OOF preds
# -----------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_proba = np.zeros(len(X_train), dtype=float)
per_fold_f1_05 = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train), 1):
    m = clone(pipe)
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y_train[tr_idx], y_train[va_idx]

    m.fit(X_tr, y_tr)
    p_va = m.predict_proba(X_va)[:, 1]
    oof_proba[va_idx] = p_va

    f1_05 = f1_score(y_va, (p_va >= 0.5).astype(int), average="macro")
    per_fold_f1_05.append(float(f1_05))

# Threshold tuning on OOF to maximize Macro-F1
thr_grid = np.linspace(0.10, 0.90, 801)  # step 0.001
best_thr, best_f1 = 0.5, -1.0
for thr in thr_grid:
    pred = (oof_proba >= thr).astype(int)
    sc = f1_score(y_train, pred, average="macro")
    if sc > best_f1:
        best_f1 = float(sc)
        best_thr = float(thr)

per_fold_f1_tuned = []
for _, va_idx in skf.split(X_train, y_train):
    y_va = y_train[va_idx]
    p_va = oof_proba[va_idx]
    per_fold_f1_tuned.append(float(f1_score(y_va, (p_va >= best_thr).astype(int), average="macro")))

oof_pred = (oof_proba >= best_thr).astype(int)
cm = confusion_matrix(y_train, oof_pred, labels=[0, 1]).tolist()

print("=== CV Results (Deterministic) ===")
print(f"Per-fold Macro-F1 @thr=0.50: {per_fold_f1_05} | mean={np.mean(per_fold_f1_05):.4f}")
print(f"Best OOF threshold: {best_thr:.3f}")
print(f"Per-fold Macro-F1 @thr={best_thr:.3f}: {per_fold_f1_tuned} | mean={np.mean(per_fold_f1_tuned):.4f}")
print(f"OOF Macro-F1 (tuned, global): {best_f1:.4f}")
print(f"OOF confusion matrix [labels 0,1]: {cm}")

# -----------------------------
# Light error analysis (prints)
# -----------------------------
err = stays_train[["stay_id", "patient_id", "unit_type", "admission_reason"]].copy()
err["y_true"] = y_train
err["proba"] = oof_proba
err["y_pred"] = oof_pred
err["err_type"] = np.where((err["y_true"] == 0) & (err["y_pred"] == 1), "FP",
                   np.where((err["y_true"] == 1) & (err["y_pred"] == 0), "FN", "OK"))
print("\nError counts:", err["err_type"].value_counts().to_dict())
print("Error rate by admission_reason:")
print(err.assign(error=(err["y_true"] != err["y_pred"]).astype(int))
          .groupby("admission_reason")["error"].mean()
          .sort_values(ascending=False)
          .round(4)
          .to_string())

print("\nTop 5 FP (highest proba but true=0):")
print(err[err["err_type"] == "FP"].sort_values("proba", ascending=False).head(5).to_string(index=False))
print("\nTop 5 FN (lowest proba but true=1):")
print(err[err["err_type"] == "FN"].sort_values("proba", ascending=True).head(5).to_string(index=False))

# -----------------------------
# Train final model on full training & predict test
# -----------------------------
final_model = clone(pipe)
final_model.fit(X_train, y_train)
test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_thr).astype(int)

submission = pd.DataFrame({
    "stay_id": stays_test["stay_id"].astype(int),
    "discharge_ready_day11": test_pred.astype(int)
})
submission.to_csv(sub_path, index=False)
print(f"\n✅ Wrote submission: {sub_path}  shape={submission.shape}")

# Save artifact (model + metadata)
artifact = {
    "model": final_model,
    "threshold": best_thr,
    "numeric_cols": numeric_cols,
    "categorical_cols": CAT_COLS,
    "seed": SEED
}
joblib.dump(artifact, artifact_path)
print(f"✅ Saved artifact: {artifact_path}")

# -----------------------------
# Append iteration log (JSONL, append-only)
# -----------------------------
def next_iteration_id(path: Path) -> int:
    if not path.exists() or path.stat().st_size == 0:
        return 0
    max_id = -1
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if "iteration_id" in obj:
                    max_id = max(max_id, int(obj["iteration_id"]))
            except Exception:
                continue
    return max_id + 1

iter_id = next_iteration_id(log_path)
timestamp = datetime.datetime.now().isoformat()

entry = {
    "version": "v0",
    "iteration_id": iter_id,
    "timestamp": timestamp,
    "data_paths_used": {
        "base_dir": str(BASE),
        "stays_train": str(stays_train_path),
        "stays_test": str(stays_test_path),
        "patients": str(patients_path),
        "vitals_timeseries": str(vitals_path)
    },
    "join_keys": {
        "patients_join_key": "patient_id",
        "vitals_join_key": "stay_id"
    },
    "feature_summary": {
        "categorical_cols": CAT_COLS,
        "numeric_feature_count": len(numeric_cols),
        "text_features": "No TF-IDF in this iteration; used low-dim keyword flags from notes_all/notes_last3 plus note length stats.",
        "engineered_numeric_vitals": [
            "per-signal stats: mean/std/min/max",
            "first/last observed + delta",
            "slope over days",
            "first3/last3 mean/std + ratio",
            "missingness count/rate",
            "derived signals: pulse pressure (pp), MAP (map)",
            "abnormal day counts + stable day count",
            "note length mean/std/last + keyword flags"
        ],
        "keyword_flag_count": int(len([c for c in numeric_cols if c.startswith('kw_')]))
    },
    "models_tried": [{
        "name": "LightGBMClassifier",
        "hyperparams": {
            "n_estimators": 800,
            "learning_rate": 0.05,
            "num_leaves": 31,
            "subsample": 0.85,
            "colsample_bytree": 0.85,
            "reg_lambda": 1.0,
            "min_child_samples": 20,
            "random_state": SEED,
            "n_jobs": 1,
            "deterministic": True,
            "force_col_wise": True
        }
    }],
    "validation_protocol": {
        "cv": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": SEED,
        "threshold_tuning": "global threshold tuned on OOF probabilities to maximize macro-F1"
    },
    "metrics": {
        "class_balance_train": {str(k): int(v) for k, v in pd.Series(y_train).value_counts().to_dict().items()},
        "macro_f1_per_fold_thr0.5": per_fold_f1_05,
        "macro_f1_per_fold_tuned_global_thr": per_fold_f1_tuned,
        "macro_f1_mean_tuned_global_thr": float(np.mean(per_fold_f1_tuned)),
        "macro_f1_oof_best_threshold": best_f1,
        "best_threshold": best_thr,
        "confusion_matrix_oof_tuned_labels_0_1": cm,
        "pred_positive_rate_oof": float(oof_pred.mean())
    },
    "thresholding_strategy": {
        "grid": {"start": 0.10, "end": 0.90, "num": 801},
        "selected_threshold": best_thr
    },
    "top_errors_failure_analysis": {
        "fp_count": int((err["err_type"] == "FP").sum()),
        "fn_count": int((err["err_type"] == "FN").sum()),
        "error_rate_by_reason": (err.assign(error=(err["y_true"] != err["y_pred"]).astype(int))
                                  .groupby("admission_reason")["error"].mean()
                                  .sort_values(ascending=False)
                                  .round(4)
                                  .to_dict()),
        "example_fp_stay_ids": err[err["err_type"] == "FP"].sort_values("proba", ascending=False).head(5)["stay_id"].astype(int).tolist(),
        "example_fn_stay_ids": err[err["err_type"] == "FN"].sort_values("proba", ascending=True).head(5)["stay_id"].astype(int).tolist()
    },
    "next_step_hypothesis": "Add low-rank TF-IDF note embeddings (SVD components) and/or ensemble (LightGBM vitals + linear TF-IDF) with weight tuned on OOF; consider GroupKFold by patient_id to sanity-check leakage/generalization.",
    "leakage_overfitting_warnings_considered": "Avoided discharge_notes.json (likely post-discharge leakage). Only used stays_* + patients + vitals_timeseries first 10 days. CV is stay-level stratified; if patients repeat across folds, score may be optimistic—consider GroupKFold check.",
    "artifacts_written": {
        "submission_csv": str(sub_path),
        "model_artifact": str(artifact_path),
        "iteration_log": str(log_path)
    }
}

with open(log_path, "a", encoding="utf-8") as f:
    f.write(json.dumps(entry) + "\n")

print(f"✅ Appended iteration log: {log_path} (iteration_id={iter_id})")

c:\Users\shend\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\shend\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\shend\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\shend\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\shend\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: U

=== CV Results (Deterministic) ===
Per-fold Macro-F1 @thr=0.50: [0.8602375960866526, 0.804496578690127, 0.801309022907901, 0.821176199128955, 0.8317105421324678] | mean=0.8238
Best OOF threshold: 0.630
Per-fold Macro-F1 @thr=0.630: [0.8630849694679481, 0.8262078554049357, 0.8204466389855235, 0.8226493892845905, 0.8112985021818611] | mean=0.8287
OOF Macro-F1 (tuned, global): 0.8293
OOF confusion matrix [labels 0,1]: [[251, 93], [57, 599]]

Error counts: {'OK': 850, 'FP': 93, 'FN': 57}
Error rate by admission_reason:
admission_reason
HF              0.1607
DiabetesComp    0.1529
PostOp          0.1527
COPD            0.1429
Pneumonia       0.1369

Top 5 FP (highest proba but true=0):
 stay_id  patient_id unit_type admission_reason  y_true    proba  y_pred err_type
    1475        3204  med_surg        Pneumonia       0 0.999879       1       FP
     482        2535  med_surg     DiabetesComp       0 0.999772       1       FP
     209         803  med_surg               HF       0 0.99952

c:\Users\shend\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


# Iteration 3
- 0.8073

In [7]:
import os, json, datetime, random, warnings, re
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MaxAbsScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.base import clone
import joblib

warnings.filterwarnings("ignore")

# =========================
# Config (deterministic)
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

WRITE_DIR = "."                 # always write outputs here (project root)
FALLBACK_DATA_DIR = "/mnt/data" # allows this cell to run in sandbox too

def resolve_read_path(fname: str) -> str:
    """Read from WRITE_DIR first; fallback to /mnt/data if needed."""
    p0 = os.path.join(WRITE_DIR, fname)
    if os.path.exists(p0):
        return p0
    p1 = os.path.join(FALLBACK_DATA_DIR, fname)
    if os.path.exists(p1):
        return p1
    return p0  # will error naturally if missing

def write_path(fname: str) -> str:
    return os.path.join(WRITE_DIR, fname)

# =========================
# Load data
# =========================
stays_train = pd.read_csv(resolve_read_path("stays_train.csv"))
stays_test  = pd.read_csv(resolve_read_path("stays_test.csv"))
patients    = pd.read_csv(resolve_read_path("patients.csv"))
with open(resolve_read_path("vitals_timeseries.json"), "r") as f:
    vitals_list = json.load(f)

# =========================
# Feature engineering
# =========================
def lin_slope(days, values):
    d = np.asarray(days, dtype=float)
    v = np.asarray(values, dtype=float)
    m = ~np.isnan(v)
    if m.sum() < 2:
        return np.nan
    d = d[m]
    v = v[m]
    d_mean = d.mean()
    v_mean = v.mean()
    denom = ((d - d_mean) ** 2).sum()
    if denom == 0:
        return np.nan
    return float(((d - d_mean) * (v - v_mean)).sum() / denom)

def extract_features(vitals_list):
    """Stay-level features from days 1-10 only (no day11+ leakage)."""
    feats = []
    for rec in vitals_list:
        sid = rec["stay_id"]
        days = sorted(rec["days"], key=lambda x: x["day"])
        day_idx = [d["day"] for d in days]

        def arr(key):
            return np.array([d.get(key, np.nan) for d in days], dtype=float)

        hr   = arr("hr")
        sbp  = arr("sbp")
        dbp  = arr("dbp")
        temp = arr("temp_c")
        rr   = arr("rr")

        # derived vitals
        pp   = sbp - dbp
        mapv = dbp + pp / 3.0

        # notes
        notes = [(d.get("note") or "") for d in days]
        notes_concat = " ".join(notes).strip()
        note_chars = np.array([len(n) for n in notes], dtype=float)

        row = {
            "stay_id": sid,
            "notes_concat": notes_concat,

            # simple note-length signals (often helpful)
            "note_chars_mean": float(note_chars.mean()),
            "note_chars_std": float(note_chars.std(ddof=0)),
            "note_chars_first": float(note_chars[0]),
            "note_chars_last": float(note_chars[-1]),
            "note_chars_delta": float(note_chars[-1] - note_chars[0]),
            "note_nonempty_days": int(np.sum(note_chars > 0)),
        }

        signals = {
            "hr": hr,
            "sbp": sbp,
            "dbp": dbp,
            "temp_c": temp,
            "rr": rr,
            "pp": pp,
            "map": mapv,
        }

        # core time-series aggregates (kept compact to reduce noise/overfit)
        for name, a in signals.items():
            row[f"{name}_mean"] = float(np.nanmean(a)) if np.isfinite(np.nanmean(a)) else np.nan
            row[f"{name}_std"]  = float(np.nanstd(a, ddof=0)) if np.isfinite(np.nanstd(a, ddof=0)) else np.nan
            row[f"{name}_min"]  = float(np.nanmin(a)) if np.isfinite(np.nanmin(a)) else np.nan
            row[f"{name}_max"]  = float(np.nanmax(a)) if np.isfinite(np.nanmax(a)) else np.nan

            row[f"{name}_first"] = float(a[0]) if np.isfinite(a[0]) else np.nan
            row[f"{name}_last"]  = float(a[-1]) if np.isfinite(a[-1]) else np.nan
            row[f"{name}_delta"] = (row[f"{name}_last"] - row[f"{name}_first"]) if (
                np.isfinite(row[f"{name}_last"]) and np.isfinite(row[f"{name}_first"])
            ) else np.nan

            row[f"{name}_slope"] = lin_slope(day_idx, a)

            last3 = a[-3:]
            first3 = a[:3]
            m_last  = float(np.nanmean(last3)) if np.isfinite(np.nanmean(last3)) else np.nan
            s_last  = float(np.nanstd(last3, ddof=0)) if np.isfinite(np.nanstd(last3, ddof=0)) else np.nan
            m_first = float(np.nanmean(first3)) if np.isfinite(np.nanmean(first3)) else np.nan

            row[f"{name}_last3_mean"] = m_last
            row[f"{name}_last3_std"]  = s_last
            row[f"{name}_first3_mean"] = m_first
            row[f"{name}_last3_first3_ratio"] = (m_last / m_first) if (
                np.isfinite(m_last) and np.isfinite(m_first) and abs(m_first) > 1e-6
            ) else np.nan

            row[f"{name}_missing_frac"] = float(np.isnan(a).mean())

        # clinically-motivated counts (simple, transparent)
        row["abn_temp_days"] = int(np.sum((temp > 38) | (temp < 36)))
        row["abn_hr_days"]   = int(np.sum(hr > 100))
        row["abn_rr_days"]   = int(np.sum(rr > 20))
        row["hypotension_days"] = int(np.sum(sbp < 90))

        row["stable_temp_days"] = int(np.sum((temp >= 36) & (temp <= 37.5)))
        row["stable_hr_days"]   = int(np.sum((hr >= 60) & (hr <= 100)))
        row["stable_rr_days"]   = int(np.sum((rr >= 12) & (rr <= 20)))
        row["stable_sbp_days"]  = int(np.sum((sbp >= 90) & (sbp <= 140)))

        feats.append(row)

    return pd.DataFrame(feats)

vitals_feat = extract_features(vitals_list)

train_df = stays_train.merge(patients, on="patient_id", how="left").merge(vitals_feat, on="stay_id", how="left")
test_df  = stays_test.merge(patients, on="patient_id", how="left").merge(vitals_feat, on="stay_id", how="left")

# =========================
# Modeling setup
# =========================
target = "discharge_ready_day11"
id_cols = ["stay_id", "patient_id"]
cat_cols = ["unit_type", "admission_reason", "sex", "insurance", "zip3"]
text_col = "notes_concat"
num_cols = [c for c in train_df.columns if c not in [target] + id_cols + cat_cols + [text_col]]

X = train_df.drop(columns=[target])
y = train_df[target].astype(int)

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", MaxAbsScaler())]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_features=2500), text_col),
    ],
    sparse_threshold=0.3
)

# L1 encourages sparse feature selection across TF-IDF + OHE + numeric
clf = LogisticRegression(
    solver="liblinear",
    penalty="l1",
    dual=False,
    C=1.0,
    max_iter=8000,
    class_weight="balanced",
    random_state=SEED
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# =========================
# Deterministic CV + OOF threshold tuning
# =========================
oof = np.zeros(len(y), dtype=float)
fold_f1_05 = []
fold_probs, fold_true, fold_indices = [], [], []

for fold, (tr, va) in enumerate(cv.split(X, y)):
    pipe = Pipeline([("preprocess", clone(preprocess)),
                     ("clf", clone(clf))])
    pipe.fit(X.iloc[tr], y.iloc[tr])
    proba = pipe.predict_proba(X.iloc[va])[:, 1]

    oof[va] = proba
    fold_probs.append(proba)
    fold_true.append(y.iloc[va].values)
    fold_indices.append(va)

    fold_f1_05.append(float(f1_score(y.iloc[va], (proba >= 0.5).astype(int), average="macro")))

# threshold grid search on OOF probabilities
grid = np.linspace(0.10, 0.90, 161)  # step=0.005
best_thr, best_f1 = 0.5, -1.0
for thr in grid:
    f1 = float(f1_score(y, (oof >= thr).astype(int), average="macro"))
    if f1 > best_f1:
        best_f1, best_thr = f1, float(thr)

fold_f1_tuned = [
    float(f1_score(yva, (p >= best_thr).astype(int), average="macro"))
    for p, yva in zip(fold_probs, fold_true)
]
cm_best = confusion_matrix(y, (oof >= best_thr).astype(int), labels=[0, 1])

print("Data:", train_df.shape, "Test:", test_df.shape)
print("5-fold CV macro-F1 @0.5 per fold:", [round(x, 4) for x in fold_f1_05], "mean", round(float(np.mean(fold_f1_05)), 4))
print("OOF tuned threshold:", round(best_thr, 3), "OOF macro-F1:", round(best_f1, 4))
print("5-fold CV macro-F1 tuned per fold:", [round(x, 4) for x in fold_f1_tuned], "mean", round(float(np.mean(fold_f1_tuned)), 4))
print("OOF confusion matrix [0,1]:\n", cm_best)

def snip(t, n=90):
    t = str(t)
    t = re.sub(r"\s+", " ", t).strip()
    return t[:n] + ("..." if len(t) > n else "")

preds_tuned = (oof >= best_thr).astype(int)
fp_idx = np.where((y.values == 0) & (preds_tuned == 1))[0]
fn_idx = np.where((y.values == 1) & (preds_tuned == 0))[0]
fp_sorted = fp_idx[np.argsort(-oof[fp_idx])] if len(fp_idx) > 0 else np.array([], dtype=int)
fn_sorted = fn_idx[np.argsort(oof[fn_idx])] if len(fn_idx) > 0 else np.array([], dtype=int)

print("\nTop FP (y=0, predicted 1):")
for i in fp_sorted[:3]:
    print({"stay_id": int(train_df.loc[i, "stay_id"]), "proba": round(float(oof[i]), 4), "note": snip(train_df.loc[i, text_col])})

print("\nTop FN (y=1, predicted 0):")
for i in fn_sorted[:3]:
    print({"stay_id": int(train_df.loc[i, "stay_id"]), "proba": round(float(oof[i]), 4), "note": snip(train_df.loc[i, text_col])})

# =========================
# Train final + predict test
# =========================
final_pipe = Pipeline([("preprocess", preprocess), ("clf", clf)])
final_pipe.fit(X, y)

test_proba = final_pipe.predict_proba(test_df)[:, 1]
test_pred = (test_proba >= best_thr).astype(int)

submission = pd.DataFrame({
    "stay_id": test_df["stay_id"].astype(int),
    "discharge_ready_day11": test_pred.astype(int)
})
sub_path = write_path("clai_ch3_v0_submission.csv")
submission.to_csv(sub_path, index=False)

# Save model artifact
model_path = write_path("clai_ch3_v0_model.joblib")
joblib.dump({"pipeline": final_pipe, "threshold": best_thr, "seed": SEED}, model_path)

# =========================
# Append iteration log (JSONL)
# =========================
log_path = write_path("clai_ch3_v0_iteration_detail.jsonl")
existing = []
if os.path.exists(log_path):
    with open(log_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                existing.append(json.loads(line))
            except Exception:
                continue
iteration_id = (max([int(o.get("iteration_id", -1)) for o in existing]) + 1) if existing else 0

# feature counts from fitted pipeline
try:
    tfidf_vocab_size = len(final_pipe.named_steps["preprocess"].named_transformers_["tfidf"].get_feature_names_out())
except Exception:
    tfidf_vocab_size = None
try:
    ohe = final_pipe.named_steps["preprocess"].named_transformers_["cat"].named_steps["onehot"]
    ohe_dim = len(ohe.get_feature_names_out())
except Exception:
    ohe_dim = None
total_feature_est = len(num_cols) + (ohe_dim or 0) + (tfidf_vocab_size or 0)

fp_examples = [{"stay_id": int(train_df.loc[i, "stay_id"]), "y_true": 0, "proba": float(oof[i]), "note_snip": snip(train_df.loc[i, text_col])} for i in fp_sorted[:3]]
fn_examples = [{"stay_id": int(train_df.loc[i, "stay_id"]), "y_true": 1, "proba": float(oof[i]), "note_snip": snip(train_df.loc[i, text_col])} for i in fn_sorted[:3]]

log_entry = {
    "version": "v0",
    "iteration_id": iteration_id,
    "timestamp": datetime.datetime.now().isoformat(),
    "data_paths_used": {
        "stays_train": resolve_read_path("stays_train.csv"),
        "stays_test": resolve_read_path("stays_test.csv"),
        "patients": resolve_read_path("patients.csv"),
        "vitals_timeseries": resolve_read_path("vitals_timeseries.json"),
        "join_keys": {"stays_patients": "patient_id", "stays_vitals": "stay_id"},
        "write_dir": os.path.abspath(WRITE_DIR),
    },
    "feature_summary": {
        "categorical_cols": cat_cols,
        "text_col": text_col,
        "numeric_feature_count": int(len(num_cols)),
        "tfidf": {
            "ngram_range": [1, 2],
            "min_df": 1,
            "max_features": 2500,
            "vocab_size_fit": int(tfidf_vocab_size) if tfidf_vocab_size is not None else None,
        },
        "engineered_numeric_notes": [
            "per-signal stats: mean/std/min/max/first/last/delta/slope",
            "windowed stats: last3 mean/std + last3/first3 ratio",
            "derived signals: pulse pressure (pp) and MAP (map)",
            "abnormal-day counts + stable day counts (simple normal ranges)",
            "note length stats (chars: mean/std/first/last/delta + nonempty days) + TF-IDF on concatenated daily notes",
        ],
        "total_feature_count_est": int(total_feature_est),
    },
    "models_tried": [
        {"name": "LogisticRegression(liblinear, L1)",
         "hyperparams": {
             "solver": "liblinear",
             "penalty": "l1",
             "dual": False,
             "C": 1.0,
             "max_iter": 8000,
             "class_weight": "balanced",
             "random_state": SEED
         }}
    ],
    "validation_protocol": {
        "cv": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": SEED,
        "macro_f1_definition": "sklearn.metrics.f1_score(average='macro')"
    },
    "metrics": {
        "macro_f1_per_fold_thr0.5": fold_f1_05,
        "macro_f1_per_fold_tuned": fold_f1_tuned,
        "macro_f1_mean_tuned": float(np.mean(fold_f1_tuned)),
        "macro_f1_oof_best_threshold": float(best_f1),
        "best_threshold": float(best_thr),
        "confusion_matrix_oof_tuned_labels_0_1": cm_best.tolist(),
        "class_balance_train": {str(k): int(v) for k, v in pd.Series(y).value_counts().sort_index().items()},
        "pred_positive_rate_oof": float(preds_tuned.mean()),
    },
    "thresholding_strategy": "Grid search over thresholds in [0.10, 0.90] step=0.005 maximizing OOF macro-F1.",
    "top_errors_failure_analysis": {
        "false_positives_top3": fp_examples,
        "false_negatives_top3": fn_examples,
        "commentary": "FPs often have upbeat mobility/discharge-planning language; FNs often occur when vitals trends/ICU unit_type dominate despite benign notes."
    },
    "next_step_hypothesis": "Try per-unit_type threshold tuning or add a tiny char-ngrams TF-IDF branch; optionally blend with a numeric-only LightGBM to reduce FPs.",
    "leakage_overfitting_warnings_considered": [
        "Only used vitals/notes from days 1-10 (vitals_timeseries.json) to predict day-11 readiness.",
        "Avoided joining discharge summaries or post-day11 fields.",
        "Threshold tuned strictly on OOF predictions (no peeking at test)."
    ],
    "artifacts_written": {
        "submission": sub_path,
        "model": model_path,
        "iteration_log": log_path
    }
}

with open(log_path, "a") as f:
    f.write(json.dumps(log_entry) + "\n")

print("\nWrote submission:", sub_path)
print("Saved model:", model_path)
print("Appended log:", log_path, "iteration_id", iteration_id)

Data: (1000, 115) Test: (1000, 114)
5-fold CV macro-F1 @0.5 per fold: [0.8215, 0.7755, 0.7864, 0.7536, 0.8166] mean 0.7907
OOF tuned threshold: 0.41 OOF macro-F1: 0.8093
5-fold CV macro-F1 tuned per fold: [0.8329, 0.7945, 0.7995, 0.798, 0.8204] mean 0.8091
OOF confusion matrix [0,1]:
 [[247  97]
 [ 72 584]]

Top FP (y=0, predicted 1):
{'stay_id': 1475, 'proba': 0.9676, 'note': 'resting comfortably fever complaint denied today mobilized with PT vitals stable DVT proph...'}
{'stay_id': 126, 'proba': 0.9458, 'note': 'ambulated with walker; tolerated well education provided on fall prevention OT reviewed ho...'}
{'stay_id': 438, 'proba': 0.9405, 'note': 'respiratory exercises taught; incentive spirometer used OT reviewed home safety resting co...'}

Top FN (y=1, predicted 0):
{'stay_id': 813, 'proba': 0.0317, 'note': 'GI function returning; flatus passed procalcitonin low GI function returning; flatus passe...'}
{'stay_id': 1086, 'proba': 0.0321, 'note': 'tolerating diet telemetry unremark

# Iteration 4
- 0.8067

In [9]:
# clai_ch3_v0 (V0 baseline) - Iteration: upgraded ensemble (LogReg TF-IDF + LightGBM) + OOF weight/threshold tuning
import os, json, re, math, random, time, datetime
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

import lightgbm as lgb
import joblib
import warnings
warnings.filterwarnings("ignore")

# ----------------------------
# Reproducibility
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ----------------------------
# File resolution (project root or /mnt/data fallback)
# ----------------------------
def resolve_file(fname: str) -> Path:
    candidates = [
        Path(".") / fname,
        Path("/mnt/data") / fname,
        Path("../") / fname,
        Path("./data") / fname,
        Path("../data") / fname,
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {fname}. Tried: {[str(c) for c in candidates]}")

stays_train_path = resolve_file("stays_train.csv")
base_dir = stays_train_path.parent
stays_test_path  = resolve_file("stays_test.csv")
patients_path    = resolve_file("patients.csv")
vitals_path      = resolve_file("vitals_timeseries.json")

# Outputs (must be exact names)
submission_path = base_dir / "clai_ch3_v0_submission.csv"
log_path        = base_dir / "clai_ch3_v0_iteration_detail.jsonl"
artifact_path   = base_dir / "clai_ch3_v0_model.joblib"

# ----------------------------
# Load data
# ----------------------------
stays_train = pd.read_csv(stays_train_path)
stays_test  = pd.read_csv(stays_test_path)
patients    = pd.read_csv(patients_path)

with open(vitals_path, "r") as f:
    vitals_data = json.load(f)

TARGET_COL = "discharge_ready_day11"
TEXT_COL = "notes_concat"
CATEGORICAL_COLS = ["unit_type", "admission_reason", "sex", "insurance", "zip3"]

# ----------------------------
# Vitals feature engineering (fast, no per-stay DataFrame construction)
# ----------------------------
SIGNALS = ["hr", "sbp", "dbp", "temp_c", "rr"]
DERIVED = ["pp", "map"]
ALL_SIGS = SIGNALS + DERIVED

NORMAL_RANGES = {
    "hr": (60, 100),
    "sbp": (90, 140),
    "dbp": (60, 90),
    "temp_c": (36.0, 37.5),
    "rr": (12, 20),
    "pp": (30, 60),    # rough heuristic
    "map": (65, 105),  # rough heuristic
}

KEYWORDS = [
    "discharge", "dc", "home", "pt", "ot", "ambulat", "walk", "stable", "improv",
    "wean", "oxygen", "o2", "pain", "afebrile", "diet", "tolerat", "follow", "ready"
]
kw_pattern = re.compile("|".join([re.escape(k) for k in KEYWORDS]), flags=re.IGNORECASE)

def safe_float(x):
    try:
        if x is None:
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def linear_slope(x, y):
    # x, y: numpy arrays; y may contain nan
    mask = ~np.isnan(y)
    if mask.sum() < 2:
        return np.nan, np.nan
    xx = x[mask]
    yy = y[mask]
    x0 = xx - xx.mean()
    denom = float(np.dot(x0, x0) + 1e-12)
    slope = float(np.dot(x0, yy - yy.mean()) / denom)
    yhat = yy.mean() + slope * x0
    ss_res = float(np.sum((yy - yhat) ** 2))
    ss_tot = float(np.sum((yy - yy.mean()) ** 2) + 1e-12)
    r2 = 1.0 - ss_res / ss_tot
    return slope, r2

def build_vitals_features_fast(vitals_data):
    rows = []
    for rec in vitals_data:
        stay_id = rec["stay_id"]
        days = rec.get("days", [])
        # sort for safety (should already be 1..10)
        days = sorted(days, key=lambda d: d.get("day", 0))
        n = len(days)

        hr   = np.array([safe_float(d.get("hr"))     for d in days], dtype=float)
        sbp  = np.array([safe_float(d.get("sbp"))    for d in days], dtype=float)
        dbp  = np.array([safe_float(d.get("dbp"))    for d in days], dtype=float)
        temp = np.array([safe_float(d.get("temp_c")) for d in days], dtype=float)
        rr   = np.array([safe_float(d.get("rr"))     for d in days], dtype=float)
        pp   = sbp - dbp
        mapv = dbp + pp / 3.0

        sig_map = {"hr": hr, "sbp": sbp, "dbp": dbp, "temp_c": temp, "rr": rr, "pp": pp, "map": mapv}

        notes = [(d.get("note") or "") for d in days]
        notes = [str(t) for t in notes]
        notes_concat = " ".join([t.strip() for t in notes if t])
        note_lens = np.fromiter((len(t) for t in notes), dtype=float, count=n) if n else np.array([], dtype=float)
        lower_notes = [t.lower() for t in notes]

        feat = {
            "stay_id": stay_id,
            "notes_concat": notes_concat,
            "note_len_sum": float(note_lens.sum()) if n else 0.0,
            "note_len_mean": float(note_lens.mean()) if n else 0.0,
            "note_len_std": float(note_lens.std()) if n else 0.0,
            "note_len_last": float(note_lens[-1]) if n else 0.0,
            "note_len_first": float(note_lens[0]) if n else 0.0,
            "note_empty_days": int(np.sum(note_lens == 0)) if n else 0,
            "kw_count_total": int(sum(len(kw_pattern.findall(txt)) for txt in lower_notes)),
            "kw_count_last3": int(sum(len(kw_pattern.findall(txt)) for txt in lower_notes[-3:])),
            "kw_count_last1": int(len(kw_pattern.findall(lower_notes[-1])) if n else 0),
        }

        x = np.arange(1, n + 1, dtype=float) if n else np.array([], dtype=float)

        for sig, y in sig_map.items():
            mask = ~np.isnan(y)
            nn = int(mask.sum())
            if nn:
                yv = y[mask]
                feat[f"{sig}_mean"] = float(yv.mean())
                feat[f"{sig}_std"] = float(yv.std())
                feat[f"{sig}_min"] = float(yv.min())
                feat[f"{sig}_max"] = float(yv.max())
                feat[f"{sig}_median"] = float(np.median(yv))
                feat[f"{sig}_q25"] = float(np.percentile(yv, 25))
                feat[f"{sig}_q75"] = float(np.percentile(yv, 75))
            else:
                feat[f"{sig}_mean"] = np.nan
                feat[f"{sig}_std"] = np.nan
                feat[f"{sig}_min"] = np.nan
                feat[f"{sig}_max"] = np.nan
                feat[f"{sig}_median"] = np.nan
                feat[f"{sig}_q25"] = np.nan
                feat[f"{sig}_q75"] = np.nan

            feat[f"{sig}_first"] = float(y[0]) if n else np.nan
            feat[f"{sig}_last"]  = float(y[-1]) if n else np.nan
            feat[f"{sig}_delta"] = float(y[-1] - y[0]) if n else np.nan

            slope, r2 = linear_slope(x, y) if n else (np.nan, np.nan)
            feat[f"{sig}_slope"] = float(slope) if slope == slope else np.nan
            feat[f"{sig}_trend_r2"] = float(r2) if r2 == r2 else np.nan

            if n >= 3:
                last3 = y[-3:]
                first3 = y[:3]
                feat[f"{sig}_last3_mean"] = float(np.nanmean(last3))
                feat[f"{sig}_last3_std"] = float(np.nanstd(last3))
                feat[f"{sig}_first3_mean"] = float(np.nanmean(first3))
                feat[f"{sig}_last3_first3_ratio"] = float(np.nanmean(last3) / (np.nanmean(first3) + 1e-6))
            else:
                feat[f"{sig}_last3_mean"] = np.nan
                feat[f"{sig}_last3_std"] = np.nan
                feat[f"{sig}_first3_mean"] = np.nan
                feat[f"{sig}_last3_first3_ratio"] = np.nan

            diffs = np.diff(y) if n else np.array([], dtype=float)
            feat[f"{sig}_diff_mean_abs"] = float(np.nanmean(np.abs(diffs))) if diffs.size else np.nan
            feat[f"{sig}_diff_max_abs"] = float(np.nanmax(np.abs(diffs))) if diffs.size else np.nan

            miss = int(np.isnan(y).sum()) if n else 0
            feat[f"{sig}_missing_count"] = miss
            feat[f"{sig}_missing_rate"] = float(miss / n) if n else np.nan

            lo, hi = NORMAL_RANGES.get(sig, (None, None))
            if lo is not None and n:
                feat[f"{sig}_abn_low_days"]  = int(np.nansum(y < lo))
                feat[f"{sig}_abn_high_days"] = int(np.nansum(y > hi))
                feat[f"{sig}_abn_days"]      = int(np.nansum((y < lo) | (y > hi)))
            else:
                feat[f"{sig}_abn_low_days"] = np.nan
                feat[f"{sig}_abn_high_days"] = np.nan
                feat[f"{sig}_abn_days"] = np.nan

        rows.append(feat)

    return pd.DataFrame(rows)

print(f"[INFO] Loading + engineering vitals features from {vitals_path} ...")
t_feat0 = time.time()
vitals_feat = build_vitals_features_fast(vitals_data)
print(f"[INFO] Vitals feature table: {vitals_feat.shape} built in {time.time()-t_feat0:.1f}s")

# ----------------------------
# Merge to modeling tables
# ----------------------------
train_df = stays_train.merge(patients, on="patient_id", how="left").merge(vitals_feat, on="stay_id", how="left")
test_df  = stays_test.merge(patients, on="patient_id", how="left").merge(vitals_feat, on="stay_id", how="left")

# Basic cleaning (important for vectorizers / OHE)
for c in CATEGORICAL_COLS:
    train_df[c] = train_df[c].fillna("missing").astype(str)
    test_df[c]  = test_df[c].fillna("missing").astype(str)
train_df[TEXT_COL] = train_df[TEXT_COL].fillna("").astype(str)
test_df[TEXT_COL]  = test_df[TEXT_COL].fillna("").astype(str)

# ----------------------------
# Feature columns
# ----------------------------
exclude_cols = ["stay_id", "patient_id", TARGET_COL] + CATEGORICAL_COLS + [TEXT_COL]
numeric_cols = [c for c in train_df.columns if c not in exclude_cols]

print(f"[INFO] Numeric cols: {len(numeric_cols)} | Cat cols: {len(CATEGORICAL_COLS)} | Text col: {TEXT_COL}")

# ----------------------------
# Models / pipelines
# ----------------------------
# Text: word + char ngrams
word_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=5000, sublinear_tf=True)
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=2000)
text_union = FeatureUnion([("word", word_vec), ("char", char_vec)])

log_preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler(with_mean=False))]), numeric_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL_COLS),
        ("text", text_union, TEXT_COL),
    ],
    sparse_threshold=0.3
)

log_clf = LogisticRegression(
    solver="liblinear",
    penalty="l2",
    C=2.0,
    dual=True,
    max_iter=3000,
    class_weight="balanced",
    random_state=SEED,
)

log_pipe = Pipeline([("prep", log_preprocess), ("clf", log_clf)])

lgb_preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), numeric_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL_COLS),
    ],
    sparse_threshold=0.3
)

lgb_params = dict(
    n_estimators=350,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_samples=30,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="binary",
    random_state=SEED,
    n_jobs=1,            # reproducibility
    is_unbalance=True,   # better than class_weight for binary in LightGBM docs
    verbose=-1,
)
# Optional determinism knobs (silently ignored if unsupported)
lgb_params.update(dict(
    deterministic=True,
    force_col_wise=True,
    seed=SEED,
    feature_fraction_seed=SEED,
    bagging_seed=SEED,
    data_random_seed=SEED,
))
lgb_clf = lgb.LGBMClassifier(**lgb_params)
lgb_pipe = Pipeline([("prep", lgb_preprocess), ("clf", lgb_clf)])

# ----------------------------
# CV + OOF prediction
# ----------------------------
X = train_df.drop(columns=[TARGET_COL])
y = train_df[TARGET_COL].astype(int).values

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_log = np.zeros(len(train_df), dtype=float)
oof_lgb = np.zeros(len(train_df), dtype=float)
fold_indices = []

print("[INFO] Running deterministic 5-fold CV (StratifiedKFold, seed=42) ...")
t_cv0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), start=1):
    X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
    X_va, y_va = X.iloc[va_idx], y[va_idx]

    log_pipe.fit(X_tr, y_tr)
    oof_log[va_idx] = log_pipe.predict_proba(X_va)[:, 1]

    lgb_pipe.fit(X_tr, y_tr)
    oof_lgb[va_idx] = lgb_pipe.predict_proba(X_va)[:, 1]

    fold_indices.append((tr_idx, va_idx))

print(f"[INFO] CV finished in {time.time()-t_cv0:.1f}s")

# ----------------------------
# Threshold + ensemble weight tuning (maximize macro-F1)
# ----------------------------
def best_threshold(y_true, probs, thr_grid):
    best_f1 = -1.0
    best_t = 0.5
    for t in thr_grid:
        pred = (probs >= t).astype(int)
        f1 = f1_score(y_true, pred, average="macro")
        if f1 > best_f1:
            best_f1 = float(f1)
            best_t = float(t)
    return best_f1, best_t

weight_grid = np.linspace(0.0, 1.0, 11)
thr_grid = np.linspace(0.1, 0.9, 81)

best = (-1.0, 0.5, 0.5)  # (macro_f1, weight_log, threshold)
for w in weight_grid:
    ens = w * oof_log + (1.0 - w) * oof_lgb
    f1, t = best_threshold(y, ens, thr_grid)
    if f1 > best[0]:
        best = (float(f1), float(w), float(t))

best_macro_f1_oof, best_w, best_t = best
oof_ens = best_w * oof_log + (1.0 - best_w) * oof_lgb

# Per-fold macro-F1 at thr=0.5 and tuned thr
macro_f1_per_fold_05 = []
macro_f1_per_fold_tuned = []
for tr_idx, va_idx in fold_indices:
    y_va = y[va_idx]
    p_va = oof_ens[va_idx]
    macro_f1_per_fold_05.append(float(f1_score(y_va, (p_va >= 0.5).astype(int), average="macro")))
    macro_f1_per_fold_tuned.append(float(f1_score(y_va, (p_va >= best_t).astype(int), average="macro")))

macro_f1_mean_tuned = float(np.mean(macro_f1_per_fold_tuned))
cm = confusion_matrix(y, (oof_ens >= best_t).astype(int), labels=[0, 1])

print("\n========== VALIDATION RESULTS ==========")
print(f"OOF best Macro-F1: {best_macro_f1_oof:.4f}")
print(f"Best ensemble weight (LogReg): {best_w:.2f}  |  (LightGBM): {1.0-best_w:.2f}")
print(f"Best threshold: {best_t:.2f}")
print(f"Per-fold Macro-F1 @0.5:  {['{:.4f}'.format(x) for x in macro_f1_per_fold_05]}")
print(f"Per-fold Macro-F1 tuned: {['{:.4f}'.format(x) for x in macro_f1_per_fold_tuned]}")
print(f"Mean(per-fold) tuned Macro-F1: {macro_f1_mean_tuned:.4f}")
print("Confusion matrix (labels 0,1) tuned threshold:")
print(cm)
print("=======================================\n")

# ----------------------------
# Failure analysis: top FP/FN examples by confidence
# ----------------------------
err_df = pd.DataFrame({
    "stay_id": train_df["stay_id"].values,
    "y_true": y,
    "prob": oof_ens,
    "y_pred": (oof_ens >= best_t).astype(int)
})
err_df["error"] = err_df["y_true"] != err_df["y_pred"]

fp = err_df[(err_df["y_true"] == 0) & (err_df["y_pred"] == 1)].sort_values("prob", ascending=False).head(5)
fn = err_df[(err_df["y_true"] == 1) & (err_df["y_pred"] == 0)].sort_values("prob", ascending=True).head(5)

top_errors = {
    "false_positives_top5_by_prob": fp[["stay_id", "y_true", "y_pred", "prob"]].to_dict(orient="records"),
    "false_negatives_top5_by_prob": fn[["stay_id", "y_true", "y_pred", "prob"]].to_dict(orient="records"),
}

# ----------------------------
# Train final models on full training + predict test + write submission
# ----------------------------
print("[INFO] Training final models on full training set ...")
t_fit0 = time.time()
log_pipe.fit(X, y)
lgb_pipe.fit(X, y)
print(f"[INFO] Final fit done in {time.time()-t_fit0:.1f}s")

print("[INFO] Predicting test set ...")
test_X = test_df.copy()
proba_log_test = log_pipe.predict_proba(test_X)[:, 1]
proba_lgb_test = lgb_pipe.predict_proba(test_X)[:, 1]
proba_ens_test = best_w * proba_log_test + (1.0 - best_w) * proba_lgb_test
pred_test = (proba_ens_test >= best_t).astype(int)

submission = pd.DataFrame({
    "stay_id": stays_test["stay_id"].values,
    "discharge_ready_day11": pred_test.astype(int)
})
submission.to_csv(submission_path, index=False)
print(f"[OK] Wrote submission: {submission_path}")
print(submission.head())

# ----------------------------
# Append iteration log (append-only)
# ----------------------------
def next_iteration_id(jsonl_path: Path) -> int:
    if not jsonl_path.exists():
        return 0
    # find last non-empty valid json line
    last_id = -1
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict) and "iteration_id" in obj:
                    last_id = max(last_id, int(obj["iteration_id"]))
            except Exception:
                continue
    return last_id + 1

iter_id = next_iteration_id(log_path)

metrics = {
    "macro_f1_oof_best_threshold": float(best_macro_f1_oof),
    "best_threshold": float(best_t),
    "best_weight_logreg": float(best_w),
    "best_weight_lgbm": float(1.0 - best_w),
    "macro_f1_per_fold_thr0.5": [float(x) for x in macro_f1_per_fold_05],
    "macro_f1_per_fold_tuned": [float(x) for x in macro_f1_per_fold_tuned],
    "macro_f1_mean_tuned": float(macro_f1_mean_tuned),
    "confusion_matrix_oof_tuned_labels_0_1": cm.tolist(),
    "class_balance_train": {str(k): int(v) for k, v in pd.Series(y).value_counts().sort_index().to_dict().items()},
    "pred_positive_rate_oof": float((oof_ens >= best_t).mean()),
}

feature_summary = {
    "categorical_cols": CATEGORICAL_COLS,
    "text_col": TEXT_COL,
    "numeric_feature_count": int(len(numeric_cols)),
    "tfidf_word": {"ngram_range": (1, 2), "min_df": 2, "max_features": 5000, "sublinear_tf": True},
    "tfidf_char": {"analyzer": "char_wb", "ngram_range": (3, 5), "min_df": 2, "max_features": 2000},
    "engineered_numeric_notes": [
        "per-signal stats: mean/std/min/max/median/q25/q75/first/last/delta",
        "trend: slope + r2 (linear fit over days)",
        "windowed stats: last3 mean/std + last3/first3 ratio",
        "dynamics: mean/max absolute day-to-day change",
        "missingness: missing_count/missing_rate",
        "derived signals: pulse pressure (pp) and MAP (map)",
        "abnormal-day counts using heuristic ranges",
        "note length stats + keyword counts (incl. last3/last1)",
        "TF-IDF on concatenated daily notes (word + char_wb ngrams)",
    ],
    "total_feature_count_est": "sparse (OHE + TF-IDF) + numeric (see numeric_feature_count)",
}

models_tried = [
    {"name": "LogisticRegression(liblinear, L2, balanced)", "hyperparams": {k: (v if isinstance(v, (int,float,str,bool,type(None))) else str(v)) for k,v in log_clf.get_params().items()}},
    {"name": "LightGBM(LGBMClassifier, is_unbalance=True)", "hyperparams": {k: (v if isinstance(v, (int,float,str,bool,type(None))) else str(v)) for k,v in lgb_params.items()}},
    {"name": "Weighted prob ensemble + threshold tuning", "hyperparams": {"weight_logreg": float(best_w), "weight_lgbm": float(1.0-best_w), "threshold": float(best_t)}},
]

validation_protocol = {
    "cv": "StratifiedKFold",
    "n_splits": 5,
    "shuffle": True,
    "random_state": SEED,
    "macro_f1_definition": "sklearn.metrics.f1_score(average='macro')",
}

thresholding_strategy = {
    "strategy": "Grid-search ensemble weight and decision threshold on OOF probabilities to maximize macro-F1",
    "weight_grid": [float(x) for x in weight_grid],
    "threshold_grid": {"min": float(thr_grid.min()), "max": float(thr_grid.max()), "step": float(thr_grid[1]-thr_grid[0]), "count": int(len(thr_grid))},
}

leakage_notes = [
    "Used only stays/patients + vitals_timeseries (days 1-10) and notes up to day 10; no day-11+ information.",
    "Did NOT join admissions_train/test or discharge_notes to avoid mismatched keys and potential outcome/leakage fields.",
    "Threshold tuned on OOF predictions (still some risk of overfitting to CV split; monitor leaderboard).",
]

next_step = "Try windowed note text (e.g., TF-IDF on last 3 days only) + add CatBoost as a third model for a stronger ensemble."

log_entry = {
    "version": "v0",
    "iteration_id": int(iter_id),
    "timestamp": datetime.datetime.now().isoformat(),
    "data_paths_used": {
        "base_dir": str(base_dir),
        "stays_train": str(stays_train_path.name),
        "stays_test": str(stays_test_path.name),
        "patients": str(patients_path.name),
        "vitals_timeseries": str(vitals_path.name),
        "join_keys": {"stays->patients": "patient_id", "stays->vitals": "stay_id"},
    },
    "feature_summary": feature_summary,
    "models_tried": models_tried,
    "validation_protocol": validation_protocol,
    "metrics": metrics,
    "thresholding_strategy": thresholding_strategy,
    "top_errors_failure_analysis": top_errors,
    "next_step_hypothesis": next_step,
    "leakage_overfitting_warnings_considered": leakage_notes,
    "artifacts_written": {
        "submission": str(submission_path),
        "model_joblib": str(artifact_path),
        "iteration_log": str(log_path),
    },
}

# Append log
with open(log_path, "a", encoding="utf-8") as f:
    f.write(json.dumps(log_entry) + "\n")
print(f"[OK] Appended iteration log: {log_path} (iteration_id={iter_id})")

# Save artifacts
artifact_bundle = {
    "version": "v0",
    "iteration_id": int(iter_id),
    "logreg_pipeline": log_pipe,
    "lgbm_pipeline": lgb_pipe,
    "ensemble_weight_logreg": float(best_w),
    "ensemble_weight_lgbm": float(1.0 - best_w),
    "threshold": float(best_t),
    "categorical_cols": CATEGORICAL_COLS,
    "text_col": TEXT_COL,
    "numeric_cols": numeric_cols,
}
joblib.dump(artifact_bundle, artifact_path)
joblib.dump(artifact_bundle, base_dir / f"clai_ch3_v0_model_iter{iter_id}.joblib")
print(f"[OK] Saved model artifacts: {artifact_path} and clai_ch3_v0_model_iter{iter_id}.joblib")

[INFO] Loading + engineering vitals features from vitals_timeseries.json ...
[INFO] Vitals feature table: (2000, 172) built in 5.5s
[INFO] Numeric cols: 171 | Cat cols: 5 | Text col: notes_concat
[INFO] Running deterministic 5-fold CV (StratifiedKFold, seed=42) ...
[INFO] CV finished in 22.6s

========== VALIDATION RESULTS ==========
OOF best Macro-F1: 0.8279
Best ensemble weight (LogReg): 0.40  |  (LightGBM): 0.60
Best threshold: 0.54
Per-fold Macro-F1 @0.5:  ['0.8570', '0.7926', '0.8177', '0.8095', '0.8400']
Per-fold Macro-F1 tuned: ['0.8592', '0.8130', '0.8367', '0.7914', '0.8378']
Mean(per-fold) tuned Macro-F1: 0.8276
Confusion matrix (labels 0,1) tuned threshold:
[[246  98]
 [ 52 604]]

[INFO] Training final models on full training set ...
[INFO] Final fit done in 4.7s
[INFO] Predicting test set ...
[OK] Wrote submission: clai_ch3_v0_submission.csv
   stay_id  discharge_ready_day11
0      407                      1
1     1594                      1
2     1382                      

# Iteration 5
- 0.7905

In [11]:
import os, json, re, random, datetime
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from lightgbm import LGBMClassifier
import joblib

# =========================
# 0) Determinism
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# =========================
# 1) Path resolution
# =========================
CANDIDATE_BASE_DIRS = [
    Path("."), Path("./healthcare"), Path("../healthcare"),
    Path("/mnt/data"),  # fallback for this sandbox
]
base_dir = None
for d in CANDIDATE_BASE_DIRS:
    if (d / "stays_train.csv").exists() and (d / "vitals_timeseries.json").exists():
        base_dir = d
        break
if base_dir is None:
    raise FileNotFoundError("Could not locate stays_train.csv and vitals_timeseries.json in common locations.")
print(f"[INFO] Using base_dir={base_dir.resolve()}")

# =========================
# 2) Load data
# =========================
stays_train = pd.read_csv(base_dir / "stays_train.csv")
stays_test  = pd.read_csv(base_dir / "stays_test.csv")
patients    = pd.read_csv(base_dir / "patients.csv")

with open(base_dir / "vitals_timeseries.json", "r") as f:
    vitals_json = json.load(f)

# =========================
# 3) Feature engineering from vitals JSON (per-stay)
# =========================
signal_cols = ["hr","sbp","dbp","temp_c","rr","pp","map"]

# Heuristic "normal ranges" for abnormal-day counts (not used as labels; just engineered features)
normal_ranges = {
    "hr": (60, 100),
    "sbp": (90, 140),
    "dbp": (60, 90),
    "temp_c": (36.0, 37.5),
    "rr": (12, 20),
    "pp": (30, 60),
    "map": (65, 100),
}

def _slope(x: np.ndarray, y: np.ndarray) -> float:
    mask = ~np.isnan(y)
    if mask.sum() < 2:
        return np.nan
    x_ = x[mask]; y_ = y[mask]
    xm = x_.mean(); ym = y_.mean()
    denom = np.sum((x_ - xm) ** 2)
    if denom == 0:
        return 0.0
    return float(np.sum((x_ - xm) * (y_ - ym)) / denom)

def _r2(x: np.ndarray, y: np.ndarray) -> float:
    mask = ~np.isnan(y)
    if mask.sum() < 2:
        return np.nan
    x_ = x[mask]; y_ = y[mask]
    sl = _slope(x_, y_)
    intercept = y_.mean() - sl * x_.mean()
    yhat = sl * x_ + intercept
    ss_res = np.sum((y_ - yhat) ** 2)
    ss_tot = np.sum((y_ - y_.mean()) ** 2)
    if ss_tot == 0:
        return 1.0
    return float(1 - ss_res / ss_tot)

vital_rows = []
for obj in vitals_json:
    sid = obj["stay_id"]
    days = sorted(obj["days"], key=lambda z: z.get("day", 0))

    day_idx = np.array([d.get("day", np.nan) for d in days], dtype=float)

    # base signals
    base = {k: np.array([d.get(k, np.nan) for d in days], dtype=float) for k in ["hr","sbp","dbp","temp_c","rr"]}
    base["pp"]  = base["sbp"] - base["dbp"]
    base["map"] = (2 * base["dbp"] + base["sbp"]) / 3.0

    # note segments
    notes = [str(d.get("note","")) if d.get("note", "") is not None else "" for d in days]
    note_all   = " ".join(notes).strip()
    note_early = " ".join(notes[:3]).strip()
    note_late  = " ".join(notes[-3:]).strip()
    note_day10 = (notes[-1] if len(notes) else "").strip()

    words = re.findall(r"[A-Za-z0-9]+", note_all.lower())
    row = {
        "stay_id": sid,
        "note_all": note_all,
        "note_early": note_early,
        "note_late": note_late,
        "note_day10": note_day10,
        "note_words_total": len(words),
        "note_unique_words": len(set(words)),
        "note_char_len": len(note_all),
    }

    for col in signal_cols:
        y = base[col]

        # basic stats
        row[f"{col}_mean"]  = np.nanmean(y)
        row[f"{col}_std"]   = np.nanstd(y)
        row[f"{col}_min"]   = np.nanmin(y)
        row[f"{col}_max"]   = np.nanmax(y)
        row[f"{col}_first"] = y[0] if len(y) else np.nan
        row[f"{col}_last"]  = y[-1] if len(y) else np.nan
        row[f"{col}_delta"] = (y[-1] - y[0]) if len(y) else np.nan
        row[f"{col}_slope"] = _slope(day_idx, y)
        row[f"{col}_r2"]    = _r2(day_idx, y)

        # windowed stats
        y_first3 = y[:3]
        y_last3  = y[-3:]
        row[f"{col}_first3_mean"] = np.nanmean(y_first3)
        row[f"{col}_last3_mean"]  = np.nanmean(y_last3)
        row[f"{col}_first3_std"]  = np.nanstd(y_first3)
        row[f"{col}_last3_std"]   = np.nanstd(y_last3)
        row[f"{col}_last3_minus_first3"] = row[f"{col}_last3_mean"] - row[f"{col}_first3_mean"]
        row[f"{col}_last3_over_first3"]  = row[f"{col}_last3_mean"] / (row[f"{col}_first3_mean"] + 1e-6)

        # volatility + missingness
        row[f"{col}_mad_diff"] = float(np.nanmean(np.abs(np.diff(y)))) if len(y) > 1 else np.nan
        row[f"{col}_missing_frac"] = float(np.mean(np.isnan(y))) if len(y) else np.nan

        # abnormal / stable counts
        lo, hi = normal_ranges[col]
        row[f"{col}_abn_count"]    = int(np.nansum((y < lo) | (y > hi)))
        row[f"{col}_stable_count"] = int(np.nansum((y >= lo) & (y <= hi)))

    vital_rows.append(row)

vital_feats = pd.DataFrame(vital_rows)
print(f"[INFO] vital_feats shape: {vital_feats.shape}")

def build_model_df(stays_df: pd.DataFrame) -> pd.DataFrame:
    df = stays_df.merge(patients, on="patient_id", how="left")
    df = df.merge(vital_feats, on="stay_id", how="left")
    return df

train_df = build_model_df(stays_train)
test_df  = build_model_df(stays_test)

# =========================
# 4) Columns
# =========================
target_col = "discharge_ready_day11"
y = train_df[target_col].astype(int).values
X = train_df.drop(columns=[target_col])

cat_cols = ["unit_type","admission_reason","sex","insurance","zip3"]
text_cols = ["note_all","note_early","note_late","note_day10"]
id_cols = ["stay_id","patient_id"]
num_cols = [c for c in X.columns if c not in cat_cols + text_cols + id_cols]

# ensure robust dtypes
for c in cat_cols:
    X[c] = X[c].astype(str)
    test_df[c] = test_df[c].astype(str)
for c in text_cols:
    X[c] = X[c].astype(str).fillna("")
    test_df[c] = test_df[c].astype(str).fillna("")

# =========================
# 5) Models
# =========================
def make_text_lr_pipeline():
    pre = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
            ("tfidf_all",  TfidfVectorizer(ngram_range=(1,2), min_df=1, max_features=4000, sublinear_tf=True), "note_all"),
            ("tfidf_day10",TfidfVectorizer(ngram_range=(1,2), min_df=1, max_features=1500, sublinear_tf=True), "note_day10"),
            ("tfidf_early",TfidfVectorizer(ngram_range=(1,2), min_df=1, max_features=1500, sublinear_tf=True), "note_early"),
            ("tfidf_late", TfidfVectorizer(ngram_range=(1,2), min_df=1, max_features=1500, sublinear_tf=True), "note_late"),
        ],
        remainder="drop",
        sparse_threshold=0.3
    )
    clf = LogisticRegression(
        solver="liblinear",
        penalty="l2",
        C=0.25,
        max_iter=1000,
        random_state=SEED
    )
    return Pipeline([("pre", pre), ("clf", clf)])

def make_lgbm_pipeline():
    pre = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3
    )
    clf = LGBMClassifier(
        n_estimators=600,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        random_state=SEED,
        n_jobs=1,
        objective="binary",
        verbosity=-1
    )
    return Pipeline([("pre", pre), ("clf", clf)])

def best_threshold(y_true, scores, grid=None):
    if grid is None:
        grid = np.linspace(0.1, 0.9, 81)
    best_thr = 0.5
    best_f1 = -1.0
    for thr in grid:
        pred = (scores >= thr).astype(int)
        f1 = f1_score(y_true, pred, average="macro")
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)
    return best_thr, float(best_f1)

def cv_oof_two_models(pipe_a, pipe_b, X, y, cv):
    oof_a = np.zeros(len(y), dtype=float)
    oof_b = np.zeros(len(y), dtype=float)
    fold_id = np.zeros(len(y), dtype=int)
    f1_a_05, f1_b_05 = [], []
    for k, (tr, va) in enumerate(cv.split(X, y)):
        fold_id[va] = k

        pipe_a.fit(X.iloc[tr], y[tr])
        pipe_b.fit(X.iloc[tr], y[tr])

        pa = pipe_a.predict_proba(X.iloc[va])[:, 1]
        pb = pipe_b.predict_proba(X.iloc[va])[:, 1]
        oof_a[va] = pa
        oof_b[va] = pb

        f1_a_05.append(f1_score(y[va], (pa >= 0.5).astype(int), average="macro"))
        f1_b_05.append(f1_score(y[va], (pb >= 0.5).astype(int), average="macro"))

    return oof_a, oof_b, fold_id, f1_a_05, f1_b_05

# =========================
# 6) Deterministic CV + Ensemble tuning
# =========================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

pipe_lr  = make_text_lr_pipeline()
pipe_lgb = make_lgbm_pipeline()

oof_lr, oof_lgb, fold_id, f1_lr_05, f1_lgb_05 = cv_oof_two_models(pipe_lr, pipe_lgb, X, y, cv)

thr_lr,  f1_lr_best  = best_threshold(y, oof_lr)
thr_lgb, f1_lgb_best = best_threshold(y, oof_lgb)

# Blend weight + threshold grid search on OOF
weight_grid = np.linspace(0, 1, 21)          # 0.00..1.00 step 0.05
thr_grid    = np.linspace(0.1, 0.9, 81)      # 0.10..0.90 step 0.01

best = {"macro_f1": -1.0, "weight_lr": 0.5, "threshold": 0.5}
for w in weight_grid:
    blend = w * oof_lr + (1 - w) * oof_lgb
    thr, f1 = best_threshold(y, blend, thr_grid)
    if f1 > best["macro_f1"]:
        best = {"macro_f1": f1, "weight_lr": float(w), "threshold": float(thr)}

blend_oof = best["weight_lr"] * oof_lr + (1 - best["weight_lr"]) * oof_lgb

# Per-fold ensemble macro-F1 at tuned threshold
f1_ens_tuned = []
for k in range(cv.get_n_splits()):
    idx = np.where(fold_id == k)[0]
    f1_ens_tuned.append(
        f1_score(y[idx], (blend_oof[idx] >= best["threshold"]).astype(int), average="macro")
    )

cm = confusion_matrix(y, (blend_oof >= best["threshold"]).astype(int), labels=[0, 1])

print("\n[CV SUMMARY]")
print(f"  LR (thr=0.5) per-fold: {np.round(f1_lr_05, 4).tolist()} | mean={np.mean(f1_lr_05):.4f}")
print(f"  LR best_thr={thr_lr:.3f} macro_f1_oof={f1_lr_best:.4f}")
print(f"  LGBM (thr=0.5) per-fold: {np.round(f1_lgb_05, 4).tolist()} | mean={np.mean(f1_lgb_05):.4f}")
print(f"  LGBM best_thr={thr_lgb:.3f} macro_f1_oof={f1_lgb_best:.4f}")
print(f"  ENSEMBLE best_w_lr={best['weight_lr']:.2f} best_thr={best['threshold']:.3f} macro_f1_oof={best['macro_f1']:.4f}")
print(f"  ENSEMBLE per-fold (tuned): {np.round(f1_ens_tuned, 4).tolist()} | mean={np.mean(f1_ens_tuned):.4f}")
print(f"  Confusion matrix (OOF, tuned thr) [rows true 0/1, cols pred 0/1]:\n{cm}")

# =========================
# 7) Train final + predict test
# =========================
pipe_lr.fit(X, y)
pipe_lgb.fit(X, y)

X_test = test_df[X.columns].copy()
proba_lr_test  = pipe_lr.predict_proba(X_test)[:, 1]
proba_lgb_test = pipe_lgb.predict_proba(X_test)[:, 1]
proba_blend_test = best["weight_lr"] * proba_lr_test + (1 - best["weight_lr"]) * proba_lgb_test

pred_test = (proba_blend_test >= best["threshold"]).astype(int)

submission = pd.DataFrame({
    "stay_id": test_df["stay_id"].astype(int),
    "discharge_ready_day11": pred_test.astype(int),
})
sub_path = Path("clai_ch3_v0_submission.csv")
submission.to_csv(sub_path, index=False)

print(f"\n[INFO] Wrote submission: {sub_path.resolve()} shape={submission.shape}")
print("[INFO] Predicted class balance on test:")
print(submission["discharge_ready_day11"].value_counts().to_string())

# =========================
# 8) Append iteration log (JSONL, append-only)
# =========================
log_path = Path("clai_ch3_v0_iteration_detail.jsonl")

def next_iteration_id(path: Path) -> int:
    if not path.exists():
        return 0
    mx = -1
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                mx = max(mx, int(obj.get("iteration_id", -1)))
            except Exception:
                continue
    return mx + 1

iter_id = next_iteration_id(log_path)
ts = datetime.datetime.now().isoformat()

# quick failure analysis: top high-confidence errors (OOF ensemble)
oof_pred = (blend_oof >= best["threshold"]).astype(int)
err_idx = np.where(oof_pred != y)[0]
conf = np.abs(blend_oof - best["threshold"])
rank = err_idx[np.argsort(-conf[err_idx])][:6]

top_errors = []
for i in rank:
    top_errors.append({
        "stay_id": int(train_df.loc[i, "stay_id"]),
        "y_true": int(y[i]),
        "score": float(blend_oof[i]),
        "note_day10_excerpt": str(train_df.loc[i, "note_day10"])[:120],
    })

# artifacts
artifacts = []
model_path = Path("clai_ch3_v0_model.joblib")
joblib.dump(
    {
        "lr_pipeline": pipe_lr,
        "lgbm_pipeline": pipe_lgb,
        "ensemble_weight_lr": best["weight_lr"],
        "ensemble_threshold": best["threshold"],
        "feature_columns": list(X.columns),
        "seed": SEED,
    },
    model_path
)
artifacts.append(str(model_path))

log_obj = {
    "version": "v0",
    "iteration_id": iter_id,
    "timestamp": ts,
    "data_paths_used": {
        "base_dir": str(base_dir),
        "stays_train": str(base_dir / "stays_train.csv"),
        "stays_test": str(base_dir / "stays_test.csv"),
        "patients": str(base_dir / "patients.csv"),
        "vitals_timeseries": str(base_dir / "vitals_timeseries.json"),
        "join_keys": ["patient_id", "stay_id"],
    },
    "feature_summary": {
        "categorical_cols": cat_cols,
        "text_cols": text_cols,
        "tfidf": {
            "note_all":   {"ngram_range": [1, 2], "min_df": 1, "max_features": 4000},
            "note_day10": {"ngram_range": [1, 2], "min_df": 1, "max_features": 1500},
            "note_early": {"ngram_range": [1, 2], "min_df": 1, "max_features": 1500},
            "note_late":  {"ngram_range": [1, 2], "min_df": 1, "max_features": 1500},
        },
        "engineered_numeric": [
            "per-signal stats (mean/std/min/max/first/last/delta/slope/r2)",
            "windowed stats (first3 vs last3 means/stds, ratios)",
            "derived signals: pulse pressure (pp) and MAP (map)",
            "missingness + day-to-day volatility (MAD of diffs)",
            "abnormal/stable day counts using heuristic normal ranges",
            "note length stats (chars/words/unique words)",
        ],
        "numeric_feature_count": int(len(num_cols)),
        "total_feature_count_est": "sparse (OHE + 4xTFIDF + numeric)",
    },
    "models_tried": [
        {"name": "LogisticRegression(liblinear, L2)", "hyperparams": {"C": 0.25, "max_iter": 1000}},
        {"name": "LGBMClassifier", "hyperparams": {
            "n_estimators": 600, "learning_rate": 0.05, "num_leaves": 31,
            "subsample": 0.9, "colsample_bytree": 0.9, "reg_lambda": 1.0,
            "n_jobs": 1, "verbosity": -1
        }},
        {"name": "Ensemble(blend)", "hyperparams": {"weight_lr": best["weight_lr"], "weight_lgbm": 1 - best["weight_lr"]}},
    ],
    "validation_protocol": {"cv": "StratifiedKFold", "n_splits": 5, "shuffle": True, "random_state": SEED},
    "metrics": {
        "macro_f1_lr_per_fold_thr0.5": [float(x) for x in f1_lr_05],
        "macro_f1_lgbm_per_fold_thr0.5": [float(x) for x in f1_lgb_05],
        "macro_f1_lr_oof_best_threshold": float(f1_lr_best),
        "best_threshold_lr": float(thr_lr),
        "macro_f1_lgbm_oof_best_threshold": float(f1_lgb_best),
        "best_threshold_lgbm": float(thr_lgb),
        "macro_f1_ensemble_oof_best": float(best["macro_f1"]),
        "best_threshold_ensemble": float(best["threshold"]),
        "best_weight_lr": float(best["weight_lr"]),
        "macro_f1_ensemble_per_fold_tuned": [float(x) for x in f1_ens_tuned],
        "confusion_matrix_oof_ensemble_tuned_labels_0_1": cm.tolist(),
        "class_balance_train": {"n": int(len(y)), "pos_rate": float(np.mean(y)), "pos": int(np.sum(y)), "neg": int(np.sum(1 - y))},
        "pred_positive_rate_oof_ensemble_tuned": float(np.mean(oof_pred)),
    },
    "thresholding_strategy": "Grid search on OOF probs: weight_lr in [0..1 step 0.05], threshold in [0.1..0.9 step 0.01], maximize macro-F1 on OOF.",
    "top_errors_failure_analysis": top_errors,
    "next_step_hypothesis": "Try patient_id-grouped CV (StratifiedGroupKFold) to assess leakage; consider CatBoost with text features or char-ngrams TF-IDF for abbreviations; possibly nested CV for blend/threshold to reduce OOF overfitting.",
    "leakage_overfitting_warnings_considered": [
        "Did NOT use discharge_notes.json (post-discharge leakage).",
        "Only used day1-10 vitals/notes + demographics/unit/reason.",
        "Blend weight & threshold tuned on OOF may slightly overfit; consider nested CV if leaderboard instability.",
    ],
    "artifacts_written": artifacts,
}

with open(log_path, "a") as f:
    f.write(json.dumps(log_obj) + "\n")

print(f"[INFO] Appended iteration log -> {log_path.resolve()} (iteration_id={iter_id})")
print("[DONE]")

[INFO] Using base_dir=D:\AgentDs\agent_ds_healthcare
[INFO] vital_feats shape: (2000, 141)

[CV SUMMARY]
  LR (thr=0.5) per-fold: [0.8092, 0.7498, 0.7799, 0.7702, 0.7847] | mean=0.7788
  LR best_thr=0.590 macro_f1_oof=0.7899
  LGBM (thr=0.5) per-fold: [0.8429, 0.7765, 0.8096, 0.7932, 0.8385] | mean=0.8121
  LGBM best_thr=0.480 macro_f1_oof=0.8147
  ENSEMBLE best_w_lr=0.45 best_thr=0.500 macro_f1_oof=0.8260
  ENSEMBLE per-fold (tuned): [0.8581, 0.7765, 0.8112, 0.8443, 0.8369] | mean=0.8254
  Confusion matrix (OOF, tuned thr) [rows true 0/1, cols pred 0/1]:
[[236 108]
 [ 41 615]]

[INFO] Wrote submission: D:\AgentDs\agent_ds_healthcare\clai_ch3_v0_submission.csv shape=(1000, 2)
[INFO] Predicted class balance on test:
discharge_ready_day11
1    725
0    275
[INFO] Appended iteration log -> D:\AgentDs\agent_ds_healthcare\clai_ch3_v0_iteration_detail.jsonl (iteration_id=4)
[DONE]


# Iteration 6
- 0.8040

In [13]:
# clai_ch3_v0 — Iteration: CatBoost(vitals+static) + Logistic(TF-IDF notes) blend w/ OOF-tuned weight+threshold
import os, json, re, random, warnings, datetime
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from catboost import CatBoostClassifier, Pool
import joblib

warnings.filterwarnings("ignore")

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# -----------------------------
# Paths
# -----------------------------
DATA_DIR = Path(".")
if not (DATA_DIR / "stays_train.csv").exists():
    alt = Path("/mnt/data")
    if (alt / "stays_train.csv").exists():
        DATA_DIR = alt

OUT_DIR = Path(".")
submission_path = OUT_DIR / "clai_ch3_v0_submission.csv"
log_path = OUT_DIR / "clai_ch3_v0_iteration_detail.jsonl"
model_path = OUT_DIR / "clai_ch3_v0_model.joblib"

print(f"Using DATA_DIR = {DATA_DIR.resolve()}")
print(f"Writing outputs to OUT_DIR = {OUT_DIR.resolve()}")

# -----------------------------
# Load data
# -----------------------------
stays_train = pd.read_csv(DATA_DIR / "stays_train.csv")
stays_test  = pd.read_csv(DATA_DIR / "stays_test.csv")
patients    = pd.read_csv(DATA_DIR / "patients.csv")

with open(DATA_DIR / "vitals_timeseries.json", "r", encoding="utf-8") as f:
    vitals_list = json.load(f)

vitals_map = {int(item["stay_id"]): item.get("days", []) for item in vitals_list}

TARGET = "discharge_ready_day11"
cat_cols = ["unit_type", "admission_reason", "sex", "insurance", "zip3"]
text_cols = ["notes_concat", "notes_last3", "note_day10"]
vital_cols = ["hr", "sbp", "dbp", "temp_c", "rr"]

# -----------------------------
# Feature engineering: vitals + notes
# -----------------------------
def build_vitals_features(stay_ids):
    rows = []
    ranges = {
        "hr": (60, 100),
        "sbp": (90, 140),
        "dbp": (60, 90),
        "temp_c": (36.0, 37.5),
        "rr": (12, 20),
        "map": (65, 105),
        "pp": (30, 60),
    }
    idx = np.arange(1, 11, dtype=float)

    for sid in stay_ids:
        sid_int = int(sid)
        days = vitals_map.get(sid_int, [])
        day_dict = {}
        for d in days:
            dd = d.get("day", None)
            if dd is None:
                continue
            try:
                day_dict[int(dd)] = d
            except Exception:
                continue

        feats = {"stay_id": sid_int}
        notes = []

        # per-day vitals + derived signals
        for day in range(1, 11):
            rec = day_dict.get(day, {})
            for vc in vital_cols:
                feats[f"{vc}_d{day}"] = rec.get(vc, np.nan)

            sbp = rec.get("sbp", np.nan)
            dbp = rec.get("dbp", np.nan)
            sbp = np.nan if sbp is None else sbp
            dbp = np.nan if dbp is None else dbp

            feats[f"pp_d{day}"]  = (sbp - dbp) if (pd.notna(sbp) and pd.notna(dbp)) else np.nan
            feats[f"map_d{day}"] = (dbp + (sbp - dbp) / 3.0) if (pd.notna(sbp) and pd.notna(dbp)) else np.nan

            note = rec.get("note", "")
            if note is None:
                note = ""
            notes.append(str(note))

        # text fields (for LR model)
        feats["notes_concat"] = " | ".join([n.strip() for n in notes if n.strip()])
        feats["notes_last3"]  = " | ".join([n.strip() for n in notes[-3:] if n.strip()])
        feats["note_day10"]   = notes[-1].strip() if notes else ""

        # numeric note-length stats (safe for CatBoost structured model)
        lens = np.array([len(n) for n in notes], dtype=float) if notes else np.array([0.0], dtype=float)
        feats["note_len_mean"] = float(lens.mean())
        feats["note_len_std"]  = float(lens.std())
        feats["note_len_last3_mean"] = float(lens[-3:].mean()) if len(lens) >= 3 else float(lens.mean())
        feats["note_len_day10"] = float(lens[-1]) if len(lens) else 0.0

        # aggregates per signal
        sigs = vital_cols + ["pp", "map"]
        for sig in sigs:
            vals = np.array([feats.get(f"{sig}_d{d}", np.nan) for d in range(1, 11)], dtype=float)
            mask = ~np.isnan(vals)
            v = vals[mask]

            feats[f"{sig}_missing"] = int((~mask).sum())
            if v.size == 0:
                feats[f"{sig}_mean"] = np.nan
                feats[f"{sig}_std"]  = np.nan
                feats[f"{sig}_min"]  = np.nan
                feats[f"{sig}_max"]  = np.nan
                feats[f"{sig}_first"] = np.nan
                feats[f"{sig}_last"]  = np.nan
                feats[f"{sig}_delta"] = np.nan
                feats[f"{sig}_slope"] = np.nan
                feats[f"{sig}_last3_mean"] = np.nan
                feats[f"{sig}_first3_mean"] = np.nan
                feats[f"{sig}_last3_std"] = np.nan
                feats[f"{sig}_first3_std"] = np.nan
                feats[f"{sig}_last3_minus_first3"] = np.nan
            else:
                feats[f"{sig}_mean"] = float(np.nanmean(vals))
                feats[f"{sig}_std"]  = float(np.nanstd(vals))
                feats[f"{sig}_min"]  = float(np.nanmin(vals))
                feats[f"{sig}_max"]  = float(np.nanmax(vals))

                first = vals[mask][0]
                last  = vals[mask][-1]
                feats[f"{sig}_first"] = float(first)
                feats[f"{sig}_last"]  = float(last)
                feats[f"{sig}_delta"] = float(last - first)

                if mask.sum() >= 2:
                    x = idx[mask]
                    yv = vals[mask]
                    xm = x.mean()
                    ym = yv.mean()
                    denom = ((x - xm) ** 2).sum()
                    slope = ((x - xm) * (yv - ym)).sum() / denom if denom != 0 else 0.0
                    feats[f"{sig}_slope"] = float(slope)
                else:
                    feats[f"{sig}_slope"] = 0.0

                last3  = vals[-3:]
                first3 = vals[:3]
                feats[f"{sig}_last3_mean"] = float(np.nanmean(last3))
                feats[f"{sig}_first3_mean"] = float(np.nanmean(first3))
                feats[f"{sig}_last3_std"] = float(np.nanstd(last3))
                feats[f"{sig}_first3_std"] = float(np.nanstd(first3))
                feats[f"{sig}_last3_minus_first3"] = float(np.nanmean(last3) - np.nanmean(first3))

        # abnormal / stable counts
        for sig, (lo, hi) in ranges.items():
            vals = np.array([feats.get(f"{sig}_d{d}", np.nan) for d in range(1, 11)], dtype=float)
            v = vals[~np.isnan(vals)]
            if v.size == 0:
                feats[f"{sig}_abn_cnt"] = 0
                feats[f"{sig}_stable_cnt"] = 0
            else:
                feats[f"{sig}_abn_cnt"] = int(((v < lo) | (v > hi)).sum())
                feats[f"{sig}_stable_cnt"] = int(((v >= lo) & (v <= hi)).sum())

        rows.append(feats)

    return pd.DataFrame(rows)

all_stay_ids = pd.concat([stays_train["stay_id"], stays_test["stay_id"]], axis=0).astype(int).unique().tolist()
vitals_feat = build_vitals_features(all_stay_ids)

train_df = (
    stays_train.merge(patients, on="patient_id", how="left")
               .merge(vitals_feat, on="stay_id", how="left")
)
test_df = (
    stays_test.merge(patients, on="patient_id", how="left")
              .merge(vitals_feat, on="stay_id", how="left")
)

y = train_df[TARGET].astype(int).to_numpy()

# -----------------------------
# Model A: CatBoost on structured features (drop raw text)
# -----------------------------
X_cb_train = train_df.drop(columns=[TARGET, "stay_id", "patient_id"] + text_cols).copy()
X_cb_test  = test_df.drop(columns=["stay_id", "patient_id"] + text_cols).copy()

# Ensure categorical types
for c in cat_cols:
    X_cb_train[c] = X_cb_train[c].astype(str).fillna("NA")
    X_cb_test[c]  = X_cb_test[c].astype(str).fillna("NA")

# Align columns exactly
X_cb_test = X_cb_test[X_cb_train.columns]

THREADS = max(1, min(8, (os.cpu_count() or 2)))

cb_params = dict(
    loss_function="Logloss",
    iterations=80,
    learning_rate=0.15,
    depth=6,
    l2_leaf_reg=8.0,
    random_seed=SEED,
    auto_class_weights="Balanced",
    bootstrap_type="Bernoulli",
    subsample=0.8,
    verbose=False,
    allow_writing_files=False,
    thread_count=THREADS,
)

# -----------------------------
# Model B: LogisticRegression + TF-IDF notes (drop per-day numeric to stabilize)
# -----------------------------
X_lr_train = train_df.drop(columns=[TARGET]).copy()
X_lr_test  = test_df.copy()

# define numeric columns for LR: exclude ids, cats, text cols; and exclude per-day "_d{n}" columns
per_day_re = re.compile(r"_d\d+$")
per_day_cols = [c for c in X_lr_train.columns if per_day_re.search(c)]
exclude = set(["stay_id", "patient_id"] + cat_cols + text_cols)
num_cols_lr = [c for c in X_lr_train.columns if (c not in exclude) and (c not in per_day_cols)]

def flatten_text(x):
    # picklable function (avoid lambda) for joblib
    if hasattr(x, "to_numpy"):
        arr = x.to_numpy()
    else:
        arr = np.asarray(x)
    return arr.astype(str).ravel()

pre_lr = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler(with_mean=False))
        ]), num_cols_lr),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
        ("txt", Pipeline([
            ("flatten", FunctionTransformer(flatten_text, validate=False)),
            ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=1, max_features=5000, sublinear_tf=True))
        ]), "notes_concat"),
    ],
    sparse_threshold=0.3
)

lr_clf = LogisticRegression(
    solver="liblinear",
    penalty="l2",
    dual=True,
    C=2.0,
    class_weight="balanced",
    max_iter=20000,
    random_state=SEED,
)

lr_pipe = Pipeline([("pre", pre_lr), ("clf", lr_clf)])

# -----------------------------
# Deterministic CV + OOF predictions
# -----------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_cb = np.zeros(len(train_df), dtype=float)
oof_lr = np.zeros(len(train_df), dtype=float)

per_fold_cb_05 = []
per_fold_lr_05 = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cb_train, y), start=1):
    # CatBoost fold
    cb = CatBoostClassifier(**cb_params)
    cb.fit(Pool(X_cb_train.iloc[tr_idx], y[tr_idx], cat_features=cat_cols))
    proba_cb = cb.predict_proba(Pool(X_cb_train.iloc[va_idx], cat_features=cat_cols))[:, 1]
    oof_cb[va_idx] = proba_cb
    per_fold_cb_05.append(f1_score(y[va_idx], (proba_cb >= 0.5).astype(int), average="macro"))

    # Logistic fold
    lr = clone(lr_pipe)
    lr.fit(X_lr_train.iloc[tr_idx], y[tr_idx])
    proba_lr = lr.predict_proba(X_lr_train.iloc[va_idx])[:, 1]
    oof_lr[va_idx] = proba_lr
    per_fold_lr_05.append(f1_score(y[va_idx], (proba_lr >= 0.5).astype(int), average="macro"))

# -----------------------------
# Tune blend weight + threshold on OOF probabilities
# -----------------------------
weights = np.linspace(0.0, 1.0, 21)  # step 0.05
thr_grid = np.arange(0.05, 0.951, 0.005)

best_w, best_thr, best_f1 = 1.0, 0.5, -1.0
for w in weights:
    p = w * oof_cb + (1.0 - w) * oof_lr
    for thr in thr_grid:
        f1 = f1_score(y, (p >= thr).astype(int), average="macro")
        if f1 > best_f1:
            best_f1 = float(f1)
            best_w = float(w)
            best_thr = float(thr)

oof_blend = best_w * oof_cb + (1.0 - best_w) * oof_lr
oof_pred = (oof_blend >= best_thr).astype(int)

# per-fold tuned
per_fold_blend_tuned = []
cms = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cb_train, y), start=1):
    preds = (oof_blend[va_idx] >= best_thr).astype(int)
    per_fold_blend_tuned.append(f1_score(y[va_idx], preds, average="macro"))
    cms.append(confusion_matrix(y[va_idx], preds, labels=[0, 1]))
cm_total = np.sum(cms, axis=0).tolist()

print("\n====================")
print("CV RESULTS (OOF)")
print("====================")
print(f"CatBoost per-fold Macro-F1 @thr=0.5: {np.round(per_fold_cb_05, 4).tolist()} | mean={np.mean(per_fold_cb_05):.4f}")
print(f"LogReg  per-fold Macro-F1 @thr=0.5: {np.round(per_fold_lr_05, 4).tolist()} | mean={np.mean(per_fold_lr_05):.4f}")
print(f"BLEND   best_w={best_w:.2f}  best_thr={best_thr:.3f}  OOF Macro-F1={best_f1:.4f}")
print(f"BLEND   per-fold Macro-F1 tuned: {np.round(per_fold_blend_tuned, 4).tolist()} | mean={np.mean(per_fold_blend_tuned):.4f}")
print(f"OOF confusion matrix [ [TN, FP], [FN, TP] ] = {cm_total}")
print(f"OOF positive rate (pred=1): {oof_pred.mean():.3f}")

# Top errors
stay_ids = train_df["stay_id"].astype(int).to_numpy()
fp_idx = np.where((oof_pred == 1) & (y == 0))[0]
fn_idx = np.where((oof_pred == 0) & (y == 1))[0]

top_fp = []
if fp_idx.size:
    fp_sorted = fp_idx[np.argsort(oof_blend[fp_idx])[::-1]][:5]
    top_fp = [{"stay_id": int(stay_ids[i]), "true": int(y[i]), "proba": float(oof_blend[i]), "pred": int(oof_pred[i])} for i in fp_sorted]

top_fn = []
if fn_idx.size:
    fn_sorted = fn_idx[np.argsort(oof_blend[fn_idx])][:5]
    top_fn = [{"stay_id": int(stay_ids[i]), "true": int(y[i]), "proba": float(oof_blend[i]), "pred": int(oof_pred[i])} for i in fn_sorted]

print("\nTop False Positives (highest proba but true=0):", top_fp)
print("Top False Negatives (lowest proba but true=1):", top_fn)

# -----------------------------
# Train final models on full training + predict test
# -----------------------------
cb_final = CatBoostClassifier(**cb_params)
cb_final.fit(Pool(X_cb_train, y, cat_features=cat_cols))

lr_final = clone(lr_pipe)
lr_final.fit(X_lr_train, y)

p_cb_test = cb_final.predict_proba(Pool(X_cb_test, cat_features=cat_cols))[:, 1]
p_lr_test = lr_final.predict_proba(X_lr_test)[:, 1]
p_test = best_w * p_cb_test + (1.0 - best_w) * p_lr_test
pred_test = (p_test >= best_thr).astype(int)

submission = pd.DataFrame({
    "stay_id": stays_test["stay_id"].astype(int),
    "discharge_ready_day11": pred_test.astype(int),
})
submission.to_csv(submission_path, index=False)
print(f"\n✅ Wrote submission: {submission_path.resolve()}  shape={submission.shape}")
print(submission.head())

# -----------------------------
# Save artifacts
# -----------------------------
bundle = {
    "version": "v0",
    "seed": SEED,
    "catboost_params": cb_params,
    "blend_weight": best_w,
    "blend_threshold": best_thr,
    "cat_cols": cat_cols,
    "text_cols": text_cols,
    "cb_feature_cols": list(X_cb_train.columns),
    "lr_pipeline": lr_final,
    "cb_model": cb_final,
}
joblib.dump(bundle, model_path)

# also keep an iteration-stamped copy
# (we'll determine iteration_id below; then save a stamped copy too)

# -----------------------------
# Append iteration log (append-only JSONL)
# -----------------------------
iteration_id = 0
if log_path.exists():
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                iteration_id = max(iteration_id, int(obj.get("iteration_id", -1)) + 1)
            except Exception:
                continue

# save stamped model copy after we know iteration_id
stamped_model_path = OUT_DIR / f"clai_ch3_v0_model_iter{iteration_id}.joblib"
joblib.dump(bundle, stamped_model_path)

log_entry = {
    "version": "v0",
    "iteration_id": iteration_id,
    "timestamp": datetime.datetime.now().isoformat(),
    "data_paths_used": {
        "base_dir": str(DATA_DIR.resolve()),
        "stays_train": str((DATA_DIR / "stays_train.csv").resolve()),
        "stays_test": str((DATA_DIR / "stays_test.csv").resolve()),
        "patients": str((DATA_DIR / "patients.csv").resolve()),
        "vitals_timeseries": str((DATA_DIR / "vitals_timeseries.json").resolve()),
        "join_keys": {
            "stays<->patients": "patient_id",
            "stays<->vitals_timeseries": "stay_id",
        },
    },
    "feature_summary": {
        "categorical_cols": cat_cols,
        "text_col": "notes_concat (TF-IDF word 1-2 grams, max_features=5000)",
        "numeric_engineering": [
            "per-day vitals (day1-10) for hr/sbp/dbp/temp_c/rr + derived PP, MAP",
            "per-signal stats: mean/std/min/max/first/last/delta/slope",
            "window stats: first3 vs last3 + last3_minus_first3",
            "abnormal-day counts & stable-day counts (clinically-motivated ranges)",
            "note length stats (mean/std/last3/day10) as numeric",
        ],
        "model_A_catboost_features": {
            "uses_text_raw": False,
            "n_features": int(X_cb_train.shape[1]),
        },
        "model_B_logreg_features": {
            "uses_per_day_numeric": False,
            "n_numeric": int(len(num_cols_lr)),
            "tfidf_max_features": 5000,
        },
    },
    "models_tried": [
        {
            "name": "CatBoostClassifier(structured-only) + LogisticRegression(TF-IDF notes) blend",
            "catboost_hyperparams": cb_params,
            "logreg_hyperparams": {
                "solver": "liblinear",
                "penalty": "l2",
                "dual": True,
                "C": 2.0,
                "class_weight": "balanced",
                "max_iter": 20000,
                "random_state": SEED,
            },
            "blend": {"weight_grid": weights.tolist(), "best_weight": best_w},
        }
    ],
    "validation_protocol": {
        "cv": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": SEED,
        "metric": "macro_f1 (sklearn.metrics.f1_score, average='macro')",
    },
    "metrics": {
        "macro_f1_oof_best": best_f1,
        "best_weight": best_w,
        "best_threshold": best_thr,
        "macro_f1_per_fold_cb_thr0.5": [float(x) for x in per_fold_cb_05],
        "macro_f1_per_fold_lr_thr0.5": [float(x) for x in per_fold_lr_05],
        "macro_f1_per_fold_blend_tuned": [float(x) for x in per_fold_blend_tuned],
        "confusion_matrix_oof_tuned_labels_0_1": cm_total,
        "class_balance_train": {str(k): int(v) for k, v in pd.Series(y).value_counts().to_dict().items()},
        "pred_positive_rate_oof": float(oof_pred.mean()),
    },
    "thresholding_strategy": {
        "strategy": "grid-search blend weight + probability threshold on out-of-fold probs to maximize macro-F1",
        "threshold_grid": {"min": 0.05, "max": 0.95, "step": 0.005},
        "weight_grid": {"min": 0.0, "max": 1.0, "step": 0.05},
        "best_weight": best_w,
        "best_threshold": best_thr,
    },
    "top_errors_failure_analysis": {
        "false_positive_count_oof": int(fp_idx.size),
        "false_negative_count_oof": int(fn_idx.size),
        "top_false_positives": top_fp,
        "top_false_negatives": top_fn,
    },
    "next_step_hypothesis": "If leaderboard still swings, try CatBoost text features for last-3-day notes with stronger regularization, or probability calibration before threshold tuning.",
    "leakage_overfitting_warnings_considered": [
        "Used only day1-10 vitals + day1-10 daily notes (no day11+ signals).",
        "Did NOT join admissions_* or discharge_notes (risk of LOS/outcome leakage & key mismatch).",
        "Blend weight and threshold tuned strictly on out-of-fold probabilities (no in-fold tuning).",
    ],
    "artifacts_written": {
        "submission_csv": str(submission_path.resolve()),
        "model_joblib": str(model_path.resolve()),
        "model_joblib_stamped": str(stamped_model_path.resolve()),
        "iteration_log_jsonl": str(log_path.resolve()),
    },
}

with open(log_path, "a", encoding="utf-8") as f:
    f.write(json.dumps(log_entry) + "\n")

print(f"\n🧾 Appended iteration log: {log_path.resolve()} (iteration_id={iteration_id})")
print(f"💾 Saved model bundle: {model_path.resolve()}")
print(f"💾 Saved stamped model: {stamped_model_path.resolve()}")

Using DATA_DIR = D:\AgentDs\agent_ds_healthcare
Writing outputs to OUT_DIR = D:\AgentDs\agent_ds_healthcare

CV RESULTS (OOF)
CatBoost per-fold Macro-F1 @thr=0.5: [0.8541, 0.8196, 0.8191, 0.8168, 0.8313] | mean=0.8282
LogReg  per-fold Macro-F1 @thr=0.5: [0.8204, 0.7428, 0.7615, 0.7533, 0.7759] | mean=0.7708
BLEND   best_w=1.00  best_thr=0.485  OOF Macro-F1=0.8318
BLEND   per-fold Macro-F1 tuned: [0.8541, 0.8247, 0.8341, 0.8155, 0.8298] | mean=0.8316
OOF confusion matrix [ [TN, FP], [FN, TP] ] = [[249, 95], [52, 604]]
OOF positive rate (pred=1): 0.699

Top False Positives (highest proba but true=0): [{'stay_id': 1475, 'true': 0, 'proba': 0.9849773571874418, 'pred': 1}, {'stay_id': 84, 'true': 0, 'proba': 0.9439644969035491, 'pred': 1}, {'stay_id': 1890, 'true': 0, 'proba': 0.9436122032965322, 'pred': 1}, {'stay_id': 438, 'true': 0, 'proba': 0.9354382798411087, 'pred': 1}, {'stay_id': 729, 'true': 0, 'proba': 0.9234723159637739, 'pred': 1}]
Top False Negatives (lowest proba but true=1): 

# Iteration 7
- 0.7974

In [15]:
import json, re, warnings, random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.base import clone

from lightgbm import LGBMClassifier
import joblib

warnings.filterwarnings("ignore")

# ======================
# Config (deterministic)
# ======================
SEED = 42
N_SPLITS = 5
THR_GRID = np.round(np.linspace(0.1, 0.9, 81), 3)  # threshold grid for macro-F1 tuning
random.seed(SEED)
np.random.seed(SEED)

# ======================
# Paths (robust)
# ======================
data_dir = Path(".")
if not (data_dir / "stays_train.csv").exists():
    data_dir = Path("/mnt/data")  # fallback for this notebook environment

out_dir = Path(".")  # always write outputs to current working directory

paths = {
    "stays_train": data_dir / "stays_train.csv",
    "stays_test": data_dir / "stays_test.csv",
    "patients": data_dir / "patients.csv",
    "vitals_timeseries": data_dir / "vitals_timeseries.json",
}

submission_path = out_dir / "clai_ch3_v0_submission.csv"
log_path = out_dir / "clai_ch3_v0_iteration_detail.jsonl"
model_path = out_dir / "clai_ch3_v0_model.joblib"

# ======================
# Iteration id (append-only log)
# ======================
iteration_id = 0
if log_path.exists():
    max_id = -1
    with open(log_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if obj.get("version") == "v0":
                    max_id = max(max_id, int(obj.get("iteration_id", -1)))
            except Exception:
                continue
    iteration_id = max_id + 1

# ======================
# Load data
# ======================
stays_train = pd.read_csv(paths["stays_train"])
stays_test = pd.read_csv(paths["stays_test"])
patients = pd.read_csv(paths["patients"])
patients["zip3"] = patients["zip3"].astype(str)

with open(paths["vitals_timeseries"], "r") as f:
    vitals_data = json.load(f)

# ======================
# Feature extraction
# ======================
vital_names = ["hr", "sbp", "dbp", "temp_c", "rr"]

kw_list = [
    # discharge planning / disposition
    "discharge", "dc", "d/c", "ready", "home", "snf", "rehab", "placement",
    "case", "social", "follow", "outpatient",
    # clinical course
    "stable", "improv", "better", "worse", "pain", "fever", "infection",
    "oxygen", "wean", "transfer", "icu", "stepdown",
    # mobility / PT
    "ambulat", "walk", "pt", "ot",
]
kw_cols = [f"kw_{re.sub('[^a-z0-9]+', '_', k)}" for k in kw_list]

def slope_from_days(day_nums: np.ndarray, arr: np.ndarray) -> float:
    mask = ~np.isnan(arr) & ~np.isnan(day_nums)
    if mask.sum() < 2:
        return np.nan
    x = day_nums[mask].astype(float)
    y = arr[mask].astype(float)
    x_mean = x.mean()
    denom = ((x - x_mean) ** 2).sum()
    if denom == 0:
        return 0.0
    return float(((x - x_mean) * (y - y.mean())).sum() / denom)

def safe_stat(func, arr, default=np.nan):
    arr = np.asarray(arr, dtype=float)
    if arr.size == 0 or np.isnan(arr).all():
        return default
    with np.errstate(all="ignore"):
        try:
            return float(func(arr))
        except Exception:
            return default

def extract_vitals_features(vitals_data):
    rows = []
    for item in vitals_data:
        sid = item.get("stay_id")
        days = item.get("days", [])
        days_sorted = sorted(days, key=lambda d: d.get("day", 0))
        day_nums = np.array([d.get("day", np.nan) for d in days_sorted], dtype=float)
        if np.isnan(day_nums).all():
            day_nums = np.arange(1, len(days_sorted) + 1, dtype=float)

        feat = {"stay_id": sid}

        # notes
        notes = []
        for d in days_sorted:
            n = d.get("note", "")
            if n is None:
                n = ""
            notes.append(str(n))
        notes_concat = " ".join(notes).strip()
        low = notes_concat.lower()

        feat["notes_concat"] = notes_concat
        feat["note_char_len"] = len(notes_concat)
        feat["note_word_count"] = len(notes_concat.split())
        feat["note_days_missing"] = int(sum(1 for n in notes if str(n).strip() == ""))

        for k, col in zip(kw_list, kw_cols):
            feat[col] = int(low.count(k))

        # vitals arrays
        arrs = {}
        for vn in vital_names:
            arrs[vn] = np.array([d.get(vn, np.nan) for d in days_sorted], dtype=float)

        sbp = arrs["sbp"]
        dbp = arrs["dbp"]
        arrs["pp"] = sbp - dbp
        arrs["map"] = (sbp + 2 * dbp) / 3.0

        for vn, arr in arrs.items():
            feat[f"{vn}_mean"] = safe_stat(np.nanmean, arr)
            feat[f"{vn}_std"] = safe_stat(np.nanstd, arr)
            feat[f"{vn}_min"] = safe_stat(np.nanmin, arr)
            feat[f"{vn}_max"] = safe_stat(np.nanmax, arr)
            feat[f"{vn}_median"] = safe_stat(np.nanmedian, arr)
            feat[f"{vn}_q25"] = safe_stat(lambda a: np.nanpercentile(a, 25), arr)
            feat[f"{vn}_q75"] = safe_stat(lambda a: np.nanpercentile(a, 75), arr)

            feat[f"{vn}_first"] = float(arr[0]) if arr.size else np.nan
            feat[f"{vn}_last"] = float(arr[-1]) if arr.size else np.nan
            feat[f"{vn}_delta"] = feat[f"{vn}_last"] - feat[f"{vn}_first"]
            feat[f"{vn}_slope"] = slope_from_days(day_nums, arr)

            if arr.size >= 3:
                feat[f"{vn}_first3_mean"] = safe_stat(np.nanmean, arr[:3])
                feat[f"{vn}_last3_mean"] = safe_stat(np.nanmean, arr[-3:])
                feat[f"{vn}_last3_std"] = safe_stat(np.nanstd, arr[-3:])
            else:
                feat[f"{vn}_first3_mean"] = np.nan
                feat[f"{vn}_last3_mean"] = np.nan
                feat[f"{vn}_last3_std"] = np.nan

            if arr.size >= 2:
                dif = np.abs(np.diff(arr))
                feat[f"{vn}_absdiff_mean"] = safe_stat(np.nanmean, dif)
                feat[f"{vn}_absdiff_max"] = safe_stat(np.nanmax, dif)
            else:
                feat[f"{vn}_absdiff_mean"] = np.nan
                feat[f"{vn}_absdiff_max"] = np.nan

            feat[f"{vn}_missing"] = int(np.isnan(arr).sum())

        # abnormal counts (simple clinical thresholds)
        hr = arrs["hr"]
        rr = arrs["rr"]
        temp = arrs["temp_c"]
        mapv = arrs["map"]

        feat["abn_hr"] = int(np.nansum((hr < 60) | (hr > 100)))
        feat["abn_rr"] = int(np.nansum((rr < 12) | (rr > 20)))
        feat["abn_temp"] = int(np.nansum(temp >= 37.8))
        feat["abn_sbp"] = int(np.nansum((sbp < 90) | (sbp > 150)))
        feat["abn_map"] = int(np.nansum((mapv < 65) | (mapv > 105)))

        rows.append(feat)

    return pd.DataFrame(rows)

vitals_df = extract_vitals_features(vitals_data)

# ======================
# Merge
# ======================
train_df = stays_train.merge(patients, on="patient_id", how="left").merge(vitals_df, on="stay_id", how="left")
test_df = stays_test.merge(patients, on="patient_id", how="left").merge(vitals_df, on="stay_id", how="left")

target_col = "discharge_ready_day11"
id_cols = ["stay_id", "patient_id"]
cat_cols = ["unit_type", "admission_reason", "sex", "insurance", "zip3"]
text_col = "notes_concat"

feature_cols = [c for c in train_df.columns if c not in id_cols + [target_col]]
numeric_cols = [c for c in feature_cols if c not in cat_cols + [text_col]]

train_df[text_col] = train_df[text_col].fillna("")
test_df[text_col] = test_df[text_col].fillna("")

X = train_df[feature_cols]
y = train_df[target_col].astype(int).values

# ======================
# Pipeline (strong baseline)
# ======================
preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), numeric_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=800, min_df=2), text_col),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

clf = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.9,
    colsample_bytree=0.8,
    min_child_samples=20,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=1,
    class_weight="balanced",
    verbose=-1
)

pipe = Pipeline([("preprocess", preprocess), ("clf", clf)])

# ======================
# Deterministic CV + threshold tuning on OOF
# ======================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_prob = np.zeros(len(y), dtype=float)
fold_f1_thr05 = []
fold_val_indices = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    mdl = clone(pipe)
    mdl.fit(X.iloc[tr_idx], y[tr_idx])
    prob = mdl.predict_proba(X.iloc[va_idx])[:, 1]
    oof_prob[va_idx] = prob
    fold_val_indices.append(va_idx)
    fold_f1_thr05.append(float(f1_score(y[va_idx], (prob >= 0.5).astype(int), average="macro")))

best_thr = 0.5
best_f1 = -1.0
for thr in THR_GRID:
    f1 = f1_score(y, (oof_prob >= thr).astype(int), average="macro")
    if f1 > best_f1:
        best_f1 = float(f1)
        best_thr = float(thr)

fold_f1_tuned = []
for va_idx in fold_val_indices:
    fold_f1_tuned.append(float(f1_score(y[va_idx], (oof_prob[va_idx] >= best_thr).astype(int), average="macro")))

cm = confusion_matrix(y, (oof_prob >= best_thr).astype(int), labels=[0, 1]).tolist()
pred_oof = (oof_prob >= best_thr).astype(int)

# ======================
# Train final + predict test
# ======================
pipe_final = clone(pipe)
pipe_final.fit(X, y)

test_probs = pipe_final.predict_proba(test_df[feature_cols])[:, 1]
test_pred = (test_probs >= best_thr).astype(int)

submission = pd.DataFrame({
    "stay_id": stays_test["stay_id"].astype(int).values,
    "discharge_ready_day11": test_pred.astype(int)
})
submission.to_csv(submission_path, index=False)

# ======================
# Save artifact (joblib)
# ======================
saved_model_path = model_path
try:
    joblib.dump({"pipeline": pipe_final, "best_threshold": best_thr, "seed": SEED}, model_path)
except Exception:
    saved_model_path = out_dir / f"clai_ch3_v0_model_iter{iteration_id}.joblib"
    joblib.dump({"pipeline": pipe_final, "best_threshold": best_thr, "seed": SEED}, saved_model_path)

# ======================
# Logging
# ======================
fp = int(((y == 0) & (pred_oof == 1)).sum())
fn = int(((y == 1) & (pred_oof == 0)).sum())
tp = int(((y == 1) & (pred_oof == 1)).sum())
tn = int(((y == 0) & (pred_oof == 0)).sum())

err_df = train_df.copy()
err_df["y"] = y
err_df["pred"] = pred_oof
err_df["err_type"] = np.where((err_df["y"] == 1) & (err_df["pred"] == 0), "FN",
                              np.where((err_df["y"] == 0) & (err_df["pred"] == 1), "FP", "OK"))
top_fp_reason = err_df[err_df["err_type"] == "FP"]["admission_reason"].value_counts().head(3).to_dict()
top_fn_reason = err_df[err_df["err_type"] == "FN"]["admission_reason"].value_counts().head(3).to_dict()

log_entry = {
    "version": "v0",
    "iteration_id": int(iteration_id),
    "timestamp": datetime.now().isoformat(),
    "data_paths_used": {k: str(v) for k, v in paths.items()},
    "join_keys": {"stays<->patients": "patient_id", "stays<->vitals": "stay_id"},
    "feature_summary": {
        "categorical_cols": cat_cols,
        "text_col": text_col,
        "numeric_feature_count": int(len(numeric_cols)),
        "tfidf": {"ngram_range": [1, 2], "min_df": 2, "max_features": 800},
        "engineered_numeric": [
            "per-signal stats mean/std/min/max/median/q25/q75/first/last/delta/slope",
            "first3/last3 stats + absdiff volatility",
            "derived pp and map",
            "abnormal-day counts for hr/rr/temp/sbp/map",
            "note length + keyword counts"
        ],
        "kw_list": kw_list
    },
    "models_tried": [{
        "name": "LGBMClassifier",
        "hyperparams": {
            "n_estimators": 400,
            "learning_rate": 0.05,
            "num_leaves": 31,
            "subsample": 0.9,
            "colsample_bytree": 0.8,
            "min_child_samples": 20,
            "reg_lambda": 1.0,
            "class_weight": "balanced",
            "random_state": SEED,
            "n_jobs": 1
        }
    }],
    "validation_protocol": {
        "cv": "StratifiedKFold",
        "n_splits": N_SPLITS,
        "shuffle": True,
        "random_state": SEED,
        "threshold_grid": THR_GRID.tolist()
    },
    "metrics": {
        "macro_f1_oof_best_threshold": float(best_f1),
        "best_threshold": float(best_thr),
        "macro_f1_per_fold_thr0.5": fold_f1_thr05,
        "macro_f1_per_fold_tuned": fold_f1_tuned,
        "macro_f1_mean_tuned": float(np.mean(fold_f1_tuned)),
        "confusion_matrix_oof_tuned_labels_0_1": cm,
        "class_balance_train": {str(k): int(v) for k, v in zip(*np.unique(y, return_counts=True))},
        "pred_positive_rate_oof": float(pred_oof.mean())
    },
    "thresholding_strategy": {
        "method": "global threshold tuned on out-of-fold probabilities to maximize macro-F1",
        "selected_threshold": float(best_thr)
    },
    "top_errors_failure_analysis": {
        "counts": {"TN": tn, "FP": fp, "FN": fn, "TP": tp},
        "top_fp_admission_reasons": top_fp_reason,
        "top_fn_admission_reasons": top_fn_reason
    },
    "next_step_hypothesis": "Try a small blend: this LightGBM (vitals) + a fast linear TF-IDF model (notes) to reduce FNs driven by text-only signals; optionally tune unit_type-specific thresholds.",
    "leakage_overfitting_warnings_considered": [
        "Avoided discharge_notes.json and other post-discharge fields.",
        "Used only day 1-10 vitals/notes + static patient demographics + stay metadata."
    ],
    "artifacts_written": {
        "submission_csv": str(submission_path),
        "iteration_log_jsonl": str(log_path),
        "model_joblib": str(saved_model_path)
    }
}

with open(log_path, "a") as f:
    f.write(json.dumps(log_entry) + "\n")

# ======================
# Print run summary
# ======================
print(f"[clai_ch3_v0] iteration_id={iteration_id}  CV_macroF1_tuned={best_f1:.4f}  best_thr={best_thr:.3f}")
print("Per-fold macro-F1 @0.5 :", [round(x, 4) for x in fold_f1_thr05])
print("Per-fold macro-F1 tuned:", [round(x, 4) for x in fold_f1_tuned])
print("OOF confusion matrix [[TN,FP],[FN,TP]]:", cm)
print(f"Wrote submission: {submission_path}  (rows={len(submission)})")
print(f"Appended log:     {log_path}")
print(f"Saved model:      {saved_model_path}")

[clai_ch3_v0] iteration_id=6  CV_macroF1_tuned=0.8261  best_thr=0.680
Per-fold macro-F1 @0.5 : [0.8443, 0.7787, 0.7721, 0.8029, 0.818]
Per-fold macro-F1 tuned: [0.8481, 0.8354, 0.8095, 0.8118, 0.8254]
OOF confusion matrix [[TN,FP],[FN,TP]]: [[270, 74], [84, 572]]
Wrote submission: clai_ch3_v0_submission.csv  (rows=1000)
Appended log:     clai_ch3_v0_iteration_detail.jsonl
Saved model:      clai_ch3_v0_model.joblib


# Iteration 8
- 0.8190

In [17]:
import os, json, re, random, warnings, datetime
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning

import joblib

# -----------------------------
# Config
# -----------------------------
SEED = 42
N_SPLITS = 5
warnings.filterwarnings("ignore", category=ConvergenceWarning)

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# -----------------------------
# Helpers: locate data
# -----------------------------
def _find_data_dir() -> Path:
    for d in [Path("."), Path("/mnt/data")]:
        if (d / "stays_train.csv").exists() and (d / "vitals_timeseries.json").exists():
            return d
    raise FileNotFoundError("Could not find stays_train.csv and vitals_timeseries.json in current dir or /mnt/data")

DATA_DIR = _find_data_dir()
OUT_DIR = DATA_DIR  # write artifacts next to data (project root)

# -----------------------------
# Load data
# -----------------------------
stays_train = pd.read_csv(DATA_DIR / "stays_train.csv")
stays_test  = pd.read_csv(DATA_DIR / "stays_test.csv")
patients    = pd.read_csv(DATA_DIR / "patients.csv")

with open(DATA_DIR / "vitals_timeseries.json", "r") as f:
    vitals_list = json.load(f)

# -----------------------------
# Feature engineering: vitals + notes per stay_id (days 1-10 only)
# -----------------------------
def build_vitals_features(vitals_records):
    rows = []
    keywords = ["discharge", "home", "stable", "improv", "ambulat", "pt", "ot", "pain", "follow", "ready"]

    for rec in vitals_records:
        sid = rec.get("stay_id")
        days = sorted(rec.get("days", []), key=lambda x: x.get("day", 0))

        day_idx = np.array([d.get("day", np.nan) for d in days], dtype=float)

        def arr(key):
            return np.array([d.get(key, np.nan) for d in days], dtype=float)

        hr = arr("hr")
        sbp = arr("sbp")
        dbp = arr("dbp")
        temp = arr("temp_c")
        rr = arr("rr")

        # Derived
        mapv = (2.0 * dbp + sbp) / 3.0
        pp = sbp - dbp
        si = hr / np.where(sbp == 0, np.nan, sbp)

        signals = {
            "hr": hr,
            "sbp": sbp,
            "dbp": dbp,
            "temp_c": temp,
            "rr": rr,
            "map": mapv,
            "pp": pp,
            "si": si,
        }

        feat = {"stay_id": sid}

        # Per-signal aggregates (global + windows + trend + variability)
        for name, a in signals.items():
            s = pd.Series(a)

            feat[f"{name}_mean"] = s.mean()
            feat[f"{name}_std"] = s.std()
            feat[f"{name}_min"] = s.min()
            feat[f"{name}_max"] = s.max()
            feat[f"{name}_median"] = s.median()
            feat[f"{name}_q25"] = s.quantile(0.25)
            feat[f"{name}_q75"] = s.quantile(0.75)
            feat[f"{name}_first"] = s.iloc[0] if len(s) else np.nan
            feat[f"{name}_last"] = s.iloc[-1] if len(s) else np.nan
            feat[f"{name}_delta"] = (feat[f"{name}_last"] - feat[f"{name}_first"]) if len(s) else np.nan
            feat[f"{name}_missing"] = int(s.isna().sum())

            finite = np.isfinite(a) & np.isfinite(day_idx)
            if finite.sum() >= 2:
                try:
                    feat[f"{name}_slope"] = float(np.polyfit(day_idx[finite], a[finite], 1)[0])
                except Exception:
                    feat[f"{name}_slope"] = np.nan
            else:
                feat[f"{name}_slope"] = np.nan

            diffs = np.abs(np.diff(a))
            feat[f"{name}_madiff"] = float(np.nanmean(diffs)) if len(diffs) else np.nan
            feat[f"{name}_maxdiff"] = float(np.nanmax(diffs)) if len(diffs) else np.nan

            last3 = s.iloc[-3:]
            first3 = s.iloc[:3]
            feat[f"{name}_mean_last3"] = last3.mean()
            feat[f"{name}_std_last3"] = last3.std()
            feat[f"{name}_mean_first3"] = first3.mean()
            feat[f"{name}_std_first3"] = first3.std()
            if pd.notnull(feat[f"{name}_mean_last3"]) and pd.notnull(feat[f"{name}_mean_first3"]):
                feat[f"{name}_ratio_last3_first3"] = float(feat[f"{name}_mean_last3"] / (feat[f"{name}_mean_first3"] + 1e-6))
            else:
                feat[f"{name}_ratio_last3_first3"] = np.nan

        # Abnormal day counts
        feat["tachy_days"] = int(np.nansum(hr > 100))
        feat["brady_days"] = int(np.nansum(hr < 60))
        feat["hypotension_days"] = int(np.nansum(sbp < 90))
        feat["hypertension_days"] = int(np.nansum(sbp > 160))
        feat["fever_days"] = int(np.nansum(temp > 38.0))
        feat["hypothermia_days"] = int(np.nansum(temp < 36.0))
        feat["tachypnea_days"] = int(np.nansum(rr > 20))
        feat["bradypnea_days"] = int(np.nansum(rr < 10))

        stable = (
            (hr >= 60) & (hr <= 100) &
            (sbp >= 90) & (sbp <= 140) &
            (temp >= 36.0) & (temp <= 37.5) &
            (rr >= 12) & (rr <= 20)
        )
        feat["stable_days"] = int(np.nansum(stable))

        # Notes
        notes = [str(d.get("note") or "") for d in days]
        feat["notes_concat"] = " ".join(notes)
        feat["notes_last3"] = " ".join(notes[-3:])

        lens = np.array([len(n) for n in notes], dtype=float)
        feat["note_len_mean"] = float(np.mean(lens)) if len(lens) else 0.0
        feat["note_len_std"] = float(np.std(lens)) if len(lens) else 0.0
        feat["note_len_last3_mean"] = float(np.mean(lens[-3:])) if len(lens) else 0.0
        feat["note_len_sum"] = float(np.sum(lens)) if len(lens) else 0.0

        text_all = feat["notes_concat"].lower()
        text_last3 = feat["notes_last3"].lower()
        for kw in keywords:
            feat[f"kw_{kw}_all"] = len(re.findall(kw, text_all))
            feat[f"kw_{kw}_last3"] = len(re.findall(kw, text_last3))

        rows.append(feat)

    return pd.DataFrame(rows)

vital_feats = build_vitals_features(vitals_list)

# Merge tables
train_df = stays_train.merge(patients, on="patient_id", how="left").merge(vital_feats, on="stay_id", how="left")
test_df  = stays_test.merge(patients, on="patient_id", how="left").merge(vital_feats, on="stay_id", how="left")

# Columns
text_cols = ["notes_concat", "notes_last3"]
cat_cols  = ["unit_type", "admission_reason", "sex", "insurance", "zip3"]

for c in text_cols:
    train_df[c] = train_df[c].fillna("")
    test_df[c]  = test_df[c].fillna("")
for c in cat_cols:
    train_df[c] = train_df[c].fillna("UNK").astype(str)
    test_df[c]  = test_df[c].fillna("UNK").astype(str)

drop_cols = ["stay_id", "patient_id", "discharge_ready_day11"] + text_cols + cat_cols
num_cols = [c for c in train_df.columns if c not in drop_cols]

X = train_df.drop(columns=["discharge_ready_day11"])
y = train_df["discharge_ready_day11"].astype(int).values
X_test = test_df.copy()

# -----------------------------
# Build models
# -----------------------------
pre_lr = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler(with_mean=False))]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("txt_all", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=15000, sublinear_tf=True), "notes_concat"),
        ("txt_last3", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=8000, sublinear_tf=True), "notes_last3"),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

lr_model = Pipeline([
    ("pre", pre_lr),
    ("clf", LogisticRegression(
        solver="saga",
        penalty="l2",
        C=1.0,
        max_iter=5000,
        tol=1e-4,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=1
    ))
])

pre_gbm = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

try:
    import lightgbm as lgb
    gbm_clf = lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=63,
        max_depth=-1,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.0,
        reg_lambda=1.0,
        min_child_samples=15,
        objective="binary",
        random_state=SEED,
        n_jobs=1,
        verbose=-1
    )
except Exception:
    # Fallback: linear model on numeric+cat only (keeps the pipeline runnable)
    gbm_clf = LogisticRegression(
        solver="liblinear",
        penalty="l2",
        C=1.0,
        max_iter=5000,
        class_weight="balanced",
        random_state=SEED
    )

gbm_model = Pipeline([
    ("pre", pre_gbm),
    ("clf", gbm_clf)
])

# -----------------------------
# Iteration id (append-only jsonl)
# -----------------------------
log_path = OUT_DIR / "clai_ch3_v0_iteration_detail.jsonl"
iteration_id = 0
if log_path.exists():
    ids = []
    with open(log_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if obj.get("version") == "v0" and "iteration_id" in obj:
                    ids.append(int(obj["iteration_id"]))
            except Exception:
                continue
    if ids:
        iteration_id = max(ids) + 1

timestamp = datetime.datetime.now().isoformat()

# -----------------------------
# Deterministic CV + OOF predictions
# -----------------------------
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(cv.split(X, y))

oof_lr = np.zeros(len(y), dtype=float)
oof_gbm = np.zeros(len(y), dtype=float)

print(f"DATA_DIR={DATA_DIR.resolve()}")
print(f"OUT_DIR={OUT_DIR.resolve()}")
print(f"Iteration {iteration_id} | timestamp={timestamp}")
print(f"Train shape: {train_df.shape} | Test shape: {test_df.shape}")
print(f"Class balance: {dict(pd.Series(y).value_counts().sort_index())}")
print(f"Numeric features: {len(num_cols)} | Categorical: {len(cat_cols)} | Text cols: {text_cols}")

for fold, (tr_idx, va_idx) in enumerate(folds, start=1):
    print(f"Fold {fold}/{N_SPLITS} ...")
    mdl_lr = clone(lr_model)
    mdl_gbm = clone(gbm_model)

    mdl_lr.fit(X.iloc[tr_idx], y[tr_idx])
    oof_lr[va_idx] = mdl_lr.predict_proba(X.iloc[va_idx])[:, 1]

    mdl_gbm.fit(X.iloc[tr_idx], y[tr_idx])
    if hasattr(mdl_gbm, "predict_proba"):
        oof_gbm[va_idx] = mdl_gbm.predict_proba(X.iloc[va_idx])[:, 1]
    else:
        # decision_function -> sigmoid
        dfun = mdl_gbm.decision_function(X.iloc[va_idx])
        oof_gbm[va_idx] = 1.0 / (1.0 + np.exp(-dfun))

# -----------------------------
# Threshold + blend weight tuning on OOF to maximize macro-F1
# -----------------------------
thresholds = np.linspace(0.05, 0.95, 181)  # step=0.005
weights = np.linspace(0.0, 1.0, 21)        # lr weight

best_score = -1.0
best_w = 0.5
best_thr = 0.5

for w in weights:
    oof_ens = w * oof_lr + (1.0 - w) * oof_gbm
    for thr in thresholds:
        pred = (oof_ens >= thr).astype(int)
        score = f1_score(y, pred, average="macro")
        if score > best_score:
            best_score = score
            best_w = float(w)
            best_thr = float(thr)

oof_ens = best_w * oof_lr + (1.0 - best_w) * oof_gbm
pred_ens_05 = (oof_ens >= 0.5).astype(int)
pred_ens_tuned = (oof_ens >= best_thr).astype(int)

# Per-fold scores
def _per_fold_scores(oof_probs, thr):
    scores = []
    cms = []
    for _, va_idx in folds:
        p = (oof_probs[va_idx] >= thr).astype(int)
        scores.append(f1_score(y[va_idx], p, average="macro"))
        cms.append(confusion_matrix(y[va_idx], p, labels=[0, 1]))
    return scores, sum(cms)

scores_05, _ = _per_fold_scores(oof_ens, 0.5)
scores_tuned, cm_tuned = _per_fold_scores(oof_ens, best_thr)

print("\nOOF tuning results (ensemble):")
print(f"  best_macro_f1_oof = {best_score:.6f}")
print(f"  best_lr_weight    = {best_w:.3f}")
print(f"  best_threshold    = {best_thr:.3f}")
print(f"  macro_f1_per_fold_thr0.5  = {[round(s,6) for s in scores_05]} | mean={np.mean(scores_05):.6f}")
print(f"  macro_f1_per_fold_tuned   = {[round(s,6) for s in scores_tuned]} | mean={np.mean(scores_tuned):.6f}")
print(f"  confusion_matrix_oof_tuned [ [TN,FP],[FN,TP] ] = {cm_tuned.tolist()}")

# Top errors
fp_idx = np.where((pred_ens_tuned == 1) & (y == 0))[0]
fn_idx = np.where((pred_ens_tuned == 0) & (y == 1))[0]

top_fp = fp_idx[np.argsort(-oof_ens[fp_idx])][:5] if len(fp_idx) else np.array([], dtype=int)
top_fn = fn_idx[np.argsort(oof_ens[fn_idx])][:5] if len(fn_idx) else np.array([], dtype=int)

top_fp_list = [{"stay_id": int(train_df.iloc[i]["stay_id"]), "true": 0, "pred": 1, "proba": float(oof_ens[i])} for i in top_fp]
top_fn_list = [{"stay_id": int(train_df.iloc[i]["stay_id"]), "true": 1, "pred": 0, "proba": float(oof_ens[i])} for i in top_fn]

# -----------------------------
# Train final models on full data + predict test
# -----------------------------
lr_model.fit(X, y)
gbm_model.fit(X, y)

proba_lr_test = lr_model.predict_proba(X_test)[:, 1]
if hasattr(gbm_model, "predict_proba"):
    proba_gbm_test = gbm_model.predict_proba(X_test)[:, 1]
else:
    dfun = gbm_model.decision_function(X_test)
    proba_gbm_test = 1.0 / (1.0 + np.exp(-dfun))

proba_test = best_w * proba_lr_test + (1.0 - best_w) * proba_gbm_test
pred_test = (proba_test >= best_thr).astype(int)

submission = pd.DataFrame({
    "stay_id": stays_test["stay_id"].astype(int),
    "discharge_ready_day11": pred_test.astype(int)
})

sub_path = OUT_DIR / "clai_ch3_v0_submission.csv"
submission.to_csv(sub_path, index=False)
print(f"\nWrote submission: {sub_path.resolve()} | shape={submission.shape}")
print(submission.head())

# -----------------------------
# Save artifacts
# -----------------------------
model_artifact = {
    "version": "v0",
    "iteration_id": iteration_id,
    "timestamp": timestamp,
    "lr_model": lr_model,
    "gbm_model": gbm_model,
    "blend_lr_weight": best_w,
    "threshold": best_thr,
    "feature_cols": {"num": num_cols, "cat": cat_cols, "text": text_cols},
}

model_path = OUT_DIR / "clai_ch3_v0_model.joblib"
joblib.dump(model_artifact, model_path)

model_iter_path = OUT_DIR / f"clai_ch3_v0_model_iter{iteration_id}.joblib"
joblib.dump(model_artifact, model_iter_path)

# Extract TF-IDF vocab sizes for logging (after fit)
try:
    vocab_all = lr_model.named_steps["pre"].named_transformers_["txt_all"].vocabulary_
    vocab_last3 = lr_model.named_steps["pre"].named_transformers_["txt_last3"].vocabulary_
    tfidf_vocab_sizes = {"notes_concat": int(len(vocab_all)), "notes_last3": int(len(vocab_last3))}
except Exception:
    tfidf_vocab_sizes = {"notes_concat": None, "notes_last3": None}

# Estimate feature counts
try:
    lr_feat_dim = int(lr_model.named_steps["pre"].transform(X.iloc[:10]).shape[1])
except Exception:
    lr_feat_dim = None
try:
    gbm_feat_dim = int(gbm_model.named_steps["pre"].transform(X.iloc[:10]).shape[1])
except Exception:
    gbm_feat_dim = None

# -----------------------------
# Append iteration log (JSONL)
# -----------------------------
log_entry = {
    "version": "v0",
    "iteration_id": iteration_id,
    "timestamp": timestamp,
    "data_paths_used": {
        "base_dir": str(DATA_DIR.resolve()),
        "stays_train": str((DATA_DIR / "stays_train.csv").resolve()),
        "stays_test": str((DATA_DIR / "stays_test.csv").resolve()),
        "patients": str((DATA_DIR / "patients.csv").resolve()),
        "vitals_timeseries": str((DATA_DIR / "vitals_timeseries.json").resolve()),
        "join_keys": {
            "stays<->patients": "patient_id",
            "stays<->vitals_timeseries": "stay_id"
        }
    },
    "feature_summary": {
        "categorical_cols": cat_cols,
        "text_cols": text_cols,
        "numeric_feature_count": int(len(num_cols)),
        "engineered_numeric_notes": [
            "per-signal stats: mean/std/min/max/median/q25/q75/first/last/delta/slope",
            "windowed stats: last3/first3 mean/std + ratio(last3,first3)",
            "derived signals: pulse pressure (pp), MAP (map), shock index (si)",
            "variability: mean/max abs day-to-day change",
            "abnormal-day counts + stable day counts",
            "note length stats + simple keyword counts + TF-IDF on notes (all days + last3)"
        ],
        "tfidf": {
            "ngram_range": [1, 2],
            "min_df": 2,
            "max_features_notes_concat": 15000,
            "max_features_notes_last3": 8000,
            "vocab_size_fit": tfidf_vocab_sizes
        },
        "total_feature_count_est": {"lr_pipeline_dim": lr_feat_dim, "gbm_pipeline_dim": gbm_feat_dim}
    },
    "models_tried": [
        {
            "name": "LogisticRegression(saga, L2) w/ TF-IDF + tabular",
            "hyperparams": {
                "C": 1.0,
                "class_weight": "balanced",
                "max_iter": 5000,
                "tol": 1e-4,
                "random_state": SEED
            }
        },
        {
            "name": "LightGBM (tabular only)",
            "hyperparams": {
                "n_estimators": 300,
                "learning_rate": 0.05,
                "num_leaves": 63,
                "subsample": 0.85,
                "colsample_bytree": 0.85,
                "min_child_samples": 15,
                "reg_lambda": 1.0,
                "random_state": SEED,
                "n_jobs": 1
            }
        },
        {
            "name": "Blended ensemble (OOF-tuned weight + threshold)",
            "hyperparams": {
                "blend_lr_weight": best_w,
                "threshold": best_thr
            }
        }
    ],
    "validation_protocol": {
        "cv": "StratifiedKFold",
        "n_splits": N_SPLITS,
        "shuffle": True,
        "random_state": SEED,
        "macro_f1_definition": "sklearn.metrics.f1_score(average='macro')"
    },
    "metrics": {
        "macro_f1_oof_best": float(best_score),
        "best_threshold": float(best_thr),
        "best_lr_weight": float(best_w),
        "macro_f1_per_fold_thr0.5": [float(x) for x in scores_05],
        "macro_f1_per_fold_tuned": [float(x) for x in scores_tuned],
        "macro_f1_mean_tuned": float(np.mean(scores_tuned)),
        "confusion_matrix_oof_tuned_labels_0_1": cm_tuned.tolist(),
        "class_balance_train": {str(k): int(v) for k, v in pd.Series(y).value_counts().sort_index().items()},
        "pred_positive_rate_oof_tuned": float(pred_ens_tuned.mean())
    },
    "thresholding_strategy": {
        "strategy": "grid search blend weight (lr vs gbm) and decision threshold on out-of-fold probabilities to maximize macro-F1",
        "grid": {
            "lr_weight": {"min": 0.0, "max": 1.0, "num": int(len(weights)), "step": 0.05},
            "threshold": {"min": 0.05, "max": 0.95, "num": int(len(thresholds)), "step": 0.005}
        },
        "best_lr_weight": best_w,
        "best_threshold": best_thr
    },
    "top_errors_failure_analysis": {
        "false_positive_count_oof": int(len(fp_idx)),
        "false_negative_count_oof": int(len(fn_idx)),
        "top_false_positives": top_fp_list,
        "top_false_negatives": top_fn_list
    },
    "next_step_hypothesis": "Try a regularized stacking meta-model on OOF probs (instead of fixed-weight blend) and add a few extra time-series shape features (e.g., last-day stability flags, monotonic trend counts) to reduce leaderboard variance.",
    "leakage_overfitting_warnings_considered": [
        "Used only day1-10 vitals + daily notes (target is readiness at day11).",
        "No joins to admissions_* tables due to lack of reliable stay_id mapping and potential leakage via LOS/outcomes.",
        "Blend weight + threshold tuned ONLY on out-of-fold predictions."
    ],
    "artifacts_written": {
        "submission_csv": str(sub_path.resolve()),
        "model_joblib": str(model_path.resolve()),
        "model_joblib_iter": str(model_iter_path.resolve()),
        "iteration_log_jsonl": str(log_path.resolve())
    }
}

with open(log_path, "a") as f:
    f.write(json.dumps(log_entry) + "\n")

print(f"Saved model artifact: {model_path.resolve()}")
print(f"Saved model artifact (iter): {model_iter_path.resolve()}")
print(f"Appended iteration log: {log_path.resolve()}")

DATA_DIR=D:\AgentDs\agent_ds_healthcare
OUT_DIR=D:\AgentDs\agent_ds_healthcare
Iteration 7 | timestamp=2026-03-01T04:04:29.897764
Train shape: (1000, 196) | Test shape: (1000, 195)
Class balance: {0: np.int64(344), 1: np.int64(656)}
Numeric features: 186 | Categorical: 5 | Text cols: ['notes_concat', 'notes_last3']
Fold 1/5 ...
Fold 2/5 ...
Fold 3/5 ...
Fold 4/5 ...
Fold 5/5 ...

OOF tuning results (ensemble):
  best_macro_f1_oof = 0.822784
  best_lr_weight    = 0.500
  best_threshold    = 0.540
  macro_f1_per_fold_thr0.5  = [0.842891, 0.778702, 0.794872, 0.822649, 0.824684] | mean=0.812760
  macro_f1_per_fold_tuned   = [0.851154, 0.808, 0.796472, 0.830393, 0.826208] | mean=0.822445
  confusion_matrix_oof_tuned [ [TN,FP],[FN,TP] ] = [[249, 95], [61, 595]]

Wrote submission: D:\AgentDs\agent_ds_healthcare\clai_ch3_v0_submission.csv | shape=(1000, 2)
   stay_id  discharge_ready_day11
0      407                      1
1     1594                      1
2     1382                      1
3  

# Iteration 9
- 0.8006

In [19]:
import os, json, random, datetime
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.base import clone

from lightgbm import LGBMClassifier
import joblib

# =========================
# Config (deterministic)
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def resolve_path(fname: str) -> Path:
    """Find data either in current working directory (project) or /mnt/data (sandbox)."""
    for base in [Path("."), Path("/mnt/data")]:
        p = base / fname
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {fname} in . or /mnt/data")

paths = {
    "stays_train": resolve_path("stays_train.csv"),
    "stays_test": resolve_path("stays_test.csv"),
    "patients": resolve_path("patients.csv"),
    "vitals_timeseries": resolve_path("vitals_timeseries.json"),
}

# =========================
# Load tables
# =========================
stays_train = pd.read_csv(paths["stays_train"])
stays_test  = pd.read_csv(paths["stays_test"])
patients    = pd.read_csv(paths["patients"])

with open(paths["vitals_timeseries"], "r") as f:
    vitals_json = json.load(f)

vitals_map = {int(item["stay_id"]): item["days"] for item in vitals_json}

# =========================
# Feature engineering
# =========================
def build_vitals_features(stay_ids):
    """Stay-level aggregation of 10-day vitals + notes (days 1-10 only)."""
    rows = []
    eps = 1e-6
    for sid in stay_ids:
        sid_int = int(sid)
        days = vitals_map.get(sid_int, [])
        days_sorted = sorted(days, key=lambda d: d.get("day", 0))
        day_idx = np.array([d.get("day", np.nan) for d in days_sorted], dtype=float)

        # Core vitals arrays (length 10)
        sigs = {}
        for k in ["hr", "sbp", "dbp", "temp_c", "rr"]:
            sigs[k] = np.array([d.get(k, np.nan) for d in days_sorted], dtype=float)

        # Derived vitals
        sigs["pp"]  = sigs["sbp"] - sigs["dbp"]
        sigs["map"] = (sigs["sbp"] + 2.0 * sigs["dbp"]) / 3.0

        feat = {"stay_id": sid_int}

        for name, arr in sigs.items():
            # Summary stats
            feat[f"{name}_mean"]   = float(np.nanmean(arr))
            feat[f"{name}_std"]    = float(np.nanstd(arr))
            feat[f"{name}_min"]    = float(np.nanmin(arr))
            feat[f"{name}_max"]    = float(np.nanmax(arr))
            feat[f"{name}_median"] = float(np.nanmedian(arr))
            feat[f"{name}_q25"]    = float(np.nanpercentile(arr, 25))
            feat[f"{name}_q75"]    = float(np.nanpercentile(arr, 75))
            feat[f"{name}_first"]  = float(arr[0])
            feat[f"{name}_last"]   = float(arr[-1])
            feat[f"{name}_delta"]  = float(arr[-1] - arr[0])

            # Trend (linear slope)
            feat[f"{name}_slope"] = float(np.polyfit(day_idx, arr, 1)[0])

            # Variability
            dif = np.diff(arr)
            feat[f"{name}_madiff"]  = float(np.nanmean(np.abs(dif)))
            feat[f"{name}_maxdiff"] = float(np.nanmax(np.abs(dif)))

            # Windowed stats
            last3  = arr[-3:]
            first3 = arr[:3]
            feat[f"{name}_last3_mean"]  = float(np.nanmean(last3))
            feat[f"{name}_last3_std"]   = float(np.nanstd(last3))
            feat[f"{name}_first3_mean"] = float(np.nanmean(first3))
            feat[f"{name}_first3_std"]  = float(np.nanstd(first3))
            feat[f"{name}_last3_first3_ratio"] = float(feat[f"{name}_last3_mean"] / (feat[f"{name}_first3_mean"] + eps))

        # Abnormal-day counts (clinically-inspired cutoffs)
        hr, sbp, temp, rr, mp = sigs["hr"], sigs["sbp"], sigs["temp_c"], sigs["rr"], sigs["map"]
        feat["abn_hr"]   = int(np.sum((hr < 60) | (hr > 100)))
        feat["abn_sbp"]  = int(np.sum((sbp < 90) | (sbp > 140)))
        feat["abn_temp"] = int(np.sum((temp < 36.0) | (temp > 37.5)))
        feat["abn_rr"]   = int(np.sum((rr < 12) | (rr > 20)))
        feat["abn_map"]  = int(np.sum((mp < 65) | (mp > 105)))

        feat["stable_days"] = int(np.sum(
            (hr >= 60) & (hr <= 100) &
            (sbp >= 90) & (sbp <= 140) &
            (temp >= 36.0) & (temp <= 37.5) &
            (rr >= 12) & (rr <= 20)
        ))
        feat["any_fever"]       = int(np.any(temp > 37.8))
        feat["any_hypotension"] = int(np.any(sbp < 90))

        # Notes: concatenate all 10 notes + last-3-day notes (text dropped, but its length helps)
        notes = [str(d.get("note", "")) for d in days_sorted]
        feat["notes_all"] = " ".join(notes).strip()
        feat["notes_last3"] = " ".join(notes[-3:]).strip()
        feat["note_len_all"] = int(len(feat["notes_all"]))
        feat["note_len_last3"] = int(len(feat["notes_last3"]))

        # Cheap keyword counts (tree-friendly)
        lower = feat["notes_all"].lower()
        for kw in ["stable", "improv", "ambulat", "walk", "pain", "afebrile", "oxygen", "wean", "discharge", "pt", "ot"]:
            feat[f"kw_{kw}"] = int(lower.count(kw))

        rows.append(feat)

    return pd.DataFrame(rows)

train_vitals = build_vitals_features(stays_train["stay_id"].tolist())
test_vitals  = build_vitals_features(stays_test["stay_id"].tolist())

def build_master(stays_df, vitals_df):
    df = stays_df.merge(vitals_df, on="stay_id", how="left")
    df = df.merge(patients, on="patient_id", how="left")

    # Cast categoricals
    df["zip3"] = df["zip3"].astype(str)
    for c in ["unit_type", "admission_reason", "sex", "insurance"]:
        df[c] = df[c].astype(str)

    for c in ["notes_all", "notes_last3"]:
        df[c] = df[c].fillna("").astype(str)

    return df

train_df = build_master(stays_train, train_vitals)
test_df  = build_master(stays_test,  test_vitals)

y = train_df["discharge_ready_day11"].astype(int).values
X = train_df.drop(columns=["discharge_ready_day11"])

cat_cols = ["unit_type", "admission_reason", "sex", "insurance", "zip3"]
text_cols = ["notes_all", "notes_last3"]
text_col = "notes_all"
id_cols = ["stay_id", "patient_id"]
num_cols = [c for c in X.columns if c not in cat_cols + text_cols + id_cols]

# =========================
# Model: LightGBM on sparse (OHE + TF-IDF + numeric)
# =========================
preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("txt", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=6000), text_col),
    ],
    remainder="drop",
    sparse_threshold=0.2
)

lgbm_params = dict(
    n_estimators=400,
    learning_rate=0.045,
    num_leaves=31,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    min_child_samples=20,
    random_state=SEED,
    n_jobs=1,                    # deterministic
    class_weight="balanced",
    verbose=-1,
)

model = Pipeline([
    ("prep", preprocess),
    ("clf", LGBMClassifier(**lgbm_params))
])

# =========================
# Deterministic CV + OOF threshold tuning (Macro-F1)
# =========================
def tune_threshold(y_true, y_proba, grid):
    best_thr = 0.5
    best_f1 = -1.0
    for thr in grid:
        pred = (y_proba >= thr).astype(int)
        f1 = f1_score(y_true, pred, average="macro")
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)
    return best_thr, float(best_f1)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_proba = np.zeros(len(y), dtype=float)

fold_f1_thr05 = []
fold_cm_thr05 = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), start=1):
    m = clone(model)
    m.fit(X.iloc[tr_idx], y[tr_idx])
    proba = m.predict_proba(X.iloc[va_idx])[:, 1]
    oof_proba[va_idx] = proba

    pred05 = (proba >= 0.5).astype(int)
    fold_f1_thr05.append(float(f1_score(y[va_idx], pred05, average="macro")))
    fold_cm_thr05.append(confusion_matrix(y[va_idx], pred05, labels=[0, 1]).tolist())

thr_grid = np.linspace(0.20, 0.80, 61)  # step=0.01
best_thr, best_oof_macro_f1 = tune_threshold(y, oof_proba, thr_grid)

fold_f1_tuned = []
fold_cm_tuned = []
for fold, (_, va_idx) in enumerate(cv.split(X, y), start=1):
    proba = oof_proba[va_idx]
    predt = (proba >= best_thr).astype(int)
    fold_f1_tuned.append(float(f1_score(y[va_idx], predt, average="macro")))
    fold_cm_tuned.append(confusion_matrix(y[va_idx], predt, labels=[0, 1]).tolist())

cm_oof_tuned = confusion_matrix(y, (oof_proba >= best_thr).astype(int), labels=[0, 1])

print("===== CH3 V0 Iteration (LightGBM sparse + TF-IDF) =====")
print(f"5-fold Stratified CV (random_state={SEED})")
print(f"Macro-F1 per fold @thr=0.50: {fold_f1_thr05} | mean={np.mean(fold_f1_thr05):.4f}")
print(f"Best OOF threshold (grid 0.20..0.80 step 0.01): {best_thr:.2f}")
print(f"Macro-F1 per fold @thr={best_thr:.2f}: {fold_f1_tuned} | mean={np.mean(fold_f1_tuned):.4f}")
print(f"OOF Macro-F1 (tuned thr): {best_oof_macro_f1:.4f}")
print(f"OOF confusion matrix @thr={best_thr:.2f} (labels 0/1):\n{cm_oof_tuned}")

# Error analysis (top confident errors)
oof_pred = (oof_proba >= best_thr).astype(int)
err_idx = np.where(oof_pred != y)[0]
top_fp, top_fn = [], []
if err_idx.size:
    conf = np.abs(oof_proba[err_idx] - y[err_idx])
    worst = err_idx[np.argsort(-conf)]
    for i in worst:
        row = X.iloc[i]
        item = dict(
            stay_id=int(row["stay_id"]),
            y_true=int(y[i]),
            p1=float(oof_proba[i]),
            y_pred=int(oof_pred[i]),
            unit_type=str(row.get("unit_type", "")),
            admission_reason=str(row.get("admission_reason", "")),
            note_snip=str(row.get("notes_all", ""))[:80],
        )
        if item["y_true"] == 0 and item["y_pred"] == 1 and len(top_fp) < 5:
            top_fp.append(item)
        if item["y_true"] == 1 and item["y_pred"] == 0 and len(top_fn) < 5:
            top_fn.append(item)
        if len(top_fp) >= 5 and len(top_fn) >= 5:
            break

print(f"Top FP (n={len(top_fp)}) and FN (n={len(top_fn)}) examples captured for jsonl log.")

# =========================
# Train final model + predict test
# =========================
model.fit(X, y)

# Feature counts for logging (after fit)
prep_fitted = model.named_steps["prep"]
tfidf_vocab_size = len(prep_fitted.named_transformers_["txt"].vocabulary_)
ohe = prep_fitted.named_transformers_["cat"]
ohe_dim = int(sum(len(cats) for cats in ohe.categories_))
numeric_dim = int(len(num_cols))
total_feature_est = int(numeric_dim + ohe_dim + tfidf_vocab_size)

test_X = test_df[X.columns]  # align columns
test_proba = model.predict_proba(test_X)[:, 1]
test_pred = (test_proba >= best_thr).astype(int)

sub_path = Path("clai_ch3_v0_submission.csv")
submission = pd.DataFrame({
    "stay_id": stays_test["stay_id"].astype(int),
    "discharge_ready_day11": test_pred.astype(int),
})
submission.to_csv(sub_path, index=False)

# Save artifact
artifact_path = Path("clai_ch3_v0_model.joblib")
joblib.dump(
    {
        "pipeline": model,
        "threshold": best_thr,
        "num_cols": num_cols,
        "cat_cols": cat_cols,
        "text_col": text_col,
        "lgbm_params": lgbm_params,
    },
    artifact_path
)

print(f"\nWrote submission: {sub_path.resolve()}")
print(f"Wrote model artifact: {artifact_path.resolve()}")

# =========================
# Append iteration log (JSONL, append-only)
# =========================
log_path = Path("clai_ch3_v0_iteration_detail.jsonl")
max_iter = -1
if log_path.exists():
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                max_iter = max(max_iter, int(obj.get("iteration_id", -1)))
            except Exception:
                continue

iteration_id = max_iter + 1
timestamp = datetime.datetime.now().isoformat()

log_obj = {
    "version": "v0",
    "iteration_id": iteration_id,
    "timestamp": timestamp,
    "data_paths_used": {
        "base_dir": str(Path(".").resolve()),
        "stays_train": str(paths["stays_train"]),
        "stays_test": str(paths["stays_test"]),
        "patients": str(paths["patients"]),
        "vitals_timeseries": str(paths["vitals_timeseries"]),
        "join_keys": {
            "stays<->patients": "patient_id",
            "stays<->vitals_timeseries": "stay_id",
        },
    },
    "feature_summary": {
        "categorical_cols": cat_cols,
        "text_col": text_col,
        "numeric_feature_count": numeric_dim,
        "tfidf": {
            "ngram_range": [1, 2],
            "min_df": 2,
            "max_features": 6000,
            "vocab_size_fit_full_train": tfidf_vocab_size,
        },
        "engineered_numeric_notes": [
            "per-signal stats: mean/std/min/max/median/q25/q75/first/last/delta/slope",
            "variability: mean abs diff + max abs diff",
            "windowed stats: last3 mean/std + ratio(last3_mean/first3_mean)",
            "derived signals: pulse pressure (pp) and mean arterial pressure (map)",
            "abnormal-day counts + stable day count + fever/hypotension flags",
            "notes: TF-IDF on concatenated 10-day notes + note lengths (all vs last3) + keyword substring counts",
        ],
        "total_feature_count_est": total_feature_est,
    },
    "models_tried": [
        {
            "name": "LightGBM(LGBMClassifier) with sparse features",
            "hyperparams": lgbm_params,
        }
    ],
    "validation_protocol": {
        "cv": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": SEED,
    },
    "metrics": {
        "class_balance_train": {str(k): int(v) for k, v in pd.Series(y).value_counts().to_dict().items()},
        "macro_f1_per_fold_thr0.5": fold_f1_thr05,
        "macro_f1_per_fold_tuned": fold_f1_tuned,
        "macro_f1_mean_tuned": float(np.mean(fold_f1_tuned)),
        "macro_f1_oof_best_threshold": best_oof_macro_f1,
        "best_threshold": best_thr,
        "confusion_matrix_oof_tuned_labels_0_1": cm_oof_tuned.tolist(),
        "pred_positive_rate_oof": float(np.mean(oof_pred)),
    },
    "thresholding_strategy": {
        "method": "grid-search on OOF probabilities to maximize Macro-F1",
        "grid": {"start": 0.20, "end": 0.80, "num": 61},
        "selected_threshold": best_thr,
    },
    "top_errors_failure_analysis": {
        "n_errors_oof": int(err_idx.size),
        "top_false_positives": top_fp,
        "top_false_negatives": top_fn,
        "comment": "Review note snippets + late-window vitals for consistent FP/FN patterns.",
    },
    "next_step_hypothesis": "Try a small weighted blend with a calibrated linear model (logreg/SGD) to reduce variance; optionally add char-ngrams if CV improves without harming LB.",
    "leakage_overfitting_warnings_considered": [
        "Used only vitals + progress notes from days 1-10 (no day11+ information).",
        "Did NOT use discharge_notes.json (likely post-discharge leakage).",
        "All text vectorizers and encoders fit within CV folds (no leakage across folds).",
    ],
    "artifacts_written": {
        "submission_csv": str(sub_path.resolve()),
        "model_joblib": str(artifact_path.resolve()),
        "iteration_log_jsonl": str(log_path.resolve()),
    },
}

with open(log_path, "a", encoding="utf-8") as f:
    f.write(json.dumps(log_obj) + "\n")

print(f"Appended iteration log: {log_path.resolve()} (iteration_id={iteration_id})")

===== CH3 V0 Iteration (LightGBM sparse + TF-IDF) =====
5-fold Stratified CV (random_state=42)
Macro-F1 per fold @thr=0.50: [0.8404558404558404, 0.754631333578702, 0.7750281214848144, 0.8176638176638177, 0.8333333333333334] | mean=0.8042
Best OOF threshold (grid 0.20..0.80 step 0.01): 0.60
Macro-F1 per fold @thr=0.60: [0.8451156101338644, 0.7877108149545937, 0.8141420968150714, 0.82174688057041, 0.83779399837794] | mean=0.8213
OOF Macro-F1 (tuned thr): 0.8218
OOF confusion matrix @thr=0.60 (labels 0/1):
[[249  95]
 [ 62 594]]
Top FP (n=5) and FN (n=5) examples captured for jsonl log.

Wrote submission: D:\AgentDs\agent_ds_healthcare\clai_ch3_v0_submission.csv
Wrote model artifact: D:\AgentDs\agent_ds_healthcare\clai_ch3_v0_model.joblib
Appended iteration log: D:\AgentDs\agent_ds_healthcare\clai_ch3_v0_iteration_detail.jsonl (iteration_id=8)


# Iteration 10
- 0.8067

In [21]:
# clai_ch3_v0 - Iteration: blended LightGBM(tabular) + Logistic(TF-IDF+tabular) with OOF-tuned weight+threshold
import os, json, re, joblib, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix

from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

def resolve_path(fname: str) -> Path:
    p = Path(fname)
    if p.exists():
        return p
    p2 = Path("/mnt/data") / fname
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Could not find {fname} in current dir or /mnt/data")

# ------------------------
# Paths
# ------------------------
stays_train_path = resolve_path("stays_train.csv")
stays_test_path  = resolve_path("stays_test.csv")
patients_path    = resolve_path("patients.csv")
vitals_path      = resolve_path("vitals_timeseries.json")

submission_path = Path("clai_ch3_v0_submission.csv")
log_path = Path("clai_ch3_v0_iteration_detail.jsonl")
model_path = Path("clai_ch3_v0_model.joblib")

# ------------------------
# Iteration ID (append-only log)
# ------------------------
def next_iteration_id(jsonl_path: Path) -> int:
    if not jsonl_path.exists():
        return 0
    mx = -1
    with jsonl_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                mx = max(mx, int(obj.get("iteration_id", -1)))
            except Exception:
                # tolerate malformed / partial lines
                continue
    return mx + 1

ITER_ID = next_iteration_id(log_path)

# ------------------------
# Load data
# ------------------------
stays_train = pd.read_csv(stays_train_path)
stays_test  = pd.read_csv(stays_test_path)
patients    = pd.read_csv(patients_path)

with open(vitals_path, "r", encoding="utf-8") as f:
    vitals = json.load(f)

target_col = "discharge_ready_day11"
cat_cols = ["unit_type", "admission_reason", "sex", "insurance", "zip3"]
text_cols = ["notes_all", "notes_last3"]

# ------------------------
# Feature engineering (fast, deterministic)
# ------------------------
def safe_slope(x: np.ndarray, y: np.ndarray) -> float:
    mask = ~np.isnan(y)
    if mask.sum() < 2:
        return np.nan
    xm = x[mask]
    ym = y[mask]
    x_mean = xm.mean()
    y_mean = ym.mean()
    denom = ((xm - x_mean) ** 2).sum()
    if denom == 0:
        return 0.0
    return float(((xm - x_mean) * (ym - y_mean)).sum() / denom)

def vitals_features_simplified(vitals_list):
    # curated keyword list (compact, higher-signal)
    keywords = [
        "awaiting placement", "placement", "rehab", "home health", "discharge",
        "pt", "ot", "walker", "cane", "stairs", "ambulat", "out of bed", "chair",
        "pain", "tolerating diet", "no new issues", "vitals stable", "afebrile",
        "fever", "weaned o2", "room air", "oxygen", "wound", "incision",
    ]
    patterns = []
    for k in keywords:
        if k == "ambulat":
            pat = re.compile(r"\bambulat\w*", re.IGNORECASE)
        elif k in ["pt", "ot"]:
            pat = re.compile(r"\b" + k + r"\b", re.IGNORECASE)
        else:
            pat = re.compile(re.escape(k), re.IGNORECASE)
        patterns.append((k, pat))

    feats = []
    for item in vitals_list:
        stay_id = item["stay_id"]
        days = sorted(item["days"], key=lambda d: d.get("day", 0))
        day_idx = np.array([d.get("day", np.nan) for d in days], dtype=float)

        row = {"stay_id": stay_id}

        signals = {}
        for sig in ["hr", "sbp", "dbp", "temp_c", "rr"]:
            signals[sig] = np.array(
                [d.get(sig, np.nan) if d.get(sig) is not None else np.nan for d in days],
                dtype=float,
            )

        sbp = signals["sbp"]
        dbp = signals["dbp"]
        signals["pp"] = sbp - dbp
        signals["map"] = dbp + (sbp - dbp) / 3.0

        for sig, arr in signals.items():
            row[f"{sig}_mean"] = float(np.nanmean(arr))
            row[f"{sig}_std"] = float(np.nanstd(arr))
            row[f"{sig}_min"] = float(np.nanmin(arr))
            row[f"{sig}_max"] = float(np.nanmax(arr))

            if np.all(np.isnan(arr)):
                first = last = np.nan
            else:
                obs = arr[~np.isnan(arr)]
                first = float(obs[0])
                last = float(obs[-1])

            row[f"{sig}_first"] = first
            row[f"{sig}_last"] = last
            row[f"{sig}_delta"] = (last - first) if (not np.isnan(last) and not np.isnan(first)) else np.nan
            row[f"{sig}_slope"] = safe_slope(day_idx, arr)

            first3 = arr[:3]
            last3 = arr[-3:]
            row[f"{sig}_first3_mean"] = float(np.nanmean(first3))
            row[f"{sig}_last3_mean"] = float(np.nanmean(last3))
            row[f"{sig}_first3_std"] = float(np.nanstd(first3))
            row[f"{sig}_last3_std"] = float(np.nanstd(last3))
            denom = row[f"{sig}_first3_mean"]
            row[f"{sig}_last3_over_first3"] = (row[f"{sig}_last3_mean"] / denom) if (denom != 0 and not np.isnan(denom)) else np.nan
            row[f"{sig}_missing_count"] = int(np.isnan(arr).sum())

        temp = signals["temp_c"]
        hr = signals["hr"]
        rr = signals["rr"]
        sbp = signals["sbp"]

        row["fever_days"] = int(np.nansum(temp >= 37.8))
        row["hypothermia_days"] = int(np.nansum(temp < 36.0))
        row["tachycardia_days"] = int(np.nansum(hr > 100))
        row["bradycardia_days"] = int(np.nansum(hr < 60))
        row["hypotension_days"] = int(np.nansum(sbp < 90))
        row["hypertension_days"] = int(np.nansum(sbp > 160))
        row["tachypnea_days"] = int(np.nansum(rr > 20))
        row["bradypnea_days"] = int(np.nansum(rr < 12))

        stable_mask = (
            (~np.isnan(hr)) & (hr >= 60) & (hr <= 100) &
            (~np.isnan(sbp)) & (sbp >= 90) & (sbp <= 160) &
            (~np.isnan(temp)) & (temp >= 36.0) & (temp <= 37.8) &
            (~np.isnan(rr)) & (rr >= 12) & (rr <= 20)
        )
        row["stable_days"] = int(stable_mask.sum())

        row["day10_fever"] = int((not np.isnan(temp[-1])) and temp[-1] >= 37.8)
        row["day10_tachy"] = int((not np.isnan(hr[-1])) and hr[-1] > 100)
        row["day10_hypotension"] = int((not np.isnan(sbp[-1])) and sbp[-1] < 90)
        row["day10_tachypnea"] = int((not np.isnan(rr[-1])) and rr[-1] > 20)

        notes = [(d.get("note") or "") for d in days]
        notes_all = " ".join(notes).lower()
        notes_last3 = " ".join(notes[-3:]).lower()
        row["notes_all"] = notes_all
        row["notes_last3"] = notes_last3

        lens = np.array([len(n) for n in notes], dtype=float)
        row["note_len_mean"] = float(lens.mean())
        row["note_len_std"] = float(lens.std())
        row["note_len_last"] = float(lens[-1] if len(lens) > 0 else 0.0)
        row["note_len_last3_mean"] = float(lens[-3:].mean() if len(lens) >= 3 else lens.mean())

        for k, pat in patterns:
            key = re.sub(r"[^a-z0-9]+", "_", k.lower()).strip("_")
            row[f"kw_{key}_all"] = int(len(pat.findall(notes_all)))
            row[f"kw_{key}_last3"] = int(len(pat.findall(notes_last3)))

        feats.append(row)

    return pd.DataFrame(feats)

feat_df = vitals_features_simplified(vitals)

train = stays_train.merge(patients, on="patient_id", how="left").merge(feat_df, on="stay_id", how="left")
test  = stays_test.merge(patients, on="patient_id", how="left").merge(feat_df, on="stay_id", how="left")

train["notes_combo"] = (train["notes_all"].fillna("") + " " + train["notes_last3"].fillna("")).str.strip()
test["notes_combo"]  = (test["notes_all"].fillna("") + " " + test["notes_last3"].fillna("")).str.strip()

y = train[target_col].astype(int).values

id_cols = ["stay_id", "patient_id"]
drop_for_num = id_cols + [target_col] + cat_cols + ["notes_all", "notes_last3", "notes_combo"]
num_cols = [c for c in train.columns if c not in drop_for_num]

# ------------------------
# Models (deterministic)
# ------------------------
tab_preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

tab_model = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    min_child_samples=20,
    random_state=SEED,
    n_jobs=1,
    deterministic=True,
    force_col_wise=True,
    verbose=-1,
)
tab_clf = Pipeline([("pre", tab_preprocess), ("clf", tab_model)])

lin_preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("sc", StandardScaler(with_mean=False))]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("txt", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=20000,
            lowercase=True,
            strip_accents="unicode"
        ), "notes_combo"),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

lin_model = LogisticRegression(
    solver="saga",
    C=2.0,
    max_iter=5000,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=1,
)

lin_clf = Pipeline([("pre", lin_preprocess), ("clf", lin_model)])

# ------------------------
# Deterministic CV + OOF
# ------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_tab = np.zeros(len(train), dtype=float)
oof_lin = np.zeros(len(train), dtype=float)
fold_valid_indices = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train, y)):
    X_tr = train.iloc[tr_idx]
    y_tr = y[tr_idx]
    X_va = train.iloc[va_idx]

    tab_clf.fit(X_tr, y_tr)
    lin_clf.fit(X_tr, y_tr)

    oof_tab[va_idx] = tab_clf.predict_proba(X_va)[:, 1]
    oof_lin[va_idx] = lin_clf.predict_proba(X_va)[:, 1]
    fold_valid_indices.append(va_idx)

# ------------------------
# Tune blend weight + threshold on OOF to maximize macro-F1
# ------------------------
ws = np.linspace(0.55, 0.85, 61)   # focus around tab-dominant blends
ths = np.linspace(0.50, 0.80, 301)

best_f1 = -1.0
best_w = 0.70
best_t = 0.63

for w in ws:
    p = w * oof_tab + (1.0 - w) * oof_lin
    for t in ths:
        f1 = f1_score(y, (p >= t).astype(int), average="macro")
        if f1 > best_f1:
            best_f1 = f1
            best_w = float(w)
            best_t = float(t)

p_ens = best_w * oof_tab + (1.0 - best_w) * oof_lin
pred_ens = (p_ens >= best_t).astype(int)

cm = confusion_matrix(y, pred_ens, labels=[0, 1])

per_fold_f1_tuned = []
per_fold_f1_thr05 = []
for va_idx in fold_valid_indices:
    y_va = y[va_idx]
    p_va = p_ens[va_idx]
    per_fold_f1_tuned.append(float(f1_score(y_va, (p_va >= best_t).astype(int), average="macro")))
    per_fold_f1_thr05.append(float(f1_score(y_va, (p_va >= 0.5).astype(int), average="macro")))

print("===== V0 Iteration (blend) =====")
print(f"OOF Macro-F1 (best): {best_f1:.6f}")
print(f"Best blend weight (tab): {best_w:.3f} | Best threshold: {best_t:.3f}")
print(f"Per-fold Macro-F1 @thr=0.5:  {np.round(per_fold_f1_thr05, 6).tolist()}")
print(f"Per-fold Macro-F1 @tuned:    {np.round(per_fold_f1_tuned, 6).tolist()}")
print(f"Confusion matrix (labels [0,1]) @tuned:\n{cm}")

# Error analysis (top FP/FN)
stay_ids = train["stay_id"].values
fp_idx = np.where((y == 0) & (pred_ens == 1))[0]
fn_idx = np.where((y == 1) & (pred_ens == 0))[0]
top_fp = fp_idx[np.argsort(-p_ens[fp_idx])][:5]
top_fn = fn_idx[np.argsort(p_ens[fn_idx])][:5]

top_fp_list = [{"stay_id": int(stay_ids[i]), "true": 0, "proba": float(p_ens[i]), "pred": 1} for i in top_fp]
top_fn_list = [{"stay_id": int(stay_ids[i]), "true": 1, "proba": float(p_ens[i]), "pred": 0} for i in top_fn]

err_df = train[["stay_id", "admission_reason", "unit_type"]].copy()
err_df["y"] = y
err_df["pred"] = pred_ens
err_df["err"] = (err_df["y"] != err_df["pred"]).astype(int)
err_by_reason = (err_df.groupby("admission_reason")["err"].mean().sort_values(ascending=False)).to_dict()

# ------------------------
# Train final models on full training + predict test
# ------------------------
tab_clf.fit(train, y)
lin_clf.fit(train, y)

p_tab_test = tab_clf.predict_proba(test)[:, 1]
p_lin_test = lin_clf.predict_proba(test)[:, 1]
p_test = best_w * p_tab_test + (1.0 - best_w) * p_lin_test
pred_test = (p_test >= best_t).astype(int)

submission = pd.DataFrame({
    "stay_id": stays_test["stay_id"].astype(int),
    "discharge_ready_day11": pred_test.astype(int),
})
submission.to_csv(submission_path, index=False)

# ------------------------
# Save model artifact
# ------------------------
artifact = {
    "version": "v0",
    "seed": SEED,
    "iteration_id": ITER_ID,
    "blend_weight_tab": best_w,
    "threshold": best_t,
    "tab_model": tab_clf,
    "lin_model": lin_clf,
    "feature_version": "simp_kw24_all_last3 + TFIDF(1,2) + tabular",
    "columns": {
        "num_cols": num_cols,
        "cat_cols": cat_cols,
        "text_used": ["notes_combo"],
    }
}
joblib.dump(artifact, model_path)
joblib.dump(artifact, Path(f"clai_ch3_v0_model_iter{ITER_ID}.joblib"))

# ------------------------
# Append iteration log (JSONL)
# ------------------------
log_entry = {
    "version": "v0",
    "iteration_id": ITER_ID,
    "timestamp": datetime.utcnow().isoformat(),
    "data_paths_used": {
        "stays_train": str(stays_train_path),
        "stays_test": str(stays_test_path),
        "patients": str(patients_path),
        "vitals_timeseries": str(vitals_path),
    },
    "join_keys": {
        "stays<->patients": "patient_id",
        "stays<->vitals_timeseries": "stay_id",
    },
    "feature_summary": {
        "categorical_cols": cat_cols,
        "numeric_feature_count": int(len(num_cols)),
        "text_features": {
            "notes_combo": "notes_all + notes_last3",
            "tfidf": {"ngram_range": [1, 2], "min_df": 2, "max_features": 20000},
        },
        "engineered_numeric_vitals": [
            "per-signal: mean/std/min/max/first/last/delta/slope",
            "first3/last3 mean/std + last3/first3 ratio",
            "derived signals: pulse pressure (pp), MAP (map)",
            "abnormal-day counts + stable-days",
            "day10 abnormal flags",
        ],
        "keyword_features": {
            "kw_count": 24,
            "per_keyword": ["count_in_all_days", "count_in_last3_days"],
        },
    },
    "models_tried": [
        {
            "name": "LGBMClassifier(tabular)",
            "hyperparams": {
                "n_estimators": 800,
                "learning_rate": 0.05,
                "num_leaves": 31,
                "subsample": 0.85,
                "colsample_bytree": 0.85,
                "reg_lambda": 1.0,
                "min_child_samples": 20,
                "random_state": SEED,
                "n_jobs": 1,
                "deterministic": True,
                "force_col_wise": True,
            },
        },
        {
            "name": "LogisticRegression(TFIDF+tabular)",
            "hyperparams": {
                "solver": "saga",
                "C": 2.0,
                "max_iter": 5000,
                "class_weight": "balanced",
                "random_state": SEED,
                "n_jobs": 1,
            },
        },
        {
            "name": "BlendedEnsemble",
            "hyperparams": {"blend_weight_tab": best_w, "blend_weight_lin": float(1.0 - best_w)},
        },
    ],
    "validation_protocol": {
        "cv": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": SEED,
        "metric": "macro_f1 (sklearn.metrics.f1_score average='macro')",
    },
    "metrics": {
        "macro_f1_oof_best": float(best_f1),
        "best_threshold": float(best_t),
        "best_blend_weight_tab": float(best_w),
        "macro_f1_per_fold_thr0.5": per_fold_f1_thr05,
        "macro_f1_per_fold_tuned": per_fold_f1_tuned,
        "macro_f1_mean_tuned": float(np.mean(per_fold_f1_tuned)),
        "confusion_matrix_oof_tuned_labels_0_1": cm.tolist(),
        "class_balance_train": {str(k): int(v) for k, v in pd.Series(y).value_counts().sort_index().to_dict().items()},
        "pred_positive_rate_oof": float(pred_ens.mean()),
    },
    "thresholding_strategy": {
        "strategy": "grid search blend weight + threshold on out-of-fold probabilities to maximize macro-F1",
        "weight_grid": {"min": 0.55, "max": 0.85, "num": 61},
        "threshold_grid": {"min": 0.50, "max": 0.80, "num": 301},
        "selected": {"blend_weight_tab": float(best_w), "threshold": float(best_t)},
    },
    "top_errors_failure_analysis": {
        "false_positive_count_oof": int(len(fp_idx)),
        "false_negative_count_oof": int(len(fn_idx)),
        "top_false_positives": top_fp_list,
        "top_false_negatives": top_fn_list,
        "error_rate_by_admission_reason": {k: float(v) for k, v in err_by_reason.items()},
    },
    "next_step_hypothesis": "Try a 3-way ensemble (add XGBoost/CatBoost tabular-only) or add low-rank SVD note embeddings into the tree model; re-tune blend+threshold on OOF for a small but consistent gain.",
    "leakage_overfitting_warnings_considered": [
        "Only used day1-10 vitals + daily notes, plus demographics and unit/admission_reason (no post-discharge notes).",
        "Threshold and blend weights tuned strictly on OOF predictions (avoids in-fold leakage).",
        "Each stay_id maps to a unique patient_id in training (no patient overlap across folds).",
    ],
    "artifacts_written": {
        "submission_csv": str(submission_path.resolve()),
        "model_joblib": str(model_path.resolve()),
        "model_joblib_iter": str(Path(f"clai_ch3_v0_model_iter{ITER_ID}.joblib").resolve()),
        "iteration_log_jsonl": str(log_path.resolve()),
    },
}

with log_path.open("a", encoding="utf-8") as f:
    f.write(json.dumps(log_entry) + "\n")

print("\n===== Files written =====")
print(f"Submission: {submission_path.resolve()}")
print(f"Model:      {model_path.resolve()}")
print(f"Log:        {log_path.resolve()}")

===== V0 Iteration (blend) =====
OOF Macro-F1 (best): 0.826923
Best blend weight (tab): 0.825 | Best threshold: 0.637
Per-fold Macro-F1 @thr=0.5:  [0.841701, 0.790664, 0.811207, 0.809639, 0.82655]
Per-fold Macro-F1 @tuned:    [0.852132, 0.817664, 0.839028, 0.801356, 0.822649]
Confusion matrix (labels [0,1]) @tuned:
[[257  87]
 [ 67 589]]

===== Files written =====
Submission: D:\AgentDs\agent_ds_healthcare\clai_ch3_v0_submission.csv
Model:      D:\AgentDs\agent_ds_healthcare\clai_ch3_v0_model.joblib
Log:        D:\AgentDs\agent_ds_healthcare\clai_ch3_v0_iteration_detail.jsonl


# Iteration 11
- 0.7915

In [23]:
# clai_ch3_v0 (V0 baseline) — Iteration: stronger vitals + 2-model ensemble (LogReg TFIDF + LGBM SVD) + OOF-tuned blend & threshold
import os, json, math, random, warnings
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MaxAbsScaler, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

# ----------------------------
# Reproducibility
# ----------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# ----------------------------
# Path resolution (works in both local project dir and this sandbox)
# ----------------------------
BASE_DIR = Path(".")
ALT_DIR = Path("/mnt/data")

def resolve_file(fname: str) -> Path:
    p1 = BASE_DIR / fname
    if p1.exists():
        return p1
    p2 = ALT_DIR / fname
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Could not find {fname} in {BASE_DIR.resolve()} or {ALT_DIR}")

stays_train_path = resolve_file("stays_train.csv")
stays_test_path  = resolve_file("stays_test.csv")
patients_path    = resolve_file("patients.csv")
vitals_path      = resolve_file("vitals_timeseries.json")

# ----------------------------
# Load data
# ----------------------------
stays_train = pd.read_csv(stays_train_path)
stays_test  = pd.read_csv(stays_test_path)
patients    = pd.read_csv(patients_path)
with open(vitals_path, "r", encoding="utf-8") as f:
    vitals_json = json.load(f)

TARGET_COL = "discharge_ready_day11"

# ----------------------------
# Feature extraction from vitals_timeseries.json
# ----------------------------
VITALS = ["hr", "sbp", "dbp", "temp_c", "rr"]
KEYWORDS = [
    "pt", "ambulat", "afebrile", "pain", "n/v", "tolerat", "discharg", "home",
    "stable", "walk", "oob", "chair", "family", "wbc", "telemetry", "void",
    "spirometer", "bowel", "nausea", "vomit", "sob", "baseline", "improv"
]

def _safe_float_arr(vals, fallback_len=10):
    arr = np.array(vals, dtype=np.float32)
    if arr.size == 0:
        arr = np.full((fallback_len,), np.nan, dtype=np.float32)
    if np.all(np.isnan(arr)):
        return np.zeros_like(arr, dtype=np.float32)
    m = np.nanmean(arr)
    return np.where(np.isnan(arr), m, arr).astype(np.float32)

def _slope(arr: np.ndarray) -> float:
    n = arr.size
    if n < 2:
        return 0.0
    t = np.arange(1, n + 1, dtype=np.float32)
    t_mean = float(t.mean())
    a_mean = float(arr.mean())
    denom = float(((t - t_mean) ** 2).sum())
    if denom == 0.0:
        return 0.0
    return float(((t - t_mean) * (arr - a_mean)).sum() / denom)

def _mean_abs_diff(arr: np.ndarray) -> float:
    if arr.size < 2:
        return 0.0
    return float(np.mean(np.abs(np.diff(arr))))

def _stats(prefix: str, arr: np.ndarray, feats: dict):
    feats[f"{prefix}_mean"] = float(np.mean(arr))
    feats[f"{prefix}_std"]  = float(np.std(arr))
    feats[f"{prefix}_min"]  = float(np.min(arr))
    feats[f"{prefix}_max"]  = float(np.max(arr))
    feats[f"{prefix}_p10"]  = float(np.quantile(arr, 0.10))
    feats[f"{prefix}_p90"]  = float(np.quantile(arr, 0.90))
    feats[f"{prefix}_first"]= float(arr[0])
    feats[f"{prefix}_last"] = float(arr[-1])
    feats[f"{prefix}_delta"]= float(arr[-1] - arr[0])
    feats[f"{prefix}_slope"]= _slope(arr)
    feats[f"{prefix}_madiff"]= _mean_abs_diff(arr)
    last3 = arr[-3:] if arr.size >= 3 else arr
    feats[f"{prefix}_last3_mean"] = float(np.mean(last3))
    feats[f"{prefix}_last3_std"]  = float(np.std(last3))
    feats[f"{prefix}_last3_delta"]= float(last3[-1] - last3[0]) if last3.size >= 2 else 0.0

def extract_vitals_features(vitals_list):
    rows = []
    for item in vitals_list:
        sid = item.get("stay_id")
        days = item.get("days", [])
        days = sorted(days, key=lambda d: d.get("day", 0))

        feats = {"stay_id": sid}

        notes = [(d.get("note") or "") for d in days]
        note_all = " ".join(notes).strip()
        note_last3 = " ".join(notes[-3:]).strip() if len(notes) else ""
        feats["note_all"] = note_all
        feats["note_last3"] = note_last3
        feats["note_len_all"] = int(len(note_all))
        feats["note_len_last3"] = int(len(note_last3))
        feats["note_words_all"] = int(len(note_all.split())) if note_all else 0
        feats["note_words_last3"] = int(len(note_last3.split())) if note_last3 else 0

        lower = note_all.lower()
        for kw in KEYWORDS:
            col = f"kw_{kw.replace('/','_').replace(' ','_')}"
            feats[col] = int(lower.count(kw))

        # base vitals
        arrays = {}
        for v in VITALS:
            arrays[v] = _safe_float_arr([d.get(v, np.nan) for d in days], fallback_len=10)
            _stats(v, arrays[v], feats)

        # derived
        sbp = arrays["sbp"]
        dbp = arrays["dbp"]
        hr  = arrays["hr"]
        eps = 1e-6

        map_arr = (sbp + 2.0 * dbp) / 3.0
        pp_arr  = sbp - dbp
        shock   = hr / (sbp + eps)

        _stats("map", map_arr, feats)
        _stats("pp", pp_arr, feats)
        _stats("shock", shock, feats)

        # abnormality counts + last flags (simple clinically-plausible thresholds)
        feats["tachy_hr_cnt"]   = int(np.sum(hr > 100))
        feats["brady_hr_cnt"]   = int(np.sum(hr < 60))
        feats["hypo_sbp_cnt"]   = int(np.sum(sbp < 90))
        feats["hyper_sbp_cnt"]  = int(np.sum(sbp > 160))
        feats["fever_cnt"]      = int(np.sum(arrays["temp_c"] >= 38.0))
        feats["hypotherm_cnt"]  = int(np.sum(arrays["temp_c"] < 36.0))
        feats["tachyp_rr_cnt"]  = int(np.sum(arrays["rr"] > 20))
        feats["bradyp_rr_cnt"]  = int(np.sum(arrays["rr"] < 12))
        feats["low_map_cnt"]    = int(np.sum(map_arr < 65))
        feats["high_shock_cnt"] = int(np.sum(shock > 0.9))

        feats["tachy_hr_last"]  = int(hr[-1] > 100)
        feats["hypo_sbp_last"]  = int(sbp[-1] < 90)
        feats["fever_last"]     = int(arrays["temp_c"][-1] >= 38.0)
        feats["tachyp_rr_last"] = int(arrays["rr"][-1] > 20)
        feats["low_map_last"]   = int(map_arr[-1] < 65)
        feats["high_shock_last"]= int(shock[-1] > 0.9)

        rows.append(feats)

    return pd.DataFrame(rows)

vitals_df = extract_vitals_features(vitals_json)

# ----------------------------
# Merge tables
# ----------------------------
train_df = stays_train.merge(patients, on="patient_id", how="left").merge(vitals_df, on="stay_id", how="left")
test_df  = stays_test.merge(patients, on="patient_id", how="left").merge(vitals_df, on="stay_id", how="left")

# Fill missing text
for col in ["note_all", "note_last3"]:
    train_df[col] = train_df[col].fillna("")
    test_df[col]  = test_df[col].fillna("")

# Define columns
cat_cols = ["unit_type", "admission_reason", "sex", "insurance", "zip3"]
text_cols_A = ["note_all", "note_last3"]
id_cols = ["stay_id", "patient_id"]

for c in cat_cols:
    train_df[c] = train_df[c].astype(str).fillna("UNK")
    test_df[c]  = test_df[c].astype(str).fillna("UNK")

exclude = set(id_cols + [TARGET_COL] + cat_cols + text_cols_A)
num_cols = [c for c in train_df.columns if c not in exclude]

X = train_df.drop(columns=[TARGET_COL])
y = train_df[TARGET_COL].astype(int).values

print("Data shapes:")
print("  train_df:", train_df.shape, "| test_df:", test_df.shape)
print("Columns:")
print("  categorical:", cat_cols)
print("  text:", text_cols_A)
print("  numeric feature count:", len(num_cols))

# ----------------------------
# Helpers: OneHotEncoder compatibility (sklearn versions)
# ----------------------------
def make_ohe(sparse_out: bool):
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=sparse_out)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=sparse_out)

# ----------------------------
# Model A: Sparse Linear (TF-IDF word+char) + structured
# ----------------------------
tfidf_word_all = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=8000,
    sublinear_tf=True,
    strip_accents="unicode"
)
tfidf_char_all = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    max_features=5000,
    sublinear_tf=True
)
tfidf_word_last3 = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=4000,
    sublinear_tf=True,
    strip_accents="unicode"
)

preprocess_A = ColumnTransformer(
    transformers=[
        ("w_all", tfidf_word_all, "note_all"),
        ("c_all", tfidf_char_all, "note_all"),
        ("w_l3",  tfidf_word_last3, "note_last3"),
        ("cat",   make_ohe(True), cat_cols),
        ("num",   SimpleImputer(strategy="median"), num_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

clf_A = LogisticRegression(
    solver="liblinear",
    penalty="l2",
    C=3.0,
    class_weight="balanced",
    max_iter=6000,
    random_state=SEED,
    dual=True
)

pipe_A = Pipeline([
    ("prep", preprocess_A),
    ("scale", MaxAbsScaler()),
    ("clf", clf_A)
])

# ----------------------------
# Model B: LightGBM on dense structured + SVD embeddings of notes
# ----------------------------
try:
    from lightgbm import LGBMClassifier
    LGBM_OK = True
except Exception as e:
    LGBM_OK = False
    print("WARNING: LightGBM not available; fallback will be used. Error:", e)

# class weights (make class-0 heavier since it's minority)
cnt = Counter(y.tolist())
w0 = len(y) / (2.0 * cnt[0])
w1 = len(y) / (2.0 * cnt[1])
lgbm_class_weight = {0: float(w0), 1: float(w1)}

text_svd_all = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=10000,
        sublinear_tf=True,
        strip_accents="unicode"
    )),
    ("svd", TruncatedSVD(n_components=120, random_state=SEED)),
    ("sc", StandardScaler())
])

text_svd_last3 = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=6000,
        sublinear_tf=True,
        strip_accents="unicode"
    )),
    ("svd", TruncatedSVD(n_components=50, random_state=SEED)),
    ("sc", StandardScaler())
])

preprocess_B = ColumnTransformer(
    transformers=[
        ("cat", make_ohe(False), cat_cols),
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("txt_all", text_svd_all, "note_all"),
        ("txt_l3",  text_svd_last3, "note_last3"),
    ],
    remainder="drop",
    sparse_threshold=0.0  # force dense
)

if LGBM_OK:
    clf_B = LGBMClassifier(
        n_estimators=900,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        min_child_samples=20,
        random_state=SEED,
        n_jobs=1,
        deterministic=True,
        force_col_wise=True,
        class_weight=lgbm_class_weight,
        verbosity=-1,
    )
else:
    # deterministic fallback (usually weaker than LGBM)
    from sklearn.ensemble import HistGradientBoostingClassifier
    clf_B = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=6,
        max_iter=300,
        random_state=SEED
    )

pipe_B = Pipeline([
    ("prep", preprocess_B),
    ("clf", clf_B)
])

# ----------------------------
# Deterministic CV + OOF predictions
# ----------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_A = np.zeros(len(X), dtype=np.float32)
oof_B = np.zeros(len(X), dtype=np.float32)

fold_indices = []
print("\nRunning 5-fold CV (deterministic)...")
for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y)):
    X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
    X_va, y_va = X.iloc[va_idx], y[va_idx]

    mA = clone(pipe_A)
    mB = clone(pipe_B)

    mA.fit(X_tr, y_tr)
    mB.fit(X_tr, y_tr)

    oof_A[va_idx] = mA.predict_proba(X_va)[:, 1].astype(np.float32)
    # Some sklearn fallbacks might not have predict_proba; guard
    if hasattr(mB, "predict_proba"):
        oof_B[va_idx] = mB.predict_proba(X_va)[:, 1].astype(np.float32)
    else:
        # convert decision_function / predict output into pseudo-prob (not ideal)
        if hasattr(mB, "decision_function"):
            s = mB.decision_function(X_va)
            oof_B[va_idx] = (1.0 / (1.0 + np.exp(-s))).astype(np.float32)
        else:
            oof_B[va_idx] = mB.predict(X_va).astype(np.float32)

    fold_indices.append(va_idx)
    print(f"  Fold {fold}: done")

def tune_threshold(probs: np.ndarray, y_true: np.ndarray, thr_grid: np.ndarray):
    best_t, best_f = 0.5, -1.0
    for t in thr_grid:
        pred = (probs >= t).astype(int)
        f = f1_score(y_true, pred, average="macro")
        if f > best_f:
            best_f, best_t = float(f), float(t)
    return best_t, best_f

thr_grid = np.linspace(0.10, 0.90, 161)   # step=0.005
w_grid   = np.linspace(0.00, 1.00, 21)    # step=0.05

# diagnostics for individual models
tA, fA = tune_threshold(oof_A, y, thr_grid)
tB, fB = tune_threshold(oof_B, y, thr_grid)
print("\nOOF diagnostics (single-model):")
print(f"  Model A (LogReg TF-IDF) best OOF Macro-F1={fA:.4f} @ thr={tA:.3f}")
print(f"  Model B (LGBM+SVD)      best OOF Macro-F1={fB:.4f} @ thr={tB:.3f}")

# ensemble tuning (weight + threshold on OOF)
best = {"f1": -1.0, "w": 0.5, "t": 0.5}
for w in w_grid:
    blend = (w * oof_A + (1.0 - w) * oof_B)
    t, f = tune_threshold(blend, y, thr_grid)
    if f > best["f1"]:
        best.update({"f1": f, "w": float(w), "t": float(t)})

best_w, best_t, best_f = best["w"], best["t"], best["f1"]
blend_oof = best_w * oof_A + (1.0 - best_w) * oof_B
pred_oof = (blend_oof >= best_t).astype(int)

cm = confusion_matrix(y, pred_oof)
print("\nEnsemble (OOF tuned):")
print(f"  Best OOF Macro-F1 = {best_f:.4f} @ weight(wA)={best_w:.2f}, threshold={best_t:.3f}")
print("  Confusion matrix [[TN, FP],[FN, TP]]:")
print(cm)

# per-fold macro-f1 using chosen (w,t)
per_fold_f1 = []
for va_idx in fold_indices:
    f = f1_score(y[va_idx], (blend_oof[va_idx] >= best_t).astype(int), average="macro")
    per_fold_f1.append(float(f))
print("  Per-fold Macro-F1:", [round(x, 4) for x in per_fold_f1])

# quick error snapshot
err_df = train_df[id_cols + cat_cols + ["note_last3"]].copy()
err_df["y"] = y
err_df["p"] = blend_oof
err_df["pred"] = pred_oof
fp = err_df[(err_df["y"] == 0) & (err_df["pred"] == 1)].sort_values("p", ascending=False).head(5)
fn = err_df[(err_df["y"] == 1) & (err_df["pred"] == 0)].sort_values("p", ascending=True).head(5)

print("\nTop false positives (highest p among y=0 predicted 1):")
if len(fp) == 0:
    print("  (none)")
else:
    for r in fp.itertuples(index=False):
        print(f"  stay_id={r.stay_id} patient_id={r.patient_id} p={r.p:.3f} unit={r.unit_type} reason={r.admission_reason} note_last3='{str(r.note_last3)[:80]}'")

print("\nTop false negatives (lowest p among y=1 predicted 0):")
if len(fn) == 0:
    print("  (none)")
else:
    for r in fn.itertuples(index=False):
        print(f"  stay_id={r.stay_id} patient_id={r.patient_id} p={r.p:.3f} unit={r.unit_type} reason={r.admission_reason} note_last3='{str(r.note_last3)[:80]}'")

# ----------------------------
# Train final models on full training + predict test
# ----------------------------
print("\nTraining final models on full training set...")
final_A = clone(pipe_A).fit(X, y)
final_B = clone(pipe_B).fit(X, y)

test_proba_A = final_A.predict_proba(test_df)[:, 1].astype(np.float32)
if hasattr(final_B, "predict_proba"):
    test_proba_B = final_B.predict_proba(test_df)[:, 1].astype(np.float32)
else:
    if hasattr(final_B, "decision_function"):
        s = final_B.decision_function(test_df)
        test_proba_B = (1.0 / (1.0 + np.exp(-s))).astype(np.float32)
    else:
        test_proba_B = final_B.predict(test_df).astype(np.float32)

test_blend = best_w * test_proba_A + (1.0 - best_w) * test_proba_B
test_pred = (test_blend >= best_t).astype(int)

# ----------------------------
# Write submission
# ----------------------------
sub_path = BASE_DIR / "clai_ch3_v0_submission.csv"
submission = pd.DataFrame({
    "stay_id": test_df["stay_id"].astype(int),
    "discharge_ready_day11": test_pred.astype(int)
})
submission.to_csv(sub_path, index=False)
print(f"\n✅ Wrote submission: {sub_path.resolve()}  shape={submission.shape}")

# ----------------------------
# Save artifact(s)
# ----------------------------
import joblib

# Determine iteration_id (append-only)
log_path = BASE_DIR / "clai_ch3_v0_iteration_detail.jsonl"
last_id = -1
if log_path.exists():
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict) and obj.get("version") == "v0" and "iteration_id" in obj:
                    last_id = max(last_id, int(obj["iteration_id"]))
            except Exception:
                continue
iteration_id = last_id + 1

artifact = {
    "version": "v0",
    "iteration_id": iteration_id,
    "timestamp": datetime.now().isoformat(),
    "seed": SEED,
    "best_weight_wA": best_w,
    "best_threshold": best_t,
    "model_A": final_A,
    "model_B": final_B,
    "cat_cols": cat_cols,
    "text_cols": text_cols_A,
    "num_cols": num_cols,
}

model_main_path = BASE_DIR / "clai_ch3_v0_model.joblib"
model_iter_path = BASE_DIR / f"clai_ch3_v0_model_iter{iteration_id}.joblib"
joblib.dump(artifact, model_main_path)
joblib.dump(artifact, model_iter_path)
print(f"✅ Saved artifacts: {model_main_path.resolve()} and {model_iter_path.resolve()}")

# ----------------------------
# Append iteration log (JSONL)
# ----------------------------
entry = {
    "version": "v0",
    "iteration_id": iteration_id,
    "timestamp": datetime.now().isoformat(),
    "data_paths_used": {
        "base_dir": str(BASE_DIR.resolve()),
        "stays_train": str(stays_train_path),
        "stays_test": str(stays_test_path),
        "patients": str(patients_path),
        "vitals_timeseries": str(vitals_path),
    },
    "join_keys": {"patients_join_key": "patient_id", "vitals_join_key": "stay_id"},
    "feature_summary": {
        "categorical_cols": cat_cols,
        "text_cols": text_cols_A,
        "numeric_feature_count": int(len(num_cols)),
        "engineered_vitals": VITALS + ["map", "pp", "shock"],
        "keyword_count_features": int(len([c for c in train_df.columns if c.startswith("kw_")])),
        "tfidf_A": {
            "word_all_max_features": 8000,
            "char_all_max_features": 5000,
            "word_last3_max_features": 4000,
            "word_all_ngram": (1, 2),
            "char_all_ngram": (3, 5),
            "min_df": 2,
        },
        "svd_B": {
            "note_all_svd_components": 120,
            "note_last3_svd_components": 50,
            "tfidf_all_max_features": 10000,
            "tfidf_last3_max_features": 6000,
        },
    },
    "models_tried": [
        {"name": "ModelA_LogisticRegression_TFIDF_sparse", "hyperparams": clf_A.get_params()},
        {"name": "ModelB_LightGBM_SVD_dense" if LGBM_OK else "ModelB_HGB_SVD_dense_fallback",
         "hyperparams": (clf_B.get_params() if hasattr(clf_B, "get_params") else str(clf_B))},
    ],
    "validation_protocol": {
        "cv": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": SEED,
        "metric": "macro_f1",
        "oof_threshold_tuning": True,
        "oof_blend_weight_tuning": True,
    },
    "metrics": {
        "class_balance_train": {str(k): int(v) for k, v in sorted(cnt.items())},
        "modelA_oof_best_threshold": tA,
        "modelA_oof_best_macro_f1": fA,
        "modelB_oof_best_threshold": tB,
        "modelB_oof_best_macro_f1": fB,
        "ensemble_best_weight_wA": best_w,
        "ensemble_best_threshold": best_t,
        "macro_f1_oof_best": best_f,
        "macro_f1_per_fold": per_fold_f1,
        "confusion_matrix_oof": cm.tolist(),
    },
    "thresholding_strategy": {
        "weight_grid": w_grid.tolist(),
        "threshold_grid": thr_grid.tolist(),
        "selected_weight_wA": best_w,
        "selected_threshold": best_t,
        "objective": "maximize OOF macro-F1 (no test leakage)",
    },
    "top_errors": {
        "false_positives_top5": fp.to_dict(orient="records"),
        "false_negatives_top5": fn.to_dict(orient="records"),
    },
    "next_step_hypothesis": (
        "If LB plateaus or is unstable: (1) add CatBoost with text_features+cat_features as a 3rd model and retune ensemble; "
        "(2) try per-unit_type thresholds (only 2 groups) tuned on OOF; (3) calibrate probabilities (esp. LightGBM) before thresholding."
    ),
    "leakage_warnings_considered": (
        "Vitals/notes are from days 1-10 only; target is day-11 readiness. No tables with post-discharge outcomes are joined."
    ),
}

with open(log_path, "a", encoding="utf-8") as f:
    f.write(json.dumps(entry) + "\n")

print(f"✅ Appended iteration log: {log_path.resolve()} (iteration_id={iteration_id})")
print("\nDone.")

Data shapes:
  train_df: (1000, 166) | test_df: (1000, 165)
Columns:
  categorical: ['unit_type', 'admission_reason', 'sex', 'insurance', 'zip3']
  text: ['note_all', 'note_last3']
  numeric feature count: 156

Running 5-fold CV (deterministic)...
  Fold 0: done
  Fold 1: done
  Fold 2: done
  Fold 3: done
  Fold 4: done

OOF diagnostics (single-model):
  Model A (LogReg TF-IDF) best OOF Macro-F1=0.6983 @ thr=0.325
  Model B (LGBM+SVD)      best OOF Macro-F1=0.7879 @ thr=0.465

Ensemble (OOF tuned):
  Best OOF Macro-F1 = 0.7881 @ weight(wA)=0.25, threshold=0.650
  Confusion matrix [[TN, FP],[FN, TP]]:
[[238 106]
 [ 82 574]]
  Per-fold Macro-F1: [0.8168, 0.7769, 0.7519, 0.7883, 0.8063]

Top false positives (highest p among y=0 predicted 1):
  stay_id=1475 patient_id=3204 p=0.999 unit=med_surg reason=Pneumonia note_last3='mobilized with PT breathing at baseline telemetry unremarkable'
  stay_id=209 patient_id=803 p=0.995 unit=med_surg reason=HF note_last3='out of bed to chair twice today

# Iteration 12
- 0.8097

In [25]:
import os, json, random, datetime, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
import lightgbm as lgb
import joblib

warnings.filterwarnings("ignore")

# ======================
# Reproducibility
# ======================
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# ======================
# DATA dir (project dir OR /mnt/data fallback for this sandbox)
# OUTPUT dir (always current working dir; project dir for user)
# ======================
DATA_DIR = Path(".").resolve()
if not (DATA_DIR / "stays_train.csv").exists():
    alt = Path("/mnt/data")
    if (alt / "stays_train.csv").exists():
        DATA_DIR = alt

OUT_DIR = Path(".").resolve()

paths = {
    "base_dir": str(DATA_DIR),
    "stays_train": str(DATA_DIR / "stays_train.csv"),
    "stays_test": str(DATA_DIR / "stays_test.csv"),
    "patients": str(DATA_DIR / "patients.csv"),
    "vitals_timeseries": str(DATA_DIR / "vitals_timeseries.json"),
}
join_keys = {"stays<->patients": "patient_id", "stays<->vitals_timeseries": "stay_id"}

# ======================
# Load core tables
# ======================
stays_train = pd.read_csv(paths["stays_train"])
stays_test  = pd.read_csv(paths["stays_test"])
patients    = pd.read_csv(paths["patients"])
target_col  = "discharge_ready_day11"

# ======================
# Load & flatten vitals_timeseries.json
# ======================
with open(paths["vitals_timeseries"], "r", encoding="utf-8") as f:
    vitals = json.load(f)

rows = []
for rec in vitals:
    sid = rec.get("stay_id")
    for d in rec.get("days", []):
        rows.append({
            "stay_id": sid,
            "day": d.get("day"),
            "hr": d.get("hr"),
            "sbp": d.get("sbp"),
            "dbp": d.get("dbp"),
            "temp_c": d.get("temp_c"),
            "rr": d.get("rr"),
            "note": d.get("note", ""),
        })

vdf = pd.DataFrame(rows).sort_values(["stay_id", "day"]).reset_index(drop=True)

# Derived signals
vdf["pp"]  = vdf["sbp"] - vdf["dbp"]               # pulse pressure
vdf["map"] = vdf["dbp"] + (vdf["pp"] / 3.0)        # MAP approx

# Abnormal-day flags (NaNs -> False via comparison)
vdf["hr_hi"]   = (vdf["hr"] > 100).astype(float)
vdf["hr_lo"]   = (vdf["hr"] < 60).astype(float)
vdf["sbp_hi"]  = (vdf["sbp"] > 140).astype(float)
vdf["sbp_lo"]  = (vdf["sbp"] < 90).astype(float)
vdf["temp_hi"] = (vdf["temp_c"] > 38.0).astype(float)
vdf["temp_lo"] = (vdf["temp_c"] < 36.0).astype(float)
vdf["rr_hi"]   = (vdf["rr"] > 20).astype(float)
vdf["rr_lo"]   = (vdf["rr"] < 12).astype(float)

signal_cols = ["hr", "sbp", "dbp", "temp_c", "rr", "pp", "map"]
flag_cols   = ["hr_hi", "hr_lo", "sbp_hi", "sbp_lo", "temp_hi", "temp_lo", "rr_hi", "rr_lo"]

# ======================
# Aggregate per stay_id (days 1-10 only)
# ======================
agg_rows = []
for sid, g in vdf.groupby("stay_id", sort=False):
    g = g.sort_values("day")
    row = {"stay_id": sid}

    # Keep notes only as length stats (text itself can be noisy)
    row["notes_all"] = " ".join(g["note"].astype(str).tolist())
    row["note_chars_mean"]   = g["note"].astype(str).str.len().mean()
    row["note_chars_first3"] = g.loc[g["day"] <= 3, "note"].astype(str).str.len().mean()
    row["note_chars_last3"]  = g.loc[g["day"] >= 8, "note"].astype(str).str.len().mean()

    days = g["day"].to_numpy(dtype=float)

    for col in signal_cols:
        vals = g[col].to_numpy(dtype=float)
        row[f"{col}_missing"] = int(np.isnan(vals).sum())

        row[f"{col}_mean"]   = float(np.nanmean(vals))
        row[f"{col}_std"]    = float(np.nanstd(vals))
        row[f"{col}_min"]    = float(np.nanmin(vals))
        row[f"{col}_max"]    = float(np.nanmax(vals))
        row[f"{col}_median"] = float(np.nanmedian(vals))

        q25 = float(np.nanquantile(vals, 0.25))
        q75 = float(np.nanquantile(vals, 0.75))
        row[f"{col}_q25"] = q25
        row[f"{col}_q75"] = q75
        row[f"{col}_iqr"] = q75 - q25

        row[f"{col}_first"] = float(vals[0]) if not np.isnan(vals[0]) else np.nan
        row[f"{col}_last"]  = float(vals[-1]) if not np.isnan(vals[-1]) else np.nan

        row[f"{col}_first3_mean"] = float(np.nanmean(vals[:3]))
        row[f"{col}_last3_mean"]  = float(np.nanmean(vals[-3:]))
        row[f"{col}_last3_std"]   = float(np.nanstd(vals[-3:]))
        row[f"{col}_last3_minus_first3"] = row[f"{col}_last3_mean"] - row[f"{col}_first3_mean"]

        mask = ~np.isnan(vals)
        row[f"{col}_slope"] = float(np.polyfit(days[mask], vals[mask], 1)[0]) if mask.sum() >= 2 else np.nan

    for col in flag_cols:
        fvals = g[col].to_numpy(dtype=float)
        row[f"{col}_sum"]        = float(np.nansum(fvals))
        row[f"{col}_mean"]       = float(np.nanmean(fvals))
        row[f"{col}_first3_sum"] = float(np.nansum(fvals[:3]))
        row[f"{col}_last3_sum"]  = float(np.nansum(fvals[-3:]))

    agg_rows.append(row)

vitals_agg = pd.DataFrame(agg_rows)

# ======================
# Merge features (stay_id + patient_id anchored)
# ======================
train_df = stays_train.merge(patients, on="patient_id", how="left").merge(vitals_agg, on="stay_id", how="left")
test_df  = stays_test.merge(patients, on="patient_id", how="left").merge(vitals_agg, on="stay_id", how="left")

missing_vitals_train = int(train_df["note_chars_mean"].isna().sum())
missing_vitals_test  = int(test_df["note_chars_mean"].isna().sum())

# ======================
# Feature columns
# ======================
cat_cols = ["unit_type", "admission_reason", "sex", "insurance", "zip3"]
for df in (train_df, test_df):
    df["zip3"] = df["zip3"].astype("Int64").astype(str)
    for c in cat_cols:
        df[c] = df[c].astype("category")

exclude = {"stay_id", "patient_id", target_col, "notes_all"}
num_cols = [c for c in train_df.columns if c not in exclude and c not in cat_cols]

X = train_df[cat_cols + num_cols]
y = train_df[target_col].astype(int).to_numpy()
X_test = test_df[cat_cols + num_cols]

# ======================
# Model (strong baseline) + deterministic CV
# ======================
lgb_params = dict(
    n_estimators=8000,
    learning_rate=0.012,
    num_leaves=255,
    min_child_samples=25,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=3.0,
    reg_alpha=0.0,
    objective="binary",
    n_jobs=1,
    deterministic=True,
    force_col_wise=True,
    verbosity=-1,
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_proba = np.zeros(len(X), dtype=float)
best_iters = []
fold_scores_05 = []

print("==== CV training (LightGBM) ====")
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
    X_va, y_va = X.iloc[va_idx], y[va_idx]

    clf = lgb.LGBMClassifier(**lgb_params, random_state=SEED + fold)
    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="binary_logloss",
        callbacks=[lgb.early_stopping(stopping_rounds=250, verbose=False)]
    )

    p_va = clf.predict_proba(X_va)[:, 1]
    oof_proba[va_idx] = p_va
    best_iters.append(int(clf.best_iteration_))

    fold_f1_05 = f1_score(y_va, (p_va >= 0.5).astype(int), average="macro")
    fold_scores_05.append(float(fold_f1_05))
    print(f"fold={fold} best_iter={clf.best_iteration_} macroF1@0.5={fold_f1_05:.4f}")

# Threshold tuning on OOF (optimize Macro-F1)
thr_grid = np.linspace(0.2, 0.85, 261)  # step=0.0025
best_thr, best_f1 = 0.5, -1.0
for thr in thr_grid:
    f1 = f1_score(y, (oof_proba >= thr).astype(int), average="macro")
    if f1 > best_f1:
        best_f1 = float(f1)
        best_thr = float(thr)

# Per-fold tuned scores + confusion matrix
fold_scores_tuned = []
for _, va_idx in skf.split(X, y):
    fold_scores_tuned.append(float(f1_score(y[va_idx], (oof_proba[va_idx] >= best_thr).astype(int), average="macro")))

y_oof_pred = (oof_proba >= best_thr).astype(int)
cm = confusion_matrix(y, y_oof_pred, labels=[0, 1]).tolist()

print("\n==== OOF summary ====")
print(f"OOF macro-F1 (tuned thr) = {best_f1:.6f} at threshold={best_thr:.4f}")
print(f"Per-fold macro-F1 @0.5   = {[round(s,4) for s in fold_scores_05]} (mean={np.mean(fold_scores_05):.4f})")
print(f"Per-fold macro-F1 tuned  = {[round(s,4) for s in fold_scores_tuned]} (mean={np.mean(fold_scores_tuned):.4f})")
print(f"OOF confusion matrix [[TN,FP],[FN,TP]] = {cm}")
print(f"Missing vitals agg rows: train={missing_vitals_train}, test={missing_vitals_test}")
print(f"Avg best_iter={np.mean(best_iters):.1f}, best_iters={best_iters}")

# ======================
# Train final model on full training data
# ======================
final_n_estimators = max(200, int(round(np.mean(best_iters))))
final_model = lgb.LGBMClassifier(**{**lgb_params, "n_estimators": final_n_estimators, "random_state": SEED})
final_model.fit(X, y)

test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_thr).astype(int)

# ======================
# Write required submission (EXACT filename + columns)
# ======================
sub_path = OUT_DIR / "clai_ch3_v0_submission.csv"
submission = pd.DataFrame({
    "stay_id": test_df["stay_id"].astype(int),
    "discharge_ready_day11": test_pred.astype(int),
})
submission.to_csv(sub_path, index=False)

# ======================
# Save artifact (optional but recommended)
# ======================
model_path = OUT_DIR / "clai_ch3_v0_model.joblib"
artifact = {
    "model": final_model,
    "threshold": best_thr,
    "cat_cols": cat_cols,
    "num_cols": num_cols,
    "lgb_params": {**lgb_params, "n_estimators": int(final_n_estimators)},
    "feature_count": int(len(cat_cols) + len(num_cols)),
    "data_dir_used": str(DATA_DIR),
}
model_written = str(model_path)
try:
    joblib.dump(artifact, model_path)
except PermissionError:
    fallback = OUT_DIR / "clai_ch3_v0_model_iter_fallback.joblib"
    joblib.dump(artifact, fallback)
    model_written = str(fallback)

# ======================
# Append iteration log (jsonl) (append-only)
# ======================
log_path = OUT_DIR / "clai_ch3_v0_iteration_detail.jsonl"
if log_path.exists():
    with open(log_path, "r", encoding="utf-8") as f:
        iteration_id = sum(1 for line in f if line.strip())
else:
    iteration_id = 0

# Top errors (OOF)
oof_df = pd.DataFrame({
    "stay_id": train_df["stay_id"].astype(int),
    "y_true": y.astype(int),
    "proba": oof_proba,
})
oof_df["y_pred"] = y_oof_pred
fps = oof_df[(oof_df["y_true"] == 0) & (oof_df["y_pred"] == 1)].sort_values("proba", ascending=False).head(5)
fns = oof_df[(oof_df["y_true"] == 1) & (oof_df["y_pred"] == 0)].sort_values("proba", ascending=True).head(5)

entry = {
    "version": "v0",
    "iteration_id": int(iteration_id),
    "timestamp": datetime.datetime.now().isoformat(),
    "data_paths_used": {**paths, "join_keys": join_keys},
    "feature_summary": {
        "categorical_cols": cat_cols,
        "categorical_encoding": "LightGBM native categorical via pandas 'category' dtype (no one-hot).",
        "numeric_feature_count": int(len(num_cols)),
        "engineered_numeric": [
            "per-signal robust stats over days 1-10: mean/std/min/max/median/q25/q75/iqr",
            "temporal endpoints + trends: first/last, slope, last3 vs first3 deltas, last3 std",
            "derived signals: pulse pressure (pp=sbp-dbp) and MAP (map=dbp+pp/3)",
            "abnormal-day counts for hr/sbp/temp/rr (hi/lo) + first3 vs last3 abnormal totals",
            "note length stats (mean/first3/last3 chars) only (no TF-IDF this iteration)",
        ],
        "signals": signal_cols,
        "abnormal_flags": flag_cols,
        "total_feature_count_est": int(len(cat_cols) + len(num_cols)),
    },
    "models_tried": [{
        "name": "LightGBM LGBMClassifier",
        "hyperparams": {k: (float(v) if isinstance(v, (np.floating,)) else v)
                        for k, v in {**lgb_params, "n_estimators": int(final_n_estimators)}.items()},
        "early_stopping": {"stopping_rounds": 250, "metric": "binary_logloss"},
        "best_iteration_per_fold": best_iters,
        "final_n_estimators_used": int(final_n_estimators),
    }],
    "validation_protocol": {
        "cv": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": SEED,
        "macro_f1_definition": "sklearn.metrics.f1_score(average='macro')",
    },
    "metrics": {
        "macro_f1_oof_best_threshold": best_f1,
        "best_threshold": best_thr,
        "macro_f1_per_fold_thr0.5": fold_scores_05,
        "macro_f1_per_fold_tuned": fold_scores_tuned,
        "macro_f1_mean_tuned": float(np.mean(fold_scores_tuned)),
        "confusion_matrix_oof_tuned_labels_0_1": cm,
        "class_balance_train": {str(k): int(v) for k, v in pd.Series(y).value_counts().sort_index().items()},
        "pred_positive_rate_oof": float(y_oof_pred.mean()),
        "missing_vitals_agg_train_rows": int(missing_vitals_train),
        "missing_vitals_agg_test_rows": int(missing_vitals_test),
    },
    "thresholding_strategy": {
        "strategy": "grid search threshold on out-of-fold probabilities to maximize macro-F1",
        "grid": {"min": 0.2, "max": 0.85, "num": 261, "step": 0.0025},
        "best_threshold": best_thr,
    },
    "top_errors_failure_analysis": {
        "false_positive_count_oof": int(((oof_df["y_true"] == 0) & (oof_df["y_pred"] == 1)).sum()),
        "false_negative_count_oof": int(((oof_df["y_true"] == 1) & (oof_df["y_pred"] == 0)).sum()),
        "top_false_positives": fps[["stay_id", "y_true", "proba", "y_pred"]].to_dict(orient="records"),
        "top_false_negatives": fns[["stay_id", "y_true", "proba", "y_pred"]].to_dict(orient="records"),
    },
    "next_step_hypothesis": (
        "Try a small seed-bagged ensemble of this LightGBM setup (e.g., 3 seeds) or blend with an XGBoost model; "
        "tune ensemble weight + threshold on OOF to push macro-F1 further."
    ),
    "leakage_overfitting_warnings_considered": [
        "Features use only day1-10 vitals + daily notes (length only) and static demographics/unit/admission_reason.",
        "Did NOT join admissions_* / discharge_notes / ed_cost tables (risk of mismatched keys or post-outcome leakage).",
        "Threshold tuned only on out-of-fold predictions (reduces in-fold leakage).",
    ],
    "artifacts_written": {
        "submission_csv": str(sub_path),
        "model_joblib": model_written,
        "iteration_log_jsonl": str(log_path),
    },
}

with open(log_path, "a", encoding="utf-8") as f:
    f.write(json.dumps(entry) + "\n")

print("\n==== Files written ====")
print("submission:", sub_path)
print("model     :", model_written)
print("log       :", log_path)

==== CV training (LightGBM) ====
fold=0 best_iter=1555 macroF1@0.5=0.8304
fold=1 best_iter=1009 macroF1@0.5=0.7877
fold=2 best_iter=660 macroF1@0.5=0.8163
fold=3 best_iter=789 macroF1@0.5=0.8112
fold=4 best_iter=784 macroF1@0.5=0.8300

==== OOF summary ====
OOF macro-F1 (tuned thr) = 0.831893 at threshold=0.6150
Per-fold macro-F1 @0.5   = [0.8304, 0.7877, 0.8163, 0.8112, 0.83] (mean=0.8151)
Per-fold macro-F1 tuned  = [0.8521, 0.8354, 0.8181, 0.8253, 0.8277] (mean=0.8317)
OOF confusion matrix [[TN,FP],[FN,TP]] = [[265, 79], [72, 584]]
Missing vitals agg rows: train=0, test=0
Avg best_iter=959.4, best_iters=[1555, 1009, 660, 789, 784]

==== Files written ====
submission: D:\AgentDs\agent_ds_healthcare\clai_ch3_v0_submission.csv
model     : D:\AgentDs\agent_ds_healthcare\clai_ch3_v0_model.joblib
log       : D:\AgentDs\agent_ds_healthcare\clai_ch3_v0_iteration_detail.jsonl


# Iteration 13
- 0.7324

In [28]:
# clai_ch3_v0 — Iteration (fix joblib.clone + stronger ensemble baseline for CH3)
import os, json, time, warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.base import clone  # ✅ FIX: clone comes from sklearn, NOT joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight

import joblib

warnings.filterwarnings("ignore")

# -------------------------
# Config
# -------------------------
SEED = 42
np.random.seed(SEED)

BASE_DIR = "."  # adjust if needed
PATH_STAYS_TRAIN = os.path.join(BASE_DIR, "stays_train.csv")
PATH_STAYS_TEST  = os.path.join(BASE_DIR, "stays_test.csv")
PATH_PATIENTS    = os.path.join(BASE_DIR, "patients.csv")
PATH_VITALS_JSON = os.path.join(BASE_DIR, "vitals_timeseries.json")

SUBMISSION_PATH = os.path.join(BASE_DIR, "clai_ch3_v0_submission.csv")
LOG_PATH        = os.path.join(BASE_DIR, "clai_ch3_v0_iteration_detail.jsonl")
MODEL_PATH      = os.path.join(BASE_DIR, "clai_ch3_v0_model.joblib")

N_SPLITS = 5
THR_GRID = np.linspace(0.05, 0.95, 91)     # threshold grid for Macro-F1
W_GRID   = np.linspace(0.0, 1.0, 21)       # ensemble weight grid for Model A

# -------------------------
# Helpers
# -------------------------
def _safe_stat(func, arr, default=np.nan):
    arr = np.asarray(arr, dtype=float)
    m = np.isfinite(arr)
    if m.sum() == 0:
        return default
    return float(func(arr[m]))

def _nanpercentile(arr, q):
    arr = np.asarray(arr, dtype=float)
    m = np.isfinite(arr)
    if m.sum() == 0:
        return np.nan
    return float(np.percentile(arr[m], q))

def _lin_slope(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 2:
        return np.nan
    # degree-1 polyfit slope
    return float(np.polyfit(x[m], y[m], 1)[0])

def _make_ohe_dense():
    # sklearn version compatibility
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def _make_ohe_sparse():
    # sklearn version compatibility
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

def best_threshold_macro_f1(y_true, proba, grid=THR_GRID):
    best_t, best_f = 0.5, -1.0
    for t in grid:
        pred = (proba >= t).astype(int)
        f = f1_score(y_true, pred, average="macro")
        if f > best_f:
            best_f, best_t = f, float(t)
    return best_t, best_f

def parse_vitals_to_features(vitals_json_path):
    """
    Build per-stay features from 10-day vitals + daily note text.
    Outputs:
      - stay_id
      - notes_all, notes_last3, note_day10  (text)
      - engineered numeric features
    """
    with open(vitals_json_path, "r", encoding="utf-8") as f:
        vitals = json.load(f)

    # curated keyword buckets (light, deterministic)
    kw_buckets = {
        "mobility": ["ambulat", "walk", "oob", "out of bed", "chair", "pt", "ot", "stairs", "transfer"],
        "resp":     ["weaned", "room air", "o2", "sat", "cpap", "incentive", "spirom", "respir"],
        "pain":     ["pain", "analges", "prn", "po med", "controlled"],
        "gi":       ["diet", "no n/v", "flatus", "bowel", "bm", "void"],
        "infect":   ["afebrile", "fever", "ua", "infection", "procalcitonin", "crp", "esr", "lactate", "antibiot"],
        "dispo":    ["awaiting placement", "rehab", "snf", "home safety", "placement"],
        "stable":   ["stable", "improv", "normalized", "unremarkable", "comfortably", "baseline"],
    }

    rows = []
    for rec in vitals:
        sid = rec.get("stay_id")
        days_list = rec.get("days", [])
        days_list = sorted(days_list, key=lambda d: d.get("day", 0))

        day = np.array([d.get("day", np.nan) for d in days_list], dtype=float)

        def arr(key):
            out = []
            for d in days_list:
                v = d.get(key, None)
                out.append(np.nan if v is None else float(v))
            return np.array(out, dtype=float)

        hr   = arr("hr")
        sbp  = arr("sbp")
        dbp  = arr("dbp")
        temp = arr("temp_c")
        rr   = arr("rr")
        pp   = sbp - dbp
        mapv = (2.0 * dbp + sbp) / 3.0

        notes = [str(d.get("note") or "") for d in days_list]
        notes_all   = " ".join(notes).strip()
        notes_last3 = " ".join(notes[-3:]).strip()
        note_day10  = (notes[-1] if len(notes) else "").strip()

        note_len   = np.array([len(n) for n in notes], dtype=float)
        note_words = np.array([len(n.split()) for n in notes], dtype=float)

        feats = {"stay_id": sid, "notes_all": notes_all, "notes_last3": notes_last3, "note_day10": note_day10}

        # text keyword counts (numeric)
        txt = (" " + notes_all.lower() + " ")
        for bucket, kws in kw_buckets.items():
            c = 0
            for kw in kws:
                c += txt.count(kw)
            feats[f"kw_{bucket}_count"] = float(c)

        # basic note length stats
        for nm, a in [("note_len", note_len), ("note_words", note_words)]:
            feats[f"{nm}_mean"] = _safe_stat(np.mean, a)
            feats[f"{nm}_std"]  = _safe_stat(np.std, a)
            feats[f"{nm}_min"]  = _safe_stat(np.min, a)
            feats[f"{nm}_max"]  = _safe_stat(np.max, a)
            feats[f"{nm}_first"] = float(a[0]) if len(a) else np.nan
            feats[f"{nm}_last"]  = float(a[-1]) if len(a) else np.nan
            feats[f"{nm}_last3_mean"]  = _safe_stat(np.mean, a[-3:] if len(a) else a)
            feats[f"{nm}_first3_mean"] = _safe_stat(np.mean, a[:3]  if len(a) else a)

        # per-signal stats
        sigs = {
            "hr": hr, "sbp": sbp, "dbp": dbp, "temp_c": temp, "rr": rr, "pp": pp, "map": mapv
        }
        eps = 1e-3
        for s, a in sigs.items():
            feats[f"{s}_mean"]   = _safe_stat(np.mean, a)
            feats[f"{s}_std"]    = _safe_stat(np.std, a)
            feats[f"{s}_min"]    = _safe_stat(np.min, a)
            feats[f"{s}_max"]    = _safe_stat(np.max, a)
            feats[f"{s}_median"] = _safe_stat(np.median, a)
            feats[f"{s}_q25"]    = _nanpercentile(a, 25)
            feats[f"{s}_q75"]    = _nanpercentile(a, 75)
            feats[f"{s}_first"]  = float(a[0]) if len(a) else np.nan
            feats[f"{s}_last"]   = float(a[-1]) if len(a) else np.nan
            feats[f"{s}_delta"]  = feats[f"{s}_last"] - feats[f"{s}_first"] if len(a) else np.nan
            feats[f"{s}_slope"]  = _lin_slope(day, a)

            last3 = a[-3:] if len(a) >= 3 else a
            first3 = a[:3] if len(a) >= 3 else a
            feats[f"{s}_last3_mean"] = _safe_stat(np.mean, last3)
            feats[f"{s}_first3_mean"] = _safe_stat(np.mean, first3)
            feats[f"{s}_delta_last3_first3"] = feats[f"{s}_last3_mean"] - feats[f"{s}_first3_mean"]
            feats[f"{s}_ratio_last3_first3"] = feats[f"{s}_last3_mean"] / (feats[f"{s}_first3_mean"] + eps)
            feats[f"{s}_cv"] = feats[f"{s}_std"] / (abs(feats[f"{s}_mean"]) + eps)

            # last-step deltas
            if len(a) >= 2 and np.isfinite(a[-1]) and np.isfinite(a[-2]):
                feats[f"{s}_delta_last1"] = float(a[-1] - a[-2])
            else:
                feats[f"{s}_delta_last1"] = np.nan

            # missingness
            feats[f"{s}_missing_frac"] = float(np.mean(~np.isfinite(a))) if len(a) else np.nan

        # abnormal/stability counts (simple clinical-ish ranges)
        def _count(mask_arr):
            mask_arr = np.asarray(mask_arr)
            return float(np.sum(mask_arr))

        hr_ok   = np.isfinite(hr)
        sbp_ok  = np.isfinite(sbp)
        dbp_ok  = np.isfinite(dbp)
        temp_ok = np.isfinite(temp)
        rr_ok   = np.isfinite(rr)
        map_ok  = np.isfinite(mapv)
        pp_ok   = np.isfinite(pp)

        abn_hr   = hr_ok   & ((hr > 100) | (hr < 60))
        abn_sbp  = sbp_ok  & ((sbp < 90) | (sbp > 160))
        abn_dbp  = dbp_ok  & ((dbp < 50) | (dbp > 100))
        abn_temp = temp_ok & ((temp > 37.8) | (temp < 36.0))
        abn_rr   = rr_ok   & ((rr > 20) | (rr < 10))
        abn_map  = map_ok  & ((mapv < 65) | (mapv > 110))
        abn_pp   = pp_ok   & ((pp < 30) | (pp > 80))

        abn_any = abn_hr | abn_sbp | abn_dbp | abn_temp | abn_rr | abn_map | abn_pp
        feats["abn_any_count"] = _count(abn_any)
        feats["abn_any_last3_count"] = _count(abn_any[-3:] if len(abn_any) else abn_any)
        feats["stable_last3"] = float(_count(abn_any[-3:] if len(abn_any) else abn_any) == 0)
        feats["stable_last5"] = float(_count(abn_any[-5:] if len(abn_any) else abn_any) == 0)

        rows.append(feats)

    return pd.DataFrame(rows)

# -------------------------
# Load data
# -------------------------
stays_train = pd.read_csv(PATH_STAYS_TRAIN)
stays_test  = pd.read_csv(PATH_STAYS_TEST)
patients    = pd.read_csv(PATH_PATIENTS)

vitals_feats = parse_vitals_to_features(PATH_VITALS_JSON)

train_df = stays_train.merge(patients, on="patient_id", how="left").merge(vitals_feats, on="stay_id", how="left")
test_df  = stays_test.merge(patients, on="patient_id", how="left").merge(vitals_feats, on="stay_id", how="left")

# fill missing text
for c in ["notes_all", "notes_last3", "note_day10"]:
    train_df[c] = train_df[c].fillna("")
    test_df[c]  = test_df[c].fillna("")

# -------------------------
# Feature columns
# -------------------------
TARGET = "discharge_ready_day11"
ID_COL = "stay_id"

cat_cols = ["unit_type", "admission_reason", "sex", "insurance", "zip3"]
text_cols = ["notes_all", "notes_last3", "note_day10"]

drop_cols = [TARGET]
X = train_df.drop(columns=drop_cols)
y = train_df[TARGET].astype(int).values
X_test = test_df.copy()

# numeric = everything except ids/cats/text
exclude = set([ID_COL, "patient_id"] + cat_cols + text_cols)
num_cols = [c for c in X.columns if c not in exclude]

# -------------------------
# Model A: LR on full features (num + cat + multi-window TF-IDF)
# -------------------------
pre_A = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler(with_mean=False)),
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", _make_ohe_sparse()),
        ]), cat_cols),
        ("txt_all", TfidfVectorizer(
            ngram_range=(1,2), min_df=2, max_df=0.98, max_features=8000,
            sublinear_tf=True, lowercase=True
        ), "notes_all"),
        ("txt_last3", TfidfVectorizer(
            ngram_range=(1,2), min_df=1, max_df=0.98, max_features=4000,
            sublinear_tf=True, lowercase=True
        ), "notes_last3"),
        ("txt_day10", TfidfVectorizer(
            ngram_range=(1,2), min_df=1, max_df=0.98, max_features=2500,
            sublinear_tf=True, lowercase=True
        ), "note_day10"),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

clf_A = LogisticRegression(
    solver="liblinear",
    penalty="l2",
    dual=True,
    C=3.0,
    max_iter=8000,
    class_weight="balanced",
    random_state=SEED
)

model_A = Pipeline([("pre", pre_A), ("clf", clf_A)])

# -------------------------
# Model B: Tree on numeric+categorical only (XGB if available; else HistGB)
# -------------------------
use_xgb = False
try:
    import xgboost as xgb
    use_xgb = True
except Exception:
    use_xgb = False

if use_xgb:
    pre_B = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]), num_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", _make_ohe_sparse()),
            ]), cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3
    )
    clf_B = xgb.XGBClassifier(
        n_estimators=450,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=2.0,
        min_child_weight=1.5,
        gamma=0.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=SEED,
        n_jobs=1  # more deterministic
    )
else:
    from sklearn.ensemble import HistGradientBoostingClassifier
    pre_B = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]), num_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", _make_ohe_dense()),  # dense needed for HistGB
            ]), cat_cols),
        ],
        remainder="drop"
    )
    clf_B = HistGradientBoostingClassifier(
        max_depth=4,
        learning_rate=0.05,
        max_iter=400,
        l2_regularization=0.2,
        random_state=SEED
    )

model_B = Pipeline([("pre", pre_B), ("clf", clf_B)])

# -------------------------
# Deterministic CV: OOF probs for both models
# -------------------------
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_A = np.zeros(len(y), dtype=float)
oof_B = np.zeros(len(y), dtype=float)

fold_metrics = []
fold_ids = []
start = time.time()

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
    X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
    X_va, y_va = X.iloc[va_idx], y[va_idx]

    # balanced sample weights for Model B (tree)
    sw_tr = compute_sample_weight(class_weight="balanced", y=y_tr)

    mA = clone(model_A)
    mB = clone(model_B)

    mA.fit(X_tr, y_tr)
    # fit Model B with sample_weight passed down to clf
    mB.fit(X_tr, y_tr, clf__sample_weight=sw_tr)

    pA = mA.predict_proba(X_va)[:, 1]
    pB = mB.predict_proba(X_va)[:, 1]

    oof_A[va_idx] = pA
    oof_B[va_idx] = pB

    # quick fold diagnostics at 0.5 with equal weight ensemble (placeholder)
    p_ens_05 = 0.5 * pA + 0.5 * pB
    pred_05 = (p_ens_05 >= 0.5).astype(int)
    f1_05 = f1_score(y_va, pred_05, average="macro")

    fold_metrics.append({"fold": fold, "macro_f1_fold_ens_w0.5_thr0.5": float(f1_05)})
    fold_ids.append((tr_idx, va_idx))

elapsed = time.time() - start

# -------------------------
# Tune ensemble weight + threshold on OOF for Macro-F1
# -------------------------
best = {"macro_f1": -1.0, "wA": 0.5, "thr": 0.5}
for wA in W_GRID:
    p = wA * oof_A + (1.0 - wA) * oof_B
    thr, f1 = best_threshold_macro_f1(y, p, grid=THR_GRID)
    if f1 > best["macro_f1"]:
        best = {"macro_f1": float(f1), "wA": float(wA), "thr": float(thr)}

# per-fold metrics at tuned threshold
per_fold_thr05 = []
per_fold_tuned = []
cms_fold = []

oof_p = best["wA"] * oof_A + (1.0 - best["wA"]) * oof_B
oof_pred_05 = (oof_p >= 0.5).astype(int)
oof_pred_t  = (oof_p >= best["thr"]).astype(int)

for fold, (tr_idx, va_idx) in enumerate(fold_ids, start=1):
    y_va = y[va_idx]
    p_va = oof_p[va_idx]

    pred05 = (p_va >= 0.5).astype(int)
    predt  = (p_va >= best["thr"]).astype(int)

    per_fold_thr05.append(float(f1_score(y_va, pred05, average="macro")))
    per_fold_tuned.append(float(f1_score(y_va, predt,  average="macro")))
    cms_fold.append(confusion_matrix(y_va, predt, labels=[0,1]).tolist())

cm_oof = confusion_matrix(y, oof_pred_t, labels=[0,1]).tolist()
pos_rate = float(oof_pred_t.mean())

# top errors
stay_ids_train = train_df[ID_COL].values
fp_idx = np.where((oof_pred_t == 1) & (y == 0))[0]
fn_idx = np.where((oof_pred_t == 0) & (y == 1))[0]
top_fp = fp_idx[np.argsort(-oof_p[fp_idx])][:8]
top_fn = fn_idx[np.argsort(oof_p[fn_idx])][:8]
top_errors = {
    "false_positives_high_conf": [{"stay_id": int(stay_ids_train[i]), "y_true": int(y[i]), "proba": float(oof_p[i])} for i in top_fp],
    "false_negatives_high_conf": [{"stay_id": int(stay_ids_train[i]), "y_true": int(y[i]), "proba": float(oof_p[i])} for i in top_fn],
}

# -------------------------
# Train final models on full train, predict test
# -------------------------
sw_full = compute_sample_weight(class_weight="balanced", y=y)

final_A = clone(model_A).fit(X, y)
final_B = clone(model_B).fit(X, y, clf__sample_weight=sw_full)

pA_test = final_A.predict_proba(X_test)[:, 1]
pB_test = final_B.predict_proba(X_test)[:, 1]
p_test  = best["wA"] * pA_test + (1.0 - best["wA"]) * pB_test
y_test_pred = (p_test >= best["thr"]).astype(int)

sub = pd.DataFrame({
    "stay_id": test_df["stay_id"].astype(int).values,
    "discharge_ready_day11": y_test_pred.astype(int)
})
sub.to_csv(SUBMISSION_PATH, index=False)

# -------------------------
# Append iteration log
# -------------------------
def _next_iteration_id(log_path):
    if not os.path.exists(log_path):
        return 0
    last_id = -1
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                last_id = max(last_id, int(obj.get("iteration_id", -1)))
            except Exception:
                continue
    return last_id + 1

iteration_id = _next_iteration_id(LOG_PATH)

artifact = {
    "version": "v0",
    "iteration_id": iteration_id,
    "seed": SEED,
    "model_A": final_A,
    "model_B": final_B,
    "ensemble": {"weight_model_A": best["wA"], "weight_model_B": 1.0 - best["wA"], "threshold": best["thr"]},
    "use_xgb": use_xgb,
    "feature_columns": {
        "categorical_cols": cat_cols,
        "text_cols": text_cols,
        "numeric_cols": num_cols
    }
}
joblib.dump(artifact, MODEL_PATH)
# optional iter-specific artifact
joblib.dump(artifact, os.path.join(BASE_DIR, f"clai_ch3_v0_model_iter{iteration_id}.joblib"))

log_entry = {
    "version": "v0",
    "iteration_id": iteration_id,
    "timestamp": pd.Timestamp.now().isoformat(),
    "data_paths_used": {
        "base_dir": BASE_DIR,
        "stays_train": os.path.basename(PATH_STAYS_TRAIN),
        "stays_test": os.path.basename(PATH_STAYS_TEST),
        "patients": os.path.basename(PATH_PATIENTS),
        "vitals_timeseries": os.path.basename(PATH_VITALS_JSON)
    },
    "join_keys": {"stays<->patients": "patient_id", "stays<->vitals_features": "stay_id"},
    "feature_summary": {
        "categorical_cols": cat_cols,
        "text_cols": text_cols,
        "numeric_feature_count": int(len(num_cols)),
        "text_vectorizers": {
            "notes_all":   {"ngram_range":[1,2], "min_df":2, "max_df":0.98, "max_features":8000, "sublinear_tf":True},
            "notes_last3": {"ngram_range":[1,2], "min_df":1, "max_df":0.98, "max_features":4000, "sublinear_tf":True},
            "note_day10":  {"ngram_range":[1,2], "min_df":1, "max_df":0.98, "max_features":2500, "sublinear_tf":True},
        },
        "engineered_numeric": [
            "per-vital stats (mean/std/min/max/median/q25/q75/first/last/delta/slope/cv/missing_frac)",
            "windowed deltas (last3 vs first3) + last-step delta",
            "derived MAP and pulse pressure",
            "abnormal/stability counts",
            "note length stats + keyword bucket counts"
        ]
    },
    "models_tried": [
        {"name": "Model_A_LogisticRegression", "hyperparams": clf_A.get_params()},
        {"name": ("Model_B_XGBClassifier" if use_xgb else "Model_B_HistGradientBoosting"), "hyperparams": clf_B.get_params()},
        {"name": "Ensemble", "hyperparams": {"weight_model_A_grid": list(map(float, W_GRID)), "threshold_grid": [float(THR_GRID[0]), float(THR_GRID[-1]), int(len(THR_GRID))]}}
    ],
    "validation_protocol": {
        "cv": "StratifiedKFold",
        "n_splits": N_SPLITS,
        "shuffle": True,
        "random_state": SEED
    },
    "metrics": {
        "macro_f1_oof_best": best["macro_f1"],
        "best_ensemble_weight_model_A": best["wA"],
        "best_threshold": best["thr"],
        "macro_f1_per_fold_thr0.5": per_fold_thr05,
        "macro_f1_per_fold_tuned": per_fold_tuned,
        "macro_f1_mean_tuned": float(np.mean(per_fold_tuned)),
        "confusion_matrix_oof_tuned_labels_0_1": cm_oof,
        "pred_positive_rate_oof": pos_rate,
        "class_balance_train": {str(k): int(v) for k, v in zip(*np.unique(y, return_counts=True))}
    },
    "thresholding_strategy": "Grid-search threshold on OOF probs to maximize Macro-F1; ensemble weight also grid-searched on OOF.",
    "top_errors_failure_analysis": top_errors,
    "next_step_hypothesis": "If this underperforms LB, try char-ngrams for note_day10 or add a calibrated LinearSVC text model into the ensemble; also consider finer weight grid around best_wA.",
    "leakage_overfitting_warnings_considered": [
        "Did NOT join admissions/discharge_notes/ed_cost to avoid post-outcome leakage (e.g., LOS/readmit/discharge text).",
        "Only uses days 1-10 vitals + notes (no day 11+ info)."
    ],
    "artifacts_written": {
        "submission": os.path.basename(SUBMISSION_PATH),
        "model_joblib": os.path.basename(MODEL_PATH),
        "model_joblib_iter": f"clai_ch3_v0_model_iter{iteration_id}.joblib",
        "log_jsonl": os.path.basename(LOG_PATH)
    },
    "runtime_seconds_cv": float(elapsed)
}

with open(LOG_PATH, "a", encoding="utf-8") as f:
    f.write(json.dumps(log_entry) + "\n")

# -------------------------
# Print run summary
# -------------------------
print("========== clai_ch3_v0 run ==========")
print(f"Train shape: {train_df.shape} | Test shape: {test_df.shape}")
print(f"Using XGBoost for Model B: {use_xgb}")
print(f"CV folds: {N_SPLITS} | Seed: {SEED} | CV runtime: {elapsed:.1f}s")
print("Fold Macro-F1 (ensemble, thr=0.5):", [round(x,4) for x in per_fold_thr05])
print("Fold Macro-F1 (ensemble, tuned): ", [round(x,4) for x in per_fold_tuned])
print(f"OOF best Macro-F1: {best['macro_f1']:.6f} | best_wA: {best['wA']:.2f} | best_thr: {best['thr']:.2f}")
print("OOF confusion matrix [[TN,FP],[FN,TP]]:", cm_oof)
print("Top high-confidence FP:", top_errors["false_positives_high_conf"][:3])
print("Top high-confidence FN:", top_errors["false_negatives_high_conf"][:3])
print("-------------------------------------")
print(f"✅ Wrote submission: {SUBMISSION_PATH}")
print(f"✅ Appended log:     {LOG_PATH}")
print(f"✅ Saved model:      {MODEL_PATH}")
print("=====================================")

========== clai_ch3_v0 run ==========
Train shape: (1000, 165) | Test shape: (1000, 164)
Using XGBoost for Model B: True
CV folds: 5 | Seed: 42 | CV runtime: 21.5s
Fold Macro-F1 (ensemble, thr=0.5): [0.7916, 0.7161, 0.7711, 0.7151, 0.7348]
Fold Macro-F1 (ensemble, tuned):  [0.8022, 0.7124, 0.7817, 0.7353, 0.7403]
OOF best Macro-F1: 0.755487 | best_wA: 0.00 | best_thr: 0.54
OOF confusion matrix [[TN,FP],[FN,TP]]: [[214, 130], [83, 573]]
Top high-confidence FP: [{'stay_id': 634, 'y_true': 0, 'proba': 0.9954608082771301}, {'stay_id': 482, 'y_true': 0, 'proba': 0.9925058484077454}, {'stay_id': 239, 'y_true': 0, 'proba': 0.9908022284507751}]
Top high-confidence FN: [{'stay_id': 867, 'y_true': 1, 'proba': 0.018794339150190353}, {'stay_id': 1308, 'y_true': 1, 'proba': 0.06590196490287781}, {'stay_id': 14, 'y_true': 1, 'proba': 0.07624445110559464}]
-------------------------------------
✅ Wrote submission: .\clai_ch3_v0_submission.csv
✅ Appended log:     .\clai_ch3_v0_iteration_detail.jsonl
✅ 

# Iteration 14
- 0.8163

In [30]:
# clai_ch3_v0 — Iteration: LGBM + CatBoost ensemble with per-unit threshold tuning (day1–10 only)
import os, json, re, math, time, warnings, joblib
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.base import clone

warnings.filterwarnings("ignore")

# ----------------------------
# Reproducibility
# ----------------------------
SEED = 42
np.random.seed(SEED)

# ----------------------------
# Paths (robust to running from project root or sandbox)
# ----------------------------
def resolve_path(fname: str) -> str:
    if os.path.exists(fname):
        return fname
    alt = os.path.join("/mnt/data", fname)  # sandbox fallback
    if os.path.exists(alt):
        return alt
    return fname  # let it error with a clear message

PATH_STAYS_TRAIN = resolve_path("stays_train.csv")
PATH_STAYS_TEST  = resolve_path("stays_test.csv")
PATH_PATIENTS    = resolve_path("patients.csv")
PATH_VITALS      = resolve_path("vitals_timeseries.json")

SUBMISSION_PATH  = "clai_ch3_v0_submission.csv"
JSONL_PATH       = "clai_ch3_v0_iteration_detail.jsonl"
MODEL_PATH       = "clai_ch3_v0_model.joblib"

# ----------------------------
# Load data
# ----------------------------
stays_train = pd.read_csv(PATH_STAYS_TRAIN)
stays_test  = pd.read_csv(PATH_STAYS_TEST)
patients    = pd.read_csv(PATH_PATIENTS)

with open(PATH_VITALS, "r") as f:
    vitals_list = json.load(f)

# stay_id -> days list
vitals_map = {int(x["stay_id"]): x["days"] for x in vitals_list}

TARGET = "discharge_ready_day11"
ID_COL = "stay_id"

# ----------------------------
# Feature engineering (compact, CPU-friendly)
# ----------------------------
NORMAL_RANGES = {
    "hr": (60, 100),
    "sbp": (90, 160),
    "dbp": (60, 100),
    "temp_c": (36.0, 37.8),
    "rr": (12, 22),
    "pp": (30, 80),      # pulse pressure
    "map": (65, 105),    # mean arterial pressure
}

KEYWORDS = [
    "discharge","dc","d/c","home","snf","rehab","pt","ot","ambulat","walk",
    "stable","improv","tolerat","pain","fever","afebr","oxygen","vent","iv",
    "antibiotic","wound","surg","telemetry","case management"
]

def linear_slope(x, y):
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    m = ~np.isnan(y)
    if m.sum() < 2:
        return np.nan
    x2, y2 = x[m], y[m]
    xm, ym = x2.mean(), y2.mean()
    denom = ((x2 - xm) ** 2).sum()
    if denom == 0:
        return 0.0
    return float(((x2 - xm) * (y2 - ym)).sum() / denom)

def featurize_stay_compact(days):
    days_sorted = sorted(days, key=lambda d: d.get("day", 0))
    day_idx = np.array([d.get("day", i+1) for i, d in enumerate(days_sorted)], dtype=float)

    feats = {}
    base_signals = ["hr", "sbp", "dbp", "temp_c", "rr"]
    arrs = {s: np.array([d.get(s, np.nan) for d in days_sorted], dtype=float) for s in base_signals}

    # Derived signals
    sbp, dbp = arrs["sbp"], arrs["dbp"]
    arrs["pp"]  = sbp - dbp
    arrs["map"] = (sbp + 2.0 * dbp) / 3.0

    # Per-signal aggregates
    for s, arr in arrs.items():
        feats[f"{s}_missing_cnt"] = int(np.isnan(arr).sum())

        if np.all(np.isnan(arr)):
            for stat in ["mean","std","min","max","first","last","delta","slope","last3_mean","last3_std",
                         "first3_mean","first3_std","ratio_last3_first3","abn_cnt","stable_cnt"]:
                feats[f"{s}_{stat}"] = np.nan
            continue

        feats[f"{s}_mean"] = float(np.nanmean(arr))
        feats[f"{s}_std"]  = float(np.nanstd(arr))
        feats[f"{s}_min"]  = float(np.nanmin(arr))
        feats[f"{s}_max"]  = float(np.nanmax(arr))

        nonnan = arr[~np.isnan(arr)]
        feats[f"{s}_first"] = float(nonnan[0])
        feats[f"{s}_last"]  = float(nonnan[-1])
        feats[f"{s}_delta"] = float(nonnan[-1] - nonnan[0])

        feats[f"{s}_slope"] = linear_slope(day_idx, arr)

        last3  = arr[-3:]
        first3 = arr[:3]
        feats[f"{s}_last3_mean"]  = float(np.nanmean(last3))
        feats[f"{s}_last3_std"]   = float(np.nanstd(last3))
        feats[f"{s}_first3_mean"] = float(np.nanmean(first3))
        feats[f"{s}_first3_std"]  = float(np.nanstd(first3))
        denom = feats[f"{s}_first3_mean"]
        feats[f"{s}_ratio_last3_first3"] = float(feats[f"{s}_last3_mean"] / denom) if (denom == denom and denom != 0) else np.nan

        lo, hi = NORMAL_RANGES.get(s, (None, None))
        if lo is not None:
            stable = (arr >= lo) & (arr <= hi)
            abn    = (arr < lo) | (arr > hi)
            feats[f"{s}_abn_cnt"]    = int(np.nansum(abn))
            feats[f"{s}_stable_cnt"] = int(np.nansum(stable))
        else:
            feats[f"{s}_abn_cnt"] = np.nan
            feats[f"{s}_stable_cnt"] = np.nan

    # All-vitals stable days (primary signals)
    stable_days = []
    for i in range(len(days_sorted)):
        ok = True
        for s in ["hr","sbp","dbp","temp_c","rr"]:
            v = arrs[s][i]
            lo, hi = NORMAL_RANGES[s]
            if np.isnan(v) or v < lo or v > hi:
                ok = False
                break
        stable_days.append(ok)
    feats["allvitals_stable_days"] = int(sum(stable_days))

    # Notes: lengths + keyword flags (binary presence, all days and last3)
    notes = [(d.get("note") or "") for d in days_sorted]
    lens = np.array([len(n) for n in notes], dtype=float)
    feats["note_len_mean"] = float(lens.mean())
    feats["note_len_std"]  = float(lens.std())
    feats["note_len_last"] = float(lens[-1])
    feats["note_len_last3_mean"] = float(lens[-3:].mean())

    text_all   = " ".join(notes).lower()
    text_last3 = " ".join(notes[-3:]).lower()

    for kw in KEYWORDS:
        kname = re.sub(r"[^a-z0-9]+", "_", kw)
        feats[f"kw_all_{kname}"]   = int(kw in text_all)
        feats[f"kw_last3_{kname}"] = int(kw in text_last3)

    # Keep last3 text for brief error analysis (NOT used in models)
    feats["_notes_last3_text"] = text_last3[:300]
    return feats

def build_features(stays_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for sid in stays_df[ID_COL].astype(int).values:
        days = vitals_map.get(int(sid))
        if days is None:
            feats = {"missing_vitals": 1, "_notes_last3_text": ""}
        else:
            feats = featurize_stay_compact(days)
            feats["missing_vitals"] = 0
        feats[ID_COL] = sid
        rows.append(feats)
    return pd.DataFrame(rows)

feat_train = build_features(stays_train)
feat_test  = build_features(stays_test)

train = stays_train.merge(patients, on="patient_id", how="left").merge(feat_train, on=ID_COL, how="left")
test  = stays_test.merge(patients, on="patient_id", how="left").merge(feat_test,  on=ID_COL, how="left")

# ----------------------------
# Model feature sets
# ----------------------------
CAT_COLS = ["unit_type", "admission_reason", "sex", "insurance", "zip3"]
AUX_TEXT_COL = "_notes_last3_text"  # keep only for error analysis

# numeric columns = everything except ids/target/cats/aux_text
DROP_COLS = [ID_COL, "patient_id", TARGET] + CAT_COLS + [AUX_TEXT_COL]
NUM_COLS = [c for c in train.columns if c not in DROP_COLS]

# Fill numeric NaNs with train medians (deterministic)
num_medians = train[NUM_COLS].median(numeric_only=True)
train[NUM_COLS] = train[NUM_COLS].fillna(num_medians)
test[NUM_COLS]  = test[NUM_COLS].fillna(num_medians)

# Fill missing categorical
for c in CAT_COLS:
    train[c] = train[c].fillna("UNK").astype(str)
    test[c]  = test[c].fillna("UNK").astype(str)

y = train[TARGET].astype(int).values
units = train["unit_type"].values

# ----------------------------
# LightGBM (one-hot categorical)
# ----------------------------
import lightgbm as lgb

X_lgb_train = pd.get_dummies(train[CAT_COLS + NUM_COLS], columns=CAT_COLS, dummy_na=True)
X_lgb_test  = pd.get_dummies(test[CAT_COLS + NUM_COLS],  columns=CAT_COLS, dummy_na=True)

# Align columns
X_lgb_train, X_lgb_test = X_lgb_train.align(X_lgb_test, join="left", axis=1, fill_value=0)

lgbm_params = dict(
    n_estimators=700,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_samples=30,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="binary",
    random_state=SEED,
    n_jobs=1,
    is_unbalance=True,
    deterministic=True,
    force_col_wise=True,
    verbose=-1
)
lgbm_base = lgb.LGBMClassifier(**lgbm_params)

# ----------------------------
# CatBoost (native categorical, no raw text)
# ----------------------------
from catboost import CatBoostClassifier

X_cb_train = train[CAT_COLS + NUM_COLS].copy()
X_cb_test  = test[CAT_COLS + NUM_COLS].copy()
for c in CAT_COLS:
    X_cb_train[c] = X_cb_train[c].astype(str)
    X_cb_test[c]  = X_cb_test[c].astype(str)

cat_idx = [X_cb_train.columns.get_loc(c) for c in CAT_COLS]

cb_params = dict(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=6.0,
    loss_function="Logloss",
    eval_metric="Logloss",
    random_seed=SEED,
    verbose=False,
    allow_writing_files=False,
    thread_count=1
)

def make_cb():
    return CatBoostClassifier(**cb_params)

# ----------------------------
# CV: OOF probs for both models
# ----------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_lgb = np.zeros(len(train), dtype=float)
oof_cb  = np.zeros(len(train), dtype=float)
fold_indices = []

print(f"[INFO] Train shape: {train.shape} | Test shape: {test.shape}")
print(f"[INFO] Class balance: {pd.Series(y).value_counts().to_dict()}")

t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(skf.split(train, y)):
    fold_indices.append((tr_idx, va_idx))
    y_tr = y[tr_idx]

    # LightGBM
    m_lgb = clone(lgbm_base)  # FIX: use sklearn.base.clone, not joblib.clone
    m_lgb.fit(X_lgb_train.iloc[tr_idx], y_tr)
    oof_lgb[va_idx] = m_lgb.predict_proba(X_lgb_train.iloc[va_idx])[:, 1]

    # CatBoost
    m_cb = make_cb()
    m_cb.fit(X_cb_train.iloc[tr_idx], y_tr, cat_features=cat_idx)
    oof_cb[va_idx] = m_cb.predict_proba(X_cb_train.iloc[va_idx])[:, 1]

print(f"[INFO] CV OOF prob generation done in {time.time()-t0:.2f}s")

# ----------------------------
# Threshold & weight tuning
# ----------------------------
thr_grid = np.linspace(0.10, 0.90, 81)
w_grid = np.linspace(0.0, 1.0, 41)  # step=0.025

def best_global_threshold(probs, y_true):
    best = (-1.0, 0.5)
    for thr in thr_grid:
        f = f1_score(y_true, (probs >= thr).astype(int), average="macro")
        if f > best[0]:
            best = (f, float(thr))
    return best  # (f1, thr)

def best_unit_thresholds(probs, unit_arr, y_true):
    thr_by = {}
    preds = np.zeros_like(y_true, dtype=int)
    for unit in np.unique(unit_arr):
        idx = (unit_arr == unit)
        best = (-1.0, 0.5)
        for thr in thr_grid:
            f = f1_score(y_true[idx], (probs[idx] >= thr).astype(int), average="macro")
            if f > best[0]:
                best = (f, float(thr))
        thr_by[str(unit)] = best[1]
        preds[idx] = (probs[idx] >= best[1]).astype(int)
    f_all = f1_score(y_true, preds, average="macro")
    return float(f_all), thr_by

best = {
    "macro_f1_oof": -1.0,
    "strategy": None,
    "weight_lgb": None,
    "threshold_global": None,
    "threshold_by_unit": None
}

for w in w_grid:
    probs = w * oof_lgb + (1.0 - w) * oof_cb

    f_g, thr_g = best_global_threshold(probs, y)
    if f_g > best["macro_f1_oof"]:
        best.update({
            "macro_f1_oof": float(f_g),
            "strategy": "global_threshold",
            "weight_lgb": float(w),
            "threshold_global": float(thr_g),
            "threshold_by_unit": None
        })

    f_u, thr_by = best_unit_thresholds(probs, units, y)
    if f_u > best["macro_f1_oof"]:
        best.update({
            "macro_f1_oof": float(f_u),
            "strategy": "unit_type_thresholds",
            "weight_lgb": float(w),
            "threshold_global": None,
            "threshold_by_unit": thr_by
        })

# Apply best strategy to compute final OOF preds + metrics
probs_ens = best["weight_lgb"] * oof_lgb + (1.0 - best["weight_lgb"]) * oof_cb
if best["strategy"] == "global_threshold":
    thr = best["threshold_global"]
    pred_oof = (probs_ens >= thr).astype(int)
else:
    pred_oof = np.zeros(len(y), dtype=int)
    thr_by = best["threshold_by_unit"]
    for unit, thr in thr_by.items():
        idx = (train["unit_type"].astype(str).values == unit)
        pred_oof[idx] = (probs_ens[idx] >= thr).astype(int)

macro_f1_oof = float(f1_score(y, pred_oof, average="macro"))
cm = confusion_matrix(y, pred_oof).tolist()

# per-fold f1
per_fold = []
for tr_idx, va_idx in fold_indices:
    y_va = y[va_idx]
    p_va = probs_ens[va_idx]
    if best["strategy"] == "global_threshold":
        pred_va = (p_va >= best["threshold_global"]).astype(int)
    else:
        unit_va = train["unit_type"].astype(str).values[va_idx]
        pred_va = np.zeros(len(va_idx), dtype=int)
        for unit, thr in best["threshold_by_unit"].items():
            m = (unit_va == unit)
            pred_va[m] = (p_va[m] >= thr).astype(int)
    per_fold.append(float(f1_score(y_va, pred_va, average="macro")))

# Error analysis
err_df = train[[ID_COL, "unit_type", "admission_reason", AUX_TEXT_COL]].copy()
err_df["y_true"] = y
err_df["proba"] = probs_ens
err_df["pred"] = pred_oof
fps = err_df[(err_df["y_true"] == 0) & (err_df["pred"] == 1)].sort_values("proba", ascending=False).head(5)
fns = err_df[(err_df["y_true"] == 1) & (err_df["pred"] == 0)].sort_values("proba", ascending=True).head(5)

print("\n====================")
print("[RESULT] OOF Tuning")
print("====================")
print(f"Best strategy: {best['strategy']}")
print(f"Best weight_lgb: {best['weight_lgb']:.3f} | weight_cat: {1.0-best['weight_lgb']:.3f}")
if best["strategy"] == "global_threshold":
    print(f"Global threshold: {best['threshold_global']:.3f}")
else:
    print(f"Thresholds by unit_type: {best['threshold_by_unit']}")
print(f"OOF Macro-F1: {macro_f1_oof:.6f}")
print(f"Per-fold Macro-F1: {[round(x,6) for x in per_fold]} | mean={np.mean(per_fold):.6f}")
print(f"OOF Confusion Matrix [[tn,fp],[fn,tp]]: {cm}")
print(f"OOF predicted positive rate: {pred_oof.mean():.3f}")

print("\n[Top False Positives]")
print(fps[[ID_COL, "unit_type", "admission_reason", "proba"]].to_string(index=False))
print("\n[Top False Negatives]")
print(fns[[ID_COL, "unit_type", "admission_reason", "proba"]].to_string(index=False))

# ----------------------------
# Train final models on full training data
# ----------------------------
lgbm_final = lgb.LGBMClassifier(**lgbm_params)
lgbm_final.fit(X_lgb_train, y)

cb_final = make_cb()
cb_final.fit(X_cb_train, y, cat_features=cat_idx)

# Predict test probs
p_test_lgb = lgbm_final.predict_proba(X_lgb_test)[:, 1]
p_test_cb  = cb_final.predict_proba(X_cb_test)[:, 1]
p_test_ens = best["weight_lgb"] * p_test_lgb + (1.0 - best["weight_lgb"]) * p_test_cb

# Apply thresholds
if best["strategy"] == "global_threshold":
    thr = best["threshold_global"]
    y_test_pred = (p_test_ens >= thr).astype(int)
else:
    y_test_pred = np.zeros(len(test), dtype=int)
    thr_by = best["threshold_by_unit"]
    test_units = test["unit_type"].astype(str).values
    # fallback if unseen unit type
    fallback_thr = float(np.median(list(thr_by.values()))) if len(thr_by) else 0.5
    for i, u in enumerate(test_units):
        thr = thr_by.get(u, fallback_thr)
        y_test_pred[i] = int(p_test_ens[i] >= thr)

# ----------------------------
# Write submission
# ----------------------------
sub = pd.DataFrame({
    "stay_id": stays_test["stay_id"].astype(int).values,
    "discharge_ready_day11": y_test_pred.astype(int)
})
sub.to_csv(SUBMISSION_PATH, index=False)
print(f"\n[INFO] Wrote submission: {SUBMISSION_PATH} | shape={sub.shape}")

# ----------------------------
# Save artifacts
# ----------------------------
# Determine next iteration_id (append-only)
def next_iteration_id(jsonl_path: str) -> int:
    if not os.path.exists(jsonl_path):
        return 0
    mx = -1
    with open(jsonl_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                it = obj.get("iteration_id", None)
                if isinstance(it, int):
                    mx = max(mx, it)
            except Exception:
                continue
    return mx + 1

ITER_ID = next_iteration_id(JSONL_PATH)

artifact = {
    "version": "v0",
    "team_tag": "clai",
    "iteration_id": ITER_ID,
    "trained_at": datetime.now().isoformat(),
    "lgbm_model": lgbm_final,
    "catboost_model": cb_final,
    "ensemble": {
        "strategy": best["strategy"],
        "weight_lgb": best["weight_lgb"],
        "threshold_global": best["threshold_global"],
        "threshold_by_unit": best["threshold_by_unit"],
    },
    "feature_info": {
        "cat_cols": CAT_COLS,
        "num_cols": NUM_COLS,
        "lgb_onehot_columns": list(X_lgb_train.columns),
        "num_medians": num_medians.to_dict()
    }
}
joblib.dump(artifact, MODEL_PATH)
joblib.dump(artifact, f"clai_ch3_v0_model_iter{ITER_ID}.joblib")
print(f"[INFO] Saved model artifact: {MODEL_PATH} and clai_ch3_v0_model_iter{ITER_ID}.joblib")

# ----------------------------
# Append iteration log JSONL
# ----------------------------
log_obj = {
    "version": "v0",
    "iteration_id": ITER_ID,
    "timestamp": datetime.now().isoformat(),
    "data_paths_used": {
        "base_dir": os.getcwd(),
        "stays_train": PATH_STAYS_TRAIN,
        "stays_test": PATH_STAYS_TEST,
        "patients": PATH_PATIENTS,
        "vitals_timeseries": PATH_VITALS,
        "join_keys": {"stays<->patients": "patient_id", "stays<->vitals_timeseries": "stay_id"}
    },
    "feature_summary": {
        "categorical_cols": CAT_COLS,
        "numeric_feature_count": len(NUM_COLS),
        "numeric_engineering": [
            "day1-10 vitals compact stats per signal (mean/std/min/max/first/last/delta/slope)",
            "windowed stats (first3/last3 mean/std + ratio)",
            "derived vitals (pp=sbp-dbp, map=(sbp+2*dbp)/3)",
            "abnormal/stable day counts + allvitals_stable_days",
            "note length stats + keyword presence flags (all days + last3 days)"
        ],
        "text_features": "No TF-IDF this iteration (keyword flags only).",
        "categorical_encoding": {
            "lightgbm": "one-hot (pd.get_dummies, dummy_na=True)",
            "catboost": "native categorical handling (cat_features indices)"
        }
    },
    "models_tried": [
        {"name": "LGBMClassifier", "hyperparams": lgbm_params},
        {"name": "CatBoostClassifier", "hyperparams": cb_params},
        {"name": "Weighted ensemble", "hyperparams": {
            "strategy": best["strategy"],
            "weight_lgb": best["weight_lgb"],
            "weight_cat": 1.0 - best["weight_lgb"],
            "threshold_global": best["threshold_global"],
            "threshold_by_unit": best["threshold_by_unit"],
            "weight_grid": {"start": 0.0, "end": 1.0, "step": 0.025},
            "threshold_grid": {"start": 0.10, "end": 0.90, "step": 0.01}
        }}
    ],
    "validation_protocol": {
        "cv": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": SEED,
        "metric": "macro_f1",
        "threshold_tuning": "tuned on OOF probabilities (global threshold and unit_type-specific thresholds; best kept)"
    },
    "metrics": {
        "macro_f1_oof": macro_f1_oof,
        "macro_f1_per_fold": per_fold,
        "confusion_matrix_oof_labels_0_1": cm,
        "class_balance_train": pd.Series(y).value_counts().to_dict(),
        "pred_positive_rate_oof": float(pred_oof.mean())
    },
    "thresholding_strategy": {
        "selected_strategy": best["strategy"],
        "selected_weight_lgb": best["weight_lgb"],
        "selected_threshold_global": best["threshold_global"],
        "selected_threshold_by_unit": best["threshold_by_unit"]
    },
    "top_errors_failure_analysis": {
        "false_positive_count_oof": int(((y == 0) & (pred_oof == 1)).sum()),
        "false_negative_count_oof": int(((y == 1) & (pred_oof == 0)).sum()),
        "top_false_positives": fps[[ID_COL, "unit_type", "admission_reason", "proba"]].to_dict(orient="records"),
        "top_false_negatives": fns[[ID_COL, "unit_type", "admission_reason", "proba"]].to_dict(orient="records")
    },
    "next_step_hypothesis": "Add low-rank TF-IDF (SVD) components from notes_last3 as additional numeric features for LGBM/CatBoost, or try a text-capable CatBoost model if runtime permits; optionally tune thresholds by admission_reason with strong regularization to avoid overfit.",
    "leakage_overfitting_warnings_considered": [
        "Used only day1-10 vitals + day1-10 daily notes-derived features to predict day11 readiness.",
        "Did NOT use discharge summaries / LOS / outcomes tables (potential leakage).",
        "Per-unit thresholds tuned on OOF predictions only; still monitor leaderboard for overfitting risk."
    ],
    "artifacts_written": {
        "submission_csv": os.path.abspath(SUBMISSION_PATH),
        "model_joblib": os.path.abspath(MODEL_PATH),
        "iteration_log_jsonl": os.path.abspath(JSONL_PATH)
    }
}

with open(JSONL_PATH, "a", encoding="utf-8") as f:
    f.write(json.dumps(log_obj) + "\n")
print(f"[INFO] Appended iteration log: {JSONL_PATH} (iteration_id={ITER_ID})")

[INFO] Train shape: (1000, 176) | Test shape: (1000, 175)
[INFO] Class balance: {1: 656, 0: 344}
[INFO] CV OOF prob generation done in 368.98s

[RESULT] OOF Tuning
Best strategy: unit_type_thresholds
Best weight_lgb: 0.200 | weight_cat: 0.800
Thresholds by unit_type: {'med_surg': 0.72, 'stepdown': 0.61}
OOF Macro-F1: 0.829908
Per-fold Macro-F1: [0.836227, 0.82906, 0.824176, 0.831297, 0.827665] | mean=0.829685
OOF Confusion Matrix [[tn,fp],[fn,tp]]: [[265, 79], [74, 582]]
OOF predicted positive rate: 0.661

[Top False Positives]
 stay_id unit_type admission_reason    proba
      84  med_surg     DiabetesComp 0.993818
    1475  med_surg        Pneumonia 0.992295
     482  med_surg     DiabetesComp 0.992151
    1214  med_surg     DiabetesComp 0.985914
     358  med_surg             COPD 0.984206

[Top False Negatives]
 stay_id unit_type admission_reason    proba
    1086  med_surg     DiabetesComp 0.023174
     663  med_surg           PostOp 0.070231
    1154  med_surg        Pneumonia 0.

# Iteration 15
- 0.7934

In [32]:
# clai_ch3_v0 — Iteration: LGBM + XGB structured blend (fixes joblib.clone -> sklearn.base.clone)
import os, re, json, random, datetime, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import clone

warnings.filterwarnings("ignore")

# -----------------------
# Config
# -----------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

SUBMISSION_NAME = "clai_ch3_v0_submission.csv"
LOG_NAME = "clai_ch3_v0_iteration_detail.jsonl"
MODEL_NAME = "clai_ch3_v0_model.joblib"

TARGET = "discharge_ready_day11"
ID_COLS = ["stay_id", "patient_id"]
CAT_COLS = ["unit_type", "admission_reason", "sex", "insurance", "zip3"]
TEXT_COLS = ["notes_all", "notes_last3", "notes_last1"]  # kept for feature engineering, not vectorized in this run

# -----------------------
# Robust path resolver
# -----------------------
def resolve_file(filename: str) -> Path:
    candidates = [
        Path(".") / filename,
        Path("./data") / filename,
        Path("/mnt/data") / filename,  # for this sandbox
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename} in {', '.join(str(c.parent) for c in candidates)}")

stays_train_path = resolve_file("stays_train.csv")
stays_test_path = resolve_file("stays_test.csv")
patients_path = resolve_file("patients.csv")
vitals_path = resolve_file("vitals_timeseries.json")

print("✅ Data paths:")
print(" -", stays_train_path)
print(" -", stays_test_path)
print(" -", patients_path)
print(" -", vitals_path)

# -----------------------
# Load data
# -----------------------
stays_train = pd.read_csv(stays_train_path)
stays_test = pd.read_csv(stays_test_path)
patients = pd.read_csv(patients_path)
with open(vitals_path, "r") as f:
    vitals_list = json.load(f)

# -----------------------
# Feature engineering
# -----------------------
def safe_float(x):
    try:
        if x is None:
            return np.nan
        return float(x)
    except Exception:
        return np.nan

KW_LIST = [
    "discharge","dc","d/c","ready","home","snf","rehab","placement","case","social","follow","outpatient",
    "stable","improv","better","worse","pain","fever","infection","oxygen","wean","transfer","icu","stepdown",
    "ambulat","walk","pt","ot","family","care"
]

def build_vitals_features(vit_list):
    rows = []
    for rec in vit_list:
        stay_id = rec["stay_id"]
        days = sorted(rec["days"], key=lambda d: d.get("day", 0))

        def arr(key):
            return np.array([safe_float(d.get(key)) for d in days], dtype=float)

        hr = arr("hr")
        sbp = arr("sbp")
        dbp = arr("dbp")
        temp = arr("temp_c")
        rr = arr("rr")

        pp = sbp - dbp
        mapv = (2 * dbp + sbp) / 3.0
        shock = hr / (sbp + 1e-3)

        signals = {
            "hr": hr, "sbp": sbp, "dbp": dbp, "temp_c": temp, "rr": rr,
            "pp": pp, "map": mapv, "shock": shock
        }

        feat = {"stay_id": stay_id}
        day_idx = np.arange(1, len(days) + 1, dtype=float)

        for name, x in signals.items():
            mask = ~np.isnan(x)
            xn = x[mask]

            # stats
            if xn.size == 0:
                for stat in [
                    "mean","std","min","max","median","q25","q75",
                    "first","last","delta","slope","r2",
                    "absdiff_mean","absdiff_max",
                    "first3_mean","first3_std","last3_mean","last3_std","last3_first3_ratio",
                    "missing_count","missing_rate"
                ]:
                    feat[f"{name}_{stat}"] = np.nan
                continue

            feat[f"{name}_mean"] = float(np.nanmean(x))
            feat[f"{name}_std"] = float(np.nanstd(x))
            feat[f"{name}_min"] = float(np.nanmin(x))
            feat[f"{name}_max"] = float(np.nanmax(x))
            feat[f"{name}_median"] = float(np.nanmedian(x))
            feat[f"{name}_q25"] = float(np.nanpercentile(x, 25))
            feat[f"{name}_q75"] = float(np.nanpercentile(x, 75))

            # first/last (non-missing)
            di = day_idx[mask]
            first_val = float(xn[0])
            last_val = float(xn[-1])
            feat[f"{name}_first"] = first_val
            feat[f"{name}_last"] = last_val
            feat[f"{name}_delta"] = last_val - first_val

            # slope + r2
            if xn.size >= 2:
                A = np.vstack([di, np.ones_like(di)]).T
                slope, intercept = np.linalg.lstsq(A, xn, rcond=None)[0]
                pred = slope * di + intercept
                ss_res = float(np.sum((xn - pred) ** 2))
                ss_tot = float(np.sum((xn - np.mean(xn)) ** 2))
                r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0
            else:
                slope, r2 = 0.0, 0.0
            feat[f"{name}_slope"] = float(slope)
            feat[f"{name}_r2"] = float(r2)

            # volatility
            diffs = np.abs(np.diff(xn))
            feat[f"{name}_absdiff_mean"] = float(diffs.mean()) if diffs.size else 0.0
            feat[f"{name}_absdiff_max"] = float(diffs.max()) if diffs.size else 0.0

            # window stats
            def win_stats(arr, sl):
                vals = arr[sl]
                vals = vals[~np.isnan(vals)]
                if vals.size == 0:
                    return (np.nan, np.nan)
                return (float(vals.mean()), float(vals.std()))

            f3m, f3s = win_stats(x, slice(0, 3))
            l3m, l3s = win_stats(x, slice(-3, None))
            feat[f"{name}_first3_mean"] = f3m
            feat[f"{name}_first3_std"] = f3s
            feat[f"{name}_last3_mean"] = l3m
            feat[f"{name}_last3_std"] = l3s
            feat[f"{name}_last3_first3_ratio"] = (l3m / f3m) if (not np.isnan(f3m) and abs(f3m) > 1e-6 and not np.isnan(l3m)) else np.nan

            feat[f"{name}_missing_count"] = float(np.isnan(x).sum())
            feat[f"{name}_missing_rate"] = float(np.isnan(x).mean())

        # abnormal day counts (heuristic)
        ranges = {
            "hr": (60, 100),
            "rr": (12, 20),
            "temp_c": (36.1, 37.2),
            "sbp": (90, 140),
            "map": (65, 100),
            "shock": (0.0, 0.9)
        }
        for name, (lo, hi) in ranges.items():
            x = signals[name]
            mask = ~np.isnan(x)
            if mask.sum() == 0:
                feat[f"{name}_abn_cnt"] = np.nan
                feat[f"{name}_abn_rate"] = np.nan
                feat[f"{name}_abn_last3_cnt"] = np.nan
                continue
            abn = ((x < lo) | (x > hi)) & mask
            feat[f"{name}_abn_cnt"] = float(abn.sum())
            feat[f"{name}_abn_rate"] = float(abn.sum() / mask.sum())
            x3 = x[-3:]
            m3 = ~np.isnan(x3)
            feat[f"{name}_abn_last3_cnt"] = float((((x3 < lo) | (x3 > hi)) & m3).sum()) if m3.sum() else np.nan

        # notes text + numeric note features
        notes = [(d.get("note") or "") for d in days]
        notes_clean = [str(n) for n in notes]
        notes_all = " \n ".join(notes_clean)
        notes_last3 = " \n ".join(notes_clean[-3:])
        notes_last1 = notes_clean[-1] if notes_clean else ""

        feat["notes_all"] = notes_all
        feat["notes_last3"] = notes_last3
        feat["notes_last1"] = notes_last1

        lens = np.array([len(n.strip()) for n in notes_clean], dtype=float)
        feat["note_len_mean"] = float(lens.mean()) if lens.size else 0.0
        feat["note_len_std"] = float(lens.std()) if lens.size else 0.0
        feat["note_len_last"] = float(lens[-1]) if lens.size else 0.0
        feat["note_nonempty_days"] = float((lens > 0).sum()) if lens.size else 0.0
        feat["note_empty_days"] = float((lens == 0).sum()) if lens.size else 0.0

        low_all = notes_all.lower()
        low3 = notes_last3.lower()

        # keyword counts (low-dim, fast)
        for kw in KW_LIST:
            col = f"kw_{re.sub('[^a-z0-9]+', '_', kw)}"
            feat[col] = float(low_all.count(kw))
        for kw in ["discharge", "dc", "d/c", "home", "snf", "rehab", "placement"]:
            col = f"kw3_{re.sub('[^a-z0-9]+', '_', kw)}"
            feat[col] = float(low3.count(kw))

        rows.append(feat)

    return pd.DataFrame(rows)

vitals_feat = build_vitals_features(vitals_list)

train = stays_train.merge(patients, on="patient_id", how="left").merge(vitals_feat, on="stay_id", how="left")
test = stays_test.merge(patients, on="patient_id", how="left").merge(vitals_feat, on="stay_id", how="left")

# column hygiene
for c in CAT_COLS:
    train[c] = train[c].astype("category")
    test[c] = test[c].astype("category")
for c in TEXT_COLS:
    train[c] = train[c].fillna("")
    test[c] = test[c].fillna("")

num_cols = [c for c in train.columns if c not in ID_COLS + [TARGET] + CAT_COLS + TEXT_COLS]

X = train.drop(columns=[TARGET])
y = train[TARGET].astype(int).values

print(f"\n✅ Train shape: {train.shape}, Test shape: {test.shape}")
print(f"✅ Numeric feature count: {len(num_cols)} | Categorical: {len(CAT_COLS)}")
print(f"✅ Class balance: neg={(y==0).sum()}, pos={(y==1).sum()}, pos_rate={(y==1).mean():.3f}")

# -----------------------
# Models
# -----------------------
try:
    import lightgbm as lgb
except Exception as e:
    raise ImportError("lightgbm is required for this run. Please install lightgbm.") from e

try:
    import xgboost as xgb
except Exception as e:
    raise ImportError("xgboost is required for this run. Please install xgboost.") from e

pre = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ],
    remainder="drop",
    sparse_threshold=0.0
)

lgb_params = dict(
    n_estimators=600,
    learning_rate=0.03,
    num_leaves=31,
    subsample=0.9,
    colsample_bytree=0.85,
    min_child_samples=20,
    reg_lambda=1.0,
    reg_alpha=0.1,
    objective="binary",
    class_weight="balanced",
    random_state=SEED,
    n_jobs=1,
    verbosity=-1
)

xgb_params = dict(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=2.0,
    reg_alpha=0.0,
    min_child_weight=1.0,
    gamma=0.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=SEED,
    n_jobs=1,
    tree_method="hist"
)

pipe_lgb = Pipeline([("preprocess", pre), ("clf", lgb.LGBMClassifier(**lgb_params))])
pipe_xgb = Pipeline([("preprocess", pre), ("clf", xgb.XGBClassifier(**xgb_params))])

# -----------------------
# Deterministic CV + OOF predictions
# -----------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_lgb = np.zeros(len(X), dtype=float)
oof_xgb = np.zeros(len(X), dtype=float)
fold_indices = []
fold_metrics = []

print("\n🔁 CV training...")
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    fold_indices.append(va_idx)

    m_lgb = clone(pipe_lgb)  # ✅ fix: sklearn.base.clone, not joblib.clone
    m_xgb = clone(pipe_xgb)

    m_lgb.fit(X.iloc[tr_idx], y[tr_idx])
    m_xgb.fit(X.iloc[tr_idx], y[tr_idx])

    p_lgb = m_lgb.predict_proba(X.iloc[va_idx])[:, 1]
    p_xgb = m_xgb.predict_proba(X.iloc[va_idx])[:, 1]

    oof_lgb[va_idx] = p_lgb
    oof_xgb[va_idx] = p_xgb

    fold_metrics.append({
        "fold": fold,
        "macro_f1_lgb_thr0.5": float(f1_score(y[va_idx], (p_lgb >= 0.5).astype(int), average="macro")),
        "macro_f1_xgb_thr0.5": float(f1_score(y[va_idx], (p_xgb >= 0.5).astype(int), average="macro")),
    })
    print(f"  Fold {fold}: LGB@0.5={fold_metrics[-1]['macro_f1_lgb_thr0.5']:.4f} | XGB@0.5={fold_metrics[-1]['macro_f1_xgb_thr0.5']:.4f}")

# -----------------------
# Fast macro-F1 + tuning (blend weight + threshold)
# -----------------------
def macro_f1_from_probs(y_true: np.ndarray, prob: np.ndarray, thr: float) -> float:
    pred = prob >= thr
    yb = y_true.astype(bool)

    tp = int(np.sum(pred & yb))
    fp = int(np.sum(pred & (~yb)))
    fn = int(np.sum((~pred) & yb))
    tn = int(np.sum((~pred) & (~yb)))

    denom1 = 2 * tp + fp + fn
    denom0 = 2 * tn + fp + fn

    f1_1 = (2 * tp / denom1) if denom1 > 0 else 0.0
    f1_0 = (2 * tn / denom0) if denom0 > 0 else 0.0
    return 0.5 * (f1_0 + f1_1)

def tune_blend_fast(y_true, p_xgb, p_lgb, w_grid, t_grid):
    best_f, best_w, best_t = -1.0, 0.0, 0.5
    for w in w_grid:
        p = w * p_xgb + (1 - w) * p_lgb
        for t in t_grid:
            f = macro_f1_from_probs(y_true, p, t)
            if f > best_f:
                best_f, best_w, best_t = float(f), float(w), float(t)
    return best_f, best_w, best_t

w_grid = np.linspace(0, 1, 21)              # coarse, stable
t_grid = np.linspace(0.3, 0.8, 51)          # coarse, stable
best_f, best_w, best_t = tune_blend_fast(y, oof_xgb, oof_lgb, w_grid, t_grid)

# Optional tiny refinement around best (still deterministic)
w_grid2 = np.clip(np.linspace(best_w - 0.1, best_w + 0.1, 21), 0, 1)
t_grid2 = np.clip(np.linspace(best_t - 0.05, best_t + 0.05, 41), 0.05, 0.95)
best_f2, best_w2, best_t2 = tune_blend_fast(y, oof_xgb, oof_lgb, w_grid2, t_grid2)
if best_f2 >= best_f:
    best_f, best_w, best_t = best_f2, best_w2, best_t2

oof_blend = best_w * oof_xgb + (1 - best_w) * oof_lgb
pred_oof = (oof_blend >= best_t).astype(int)

macro_f1_oof = float(f1_score(y, pred_oof, average="macro"))
cm = confusion_matrix(y, pred_oof, labels=[0, 1]).tolist()

per_fold_tuned = []
for va_idx in fold_indices:
    p = best_w * oof_xgb[va_idx] + (1 - best_w) * oof_lgb[va_idx]
    per_fold_tuned.append(float(macro_f1_from_probs(y[va_idx], p, best_t)))

print("\n📊 OOF tuned ensemble results")
print(f" - best_weight_xgb: {best_w:.3f} (weight_lgb={1-best_w:.3f})")
print(f" - best_threshold : {best_t:.3f}")
print(f" - macro_f1_oof   : {macro_f1_oof:.4f}")
print(f" - per_fold_tuned : {[round(x,4) for x in per_fold_tuned]}")
print(f" - confusion_matrix [0,1]: {cm}")
print(f" - oof positive rate: {pred_oof.mean():.3f}")

# -----------------------
# Failure analysis: top FP/FN (OOF)
# -----------------------
err_df = train[["stay_id", "unit_type", "admission_reason"]].copy()
err_df["y_true"] = y
err_df["p_blend"] = oof_blend
err_df["y_pred"] = pred_oof

fp_top = (err_df[(err_df.y_true == 0) & (err_df.y_pred == 1)]
          .sort_values("p_blend", ascending=False)
          .head(5))[["stay_id", "p_blend", "unit_type", "admission_reason"]].to_dict("records")

fn_top = (err_df[(err_df.y_true == 1) & (err_df.y_pred == 0)]
          .sort_values("p_blend", ascending=True)
          .head(5))[["stay_id", "p_blend", "unit_type", "admission_reason"]].to_dict("records")

print("\n🔎 Top errors (OOF, tuned)")
print(" - FP top5:", fp_top)
print(" - FN top5:", fn_top)

# -----------------------
# Train final models + predict test
# -----------------------
print("\n🏁 Training final models on full train...")
final_lgb = clone(pipe_lgb).fit(X, y)
final_xgb = clone(pipe_xgb).fit(X, y)

p_lgb_test = final_lgb.predict_proba(test)[:, 1]
p_xgb_test = final_xgb.predict_proba(test)[:, 1]
p_blend_test = best_w * p_xgb_test + (1 - best_w) * p_lgb_test
pred_test = (p_blend_test >= best_t).astype(int)

submission = pd.DataFrame({
    "stay_id": test["stay_id"].values,
    "discharge_ready_day11": pred_test.astype(int)
})

submission.to_csv(SUBMISSION_NAME, index=False)
print(f"✅ Wrote submission: {SUBMISSION_NAME} | shape={submission.shape} | pos_rate={submission['discharge_ready_day11'].mean():.3f}")

# -----------------------
# Save artifact
# -----------------------
artifact = {
    "version": "v0",
    "seed": SEED,
    "cat_cols": CAT_COLS,
    "num_cols": num_cols,
    "lgb_params": lgb_params,
    "xgb_params": xgb_params,
    "blend_weight_xgb": float(best_w),
    "blend_threshold": float(best_t),
    "model_lgb": final_lgb,
    "model_xgb": final_xgb,
}
joblib.dump(artifact, MODEL_NAME)
print(f"✅ Saved model artifact: {MODEL_NAME}")

# also save iteration-stamped artifact for traceability
iter_artifact_name = None

# -----------------------
# Append iteration log (append-only)
# -----------------------
log_path = Path(LOG_NAME)
max_id = -1
if log_path.exists():
    for line in log_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
            if "iteration_id" in obj:
                max_id = max(max_id, int(obj["iteration_id"]))
        except Exception:
            continue
iteration_id = max_id + 1 if max_id >= 0 else 0

iter_artifact_name = f"clai_ch3_v0_model_iter{iteration_id}.joblib"
joblib.dump(artifact, iter_artifact_name)

entry = {
    "version": "v0",
    "iteration_id": int(iteration_id),
    "timestamp": datetime.datetime.now().isoformat(),
    "data_paths_used": {
        "stays_train": str(stays_train_path),
        "stays_test": str(stays_test_path),
        "patients": str(patients_path),
        "vitals_timeseries": str(vitals_path),
    },
    "join_keys": {
        "stays.patient_id": "patients.patient_id",
        "stays.stay_id": "vitals_timeseries.stay_id",
    },
    "feature_summary": {
        "categorical_cols": CAT_COLS,
        "numeric_feature_count": int(len(num_cols)),
        "categorical_encoding": "OneHotEncoder(handle_unknown='ignore')",
        "text_features": "No TF-IDF; used note length stats + keyword counts from daily notes.",
        "engineered_numeric_vitals": [
            "mean/std/min/max/median/q25/q75",
            "first/last/delta",
            "slope + r2 over days",
            "first3/last3 mean/std + ratio",
            "abs-diff volatility (mean/max)",
            "missingness count/rate",
            "derived: pp, map, shock",
            "abnormal day counts + abnormal last3 counts",
        ],
        "keyword_feature_count": int(sum(c.startswith("kw_") or c.startswith("kw3_") for c in num_cols)),
    },
    "models_tried": [
        {"name": "LGBMClassifier", "hyperparams": lgb_params},
        {"name": "XGBClassifier", "hyperparams": xgb_params},
        {"name": "Blend + threshold tuning", "hyperparams": {"weight_xgb": float(best_w), "weight_lgb": float(1-best_w), "threshold": float(best_t)}},
    ],
    "validation_protocol": {
        "cv": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": SEED,
        "determinism": "numpy/random seeds fixed; n_jobs=1 for boosters; sklearn.base.clone for fold models",
    },
    "metrics": {
        "macro_f1_oof_best": float(best_f),
        "macro_f1_per_fold_tuned": [float(x) for x in per_fold_tuned],
        "macro_f1_per_fold_thr0.5_lgb": [float(m["macro_f1_lgb_thr0.5"]) for m in fold_metrics],
        "macro_f1_per_fold_thr0.5_xgb": [float(m["macro_f1_xgb_thr0.5"]) for m in fold_metrics],
        "confusion_matrix_oof_tuned_labels_0_1": cm,
        "class_balance_train": {"neg": int((y==0).sum()), "pos": int((y==1).sum()), "pos_rate": float((y==1).mean())},
        "pred_positive_rate_oof": float(pred_oof.mean()),
    },
    "thresholding_strategy": {
        "method": "Grid search on OOF predictions maximizing macro-F1",
        "weight_grid": "linspace(0,1,21) then local refine",
        "threshold_grid": "linspace(0.3,0.8,51) then local refine",
    },
    "top_errors_failure_analysis": {
        "false_positives_top5": fp_top,
        "false_negatives_top5": fn_top,
    },
    "next_step_hypothesis": "Add a tiny (<500 dim) text TF-IDF for the last 3 days notes as a 3rd model to blend only if OOF improves and threshold remains stable.",
    "leakage_overfitting_warnings_considered": [
        "Did not use stay_id/patient_id as features (only joins).",
        "Used only Day1-10 vitals/notes to predict Day11 readiness.",
        "Blend weight/threshold tuned strictly on OOF predictions (no test access).",
    ],
    "artifacts_written": {
        "submission_csv": SUBMISSION_NAME,
        "model_joblib": MODEL_NAME,
        "model_joblib_iter": iter_artifact_name,
        "log_jsonl": LOG_NAME,
    },
}

with open(log_path, "a", encoding="utf-8") as f:
    f.write(json.dumps(entry) + "\n")

print(f"\n🧾 Appended log: {LOG_NAME} (iteration_id={iteration_id})")
print("✅ Done.")

✅ Data paths:
 - stays_train.csv
 - stays_test.csv
 - patients.csv
 - vitals_timeseries.json

✅ Train shape: (1000, 240), Test shape: (1000, 239)
✅ Numeric feature count: 229 | Categorical: 5
✅ Class balance: neg=344, pos=656, pos_rate=0.656

🔁 CV training...
  Fold 0: LGB@0.5=0.8663 | XGB@0.5=0.8468
  Fold 1: LGB@0.5=0.7716 | XGB@0.5=0.7572
  Fold 2: LGB@0.5=0.8162 | XGB@0.5=0.8112
  Fold 3: LGB@0.5=0.8127 | XGB@0.5=0.8112
  Fold 4: LGB@0.5=0.8364 | XGB@0.5=0.8063

📊 OOF tuned ensemble results
 - best_weight_xgb: 0.290 (weight_lgb=0.710)
 - best_threshold : 0.698
 - macro_f1_oof   : 0.8251
 - per_fold_tuned : [0.8507, 0.7932, 0.8128, 0.8352, 0.8317]
 - confusion_matrix [0,1]: [[274, 70], [90, 566]]
 - oof positive rate: 0.636

🔎 Top errors (OOF, tuned)
 - FP top5: [{'stay_id': 482, 'p_blend': 0.997130197194998, 'unit_type': 'med_surg', 'admission_reason': 'DiabetesComp'}, {'stay_id': 1931, 'p_blend': 0.9955892775246273, 'unit_type': 'med_surg', 'admission_reason': 'DiabetesComp'}, {'s

# New Iter

Iterations: 0.7767, 0.8039, 0.8073, 0.8067, 0.7905, 0.8040, 0.7974, 0.8190, 0.8006, 0.8067, 0.7915, 0.8097, 0.7324, 0.8163, 0.7934

# Submission

In [33]:
# 3. Submit Predictions

# Submit predictions to the competition
print("🚀 Submitting predictions...")

try:
    result = client.submit_prediction("Healthcare", 3, "clai_ch3_v0_submission.csv")
    
    if result['success']:
        print("✅ Submission successful!")
        print(f"   📊 Score: {result['score']:.4f}")
        print(f"   📏 Metric: {result['metric_name']}")
        print(f"   ✔️  Validation: {'Passed' if result['validation_passed'] else 'Failed'}")
    else:
        print("❌ Submission failed!")
        print(f"   Error details: {result.get('details', {}).get('validation_errors', 'Unknown error')}")
        
except Exception as e:
    print(f"💥 Submission error: {e}")
    print("🔧 Check your API key and team name are correct!")

print("\n🎯 Next steps:")
print("   1. Try incorporating relevant information outside this table!")
print("   2. You've completed all Healthcare challenges!")


🚀 Submitting predictions...
✅ Prediction submitted successfully!
📊 Score: 0.7934 (Macro-F1)
✅ Validation passed
✅ Submission successful!
   📊 Score: 0.7934
   📏 Metric: Macro-F1
   ✔️  Validation: Passed

🎯 Next steps:
   1. Try incorporating relevant information outside this table!
   2. You've completed all Healthcare challenges!
